# Les GAB du parc : comprendre leur activité et repérer ceux qui sortent de l'ordinaire

**À quoi sert ce notebook.** Il lit la fiche d'identité des GAB (une ligne par GAB et par jour, sur 3 ans) et répond à trois questions :

1. **Comment fonctionne le parc ?** Volumes, rythmes de la semaine et de l'année, types de cartes, billets, argent dans les GAB.
2. **Quels GAB sortent de l'ordinaire, et pourquoi ?** Chaque jour, les GAB les plus atypiques, avec une phrase claire qui explique ce qui ne va pas.
3. **Que montrent les cas réels ?** Une inversion de cassettes, des GAB dégradés pendant des manifestations, une rafale de cartes étrangères.

**Comment le lire.** Chaque section commence par « En bref », dit ce que vous voyez et à quoi ça sert. Les détails techniques sont rangés
dans des encadrés « Comment c'est calculé », à ouvrir seulement si besoin. Pour aller vite : la **section 19 (tableau de bord)** donne
l'essentiel, la **section 12** la liste du dernier jour, la **partie C** les trois cas.

**Accès rapide** : [**L'essentiel sur une page**](#essentiel) · [Tableau de bord](#s19) · [Vision annuelle](#s19bis) · [Les GAB à regarder aujourd'hui](#s12) · [Télécharger l'Excel](#s20)

**Le fichier Excel** (section 20) reprend tous les résultats : un lien de téléchargement apparaît en fin de notebook.

**Les mots utilisés**

| Mot | Ce qu'il veut dire |
|---|---|
| **GAB-jour** | un GAB, un jour donné. Un GAB en alerte pendant 10 jours compte 10 GAB-jours, mais un seul GAB |
| **Habitude** | ce que fait d'ordinaire **ce GAB** ce jour-là de la semaine (pour un lundi : ses 4 derniers lundis) |
| **Anomalie** | une situation anormale précise, avec un nom clair (ex. « GAB actif sans aucun retrait ») et une phrase chiffrée |
| **Niveau** | **Critique** : à traiter tout de suite ; **Élevé** : à traiter ; **À surveiller** : à garder à l'œil ; **Normal** : rien à signaler |
| **Rang du jour** | le classement du GAB parmi tout le parc ce jour-là : 1 = le plus atypique |
| **Indice** | 1 = un jour ordinaire ; 1,2 = 20 % au-dessus ; 0,7 = 30 % en dessous |

<a id='sommaire'></a>
**Sommaire** (cliquer sur un numéro pour y aller)

| Partie | Section | Ce qu'on y trouve |
|---|---|---|
| | [★](#essentiel) | **L'essentiel pour le métier : les GAB à regarder, leur diagnostic, les chiffres, les actions** |
| | [1](#s1) | Réglages (pour la personne qui lance le notebook) |
| A. Comprendre le parc | [2](#s2) | Jusqu'à quelle date les données sont-elles fiables ? |
| | [3](#s3) | Le parc en chiffres |
| | [4](#s4) | Les rythmes de la semaine et de l'année |
| | [5](#s5) | À quoi ressemble un retrait ordinaire |
| | [6](#s6) | Où sont les GAB |
| | [7](#s7) | Les types de GAB (familles) |
| | [7 bis](#s7bis) | Billets, argent dans les GAB, cartes étrangères, heures, qualité des données |
| | [7 ter](#s7ter) | Le rythme propre de chaque GAB |
| B. Repérer les GAB anormaux | [8](#s8) | L'habitude de chaque GAB |
| | [9](#s9) | Ce qui se passe autour du GAB (voisins) |
| | [10](#s10) | La liste des anomalies recherchées |
| | [11](#s11) | Ce que trouve la détection sur toute la période |
| | [12](#s12) | Le dernier jour : les GAB à regarder ; rejouer n'importe quel jour ; un exemple par anomalie |
| | [13](#s13) | La méthode retrouve-t-elle les incidents connus ? |
| C. Vérification sur les cas connus (usage interne) | [14](#s14) | L'inversion de cassettes |
| | [15](#s15) | Les GAB dégradés pendant les manifestations |
| | [16](#s16) | La rafale de cartes étrangères |
| | [17](#s17) | Les records des réseaux étrangers |
| D. Pour le métier | [18](#s18) | Billets à prévoir, GAB qui se vident, créneau d'intervention, pannes, GAB sous-utilisés ou saturés |
| | [19](#s19) | **Le tableau de bord des 3 ans** |
| | [19 bis](#s19bis) | **La vision annuelle** : chaque année côte à côte |
| E. Le fichier Excel | [20](#s20) | Télécharger les résultats |

<details><summary><i>Pour la personne qui lance le notebook</i></summary>

Entrées : la fiche `fiche_gab_jour` et la recette des voisins `voisins_gab`. Modifier seulement la section 1, puis tout exécuter. Le notebook
lit toute la fiche (toutes les colonnes, les 3 ans) : compter 20 à 40 minutes et une quinzaine de Go de mémoire. Aucun modèle d'apprentissage
ici : uniquement des comparaisons de chaque GAB à ses habitudes.

</details>

<a href="#sommaire" style="font-size:12px">↑ retour au sommaire</a>
<a id="s0"></a>
## 0. Préparation technique (rien à lire)
Rien à modifier ici. Cette cellule charge les bibliothèques et définit les quelques fonctions qui mettent en forme
les résultats (encadrés, tuiles de chiffres, tableaux, graphiques). Les graphiques sont interactifs (Plotly : survoler
un point affiche sa valeur) ; si Plotly manque, les mêmes graphiques sont dessinés avec matplotlib.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import base64
import unicodedata
import zipfile
from io import BytesIO

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import dataiku
from IPython.display import display, HTML

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 250)

# --- Plotly (interactif) si disponible, sinon matplotlib
try:
    import plotly.graph_objs as go
    from plotly.offline import init_notebook_mode, iplot
    init_notebook_mode(connected=False)          # la bibliothèque est embarquée dans le notebook : fonctionne sans internet
    PLOTLY = True
except Exception as _e:
    PLOTLY = False
    print("Plotly indisponible ({}) : graphiques matplotlib.".format(_e))

# --- Palette unique pour tout le notebook
BLEU, BLEU_CLAIR, GRIS, GRIS_CLAIR, NOIR = "#2f6db5", "#9dbfe3", "#8a8f98", "#d9d8d3", "#1f2328"
COULEURS_NIVEAU = {"Critique": "#cf222e", "Élevé": "#e36209", "À surveiller": "#c99a06", "Normal": "#b8b8b8"}
ORDRE_NIVEAUX = ["Critique", "Élevé", "À surveiller", "Normal"]
SERIE = [BLEU, "#e36209", "#2da44e", "#8250df", "#bf3989", "#1b7c83", "#9a6700", "#57606a"]

CSS = """<style>
.g-note{border-left:4px solid #2f6db5;background:#f6f8fa;padding:8px 12px;margin:8px 0;font-size:13.5px;line-height:1.45;color:#1f2328}
.g-note.warn{border-color:#d29922;background:#fff8e6}.g-note.bad{border-color:#cf222e;background:#fff1f0}.g-note.ok{border-color:#1a7f37;background:#eefbf1}
.g-ret{border:1px solid #d0d7de;border-left:6px solid #2f6db5;border-radius:4px;background:#fff;padding:10px 14px;margin:10px 0;font-size:13.5px;line-height:1.5}
.g-ret b.t{display:block;color:#2f6db5;text-transform:uppercase;font-size:11px;letter-spacing:.06em;margin-bottom:4px}
.g-ret ul{margin:4px 0 0 18px;padding:0}.g-ret li{margin:2px 0}
.g-band{background:#1f2328;color:#fff;padding:10px 16px;margin:14px 0 6px 0;border-radius:4px}
.g-band b{font-size:17px;letter-spacing:.02em}.g-band span.sub{display:block;color:#c9d1d9;font-size:12.5px;margin-top:4px}
.g-kpis{display:flex;flex-wrap:wrap;gap:10px;margin:8px 0}
.g-kpi{flex:1 1 150px;min-width:150px;max-width:230px;border:1px solid #d0d7de;border-radius:6px;padding:10px 12px;background:#fff}
.g-kpi .l{font-size:11.5px;color:#57606a;text-transform:uppercase;letter-spacing:.04em}
.g-kpi .v{font-size:24px;font-weight:600;color:#1f2328;margin:2px 0;font-variant-numeric:tabular-nums}
.g-kpi .s{font-size:12px;color:#57606a}.g-kpi .s.up{color:#1a7f37}.g-kpi .s.down{color:#cf222e}
table.g{border-collapse:collapse;border:0 !important;font-size:12.5px;margin:6px 0;font-family:inherit}
table.g caption{text-align:left;font-weight:600;font-size:13.5px;padding:4px 0 6px 0;color:#1f2328}
table.g th{background:#f6f8fa;color:#1f2328;text-align:left !important;padding:6px 10px;border-bottom:2px solid #d0d7de;font-weight:600;white-space:nowrap}
table.g tbody tr td,table.g tr td{padding:5px 10px;border-bottom:1px solid #eaeef2;vertical-align:top;text-align:left !important}table.g tr:nth-child(even) td{background:#fbfcfd}
table.g tbody tr td.n,table.g tr td.n{text-align:right !important;font-variant-numeric:tabular-nums;white-space:nowrap}table.g tr td.d{max-width:620px;line-height:1.4}
table.g tr td.v{max-width:200px;color:#57606a;font-size:11.5px}table.g tr td.m{color:#57606a;font-size:11.5px}
.pill{display:inline-block;padding:1px 8px;border-radius:10px;font-size:11.5px;font-weight:600;color:#fff;white-space:nowrap}
.pill.c{background:#cf222e}.pill.e{background:#e36209}.pill.s{background:#c99a06}.pill.n{background:#b8b8b8;color:#1f2328}.pill.i{background:#8a8f98}
.g-leg{font-size:12px;color:#57606a;margin:2px 0 8px 0}
</style>"""
display(HTML(CSS))
# bouton « Afficher / masquer le code » : les lecteurs du métier voient les textes, les tableaux et les graphiques sans le code
display(HTML("""<style>div.code_cell div.input, .jp-CodeCell .jp-Cell-inputWrapper {display:none}</style>
<button onclick="var v = this.getAttribute('data-v') != '1'; this.setAttribute('data-v', v ? '1' : '0');
  document.querySelectorAll('div.code_cell div.input, .jp-CodeCell .jp-Cell-inputWrapper').forEach(function (z) { z.style.display = v ? 'flex' : 'none'; });
  this.innerText = v ? 'Masquer le code' : 'Afficher le code';"
  style="padding:6px 14px;border:1px solid #2f6db5;border-radius:6px;background:#fff;color:#2f6db5;font-weight:600;cursor:pointer">Afficher le code</button>
<span style="font-size:12px;color:#57606a;margin-left:8px">Le code est masqué pour la lecture : ce bouton le fait réapparaître.</span>"""))


CLASSE_NIVEAU = {"Critique": "c", "Élevé": "e", "À surveiller": "s", "Normal": "n", "information": "i"}


def jfr(d, court=False):
    """Date en français : « lundi 08/11/2026 » (ou « lun. 08/11 »)."""
    d = pd.Timestamp(d)
    j = ["lundi", "mardi", "mercredi", "jeudi", "vendredi", "samedi", "dimanche"][d.dayofweek]
    return (j[:3] + ". " + d.strftime("%d/%m")) if court else (j + " " + d.strftime("%d/%m/%Y"))


def nb(x, dec=0):
    """Nombre lisible : 12 345 ou 12 345,6 ; '-' si manquant."""
    if x is None or (isinstance(x, float) and np.isnan(x)) or (not isinstance(x, (int, float, np.integer, np.floating))):
        return "-" if x is None or (isinstance(x, float) and np.isnan(x)) else str(x)
    s = "{:,.{}f}".format(float(x), dec).replace(",", " ").replace(".", ",")
    return s


def pct(x, dec=1):
    return "-" if x is None or pd.isnull(x) else ("{:.{}f} %".format(100.0 * x, dec)).replace(".", ",")


EN_CLAIR = [(" (loi de Poisson)", ""), ("écart ≥ 3 variations habituelles", "nettement au-dessus de ses variations habituelles"),
            ("écart ≤ −3 variations habituelles", "nettement en dessous de ses variations habituelles"), (" (écart médian ≥ 2)", ""), (" (écart médian ≤ −2)", ""),
            ("distance de Jensen-Shannon ≥ 0,25 entre la répartition horaire du jour et celle des 28 derniers jours", "les retraits se répartissent dans la journée très différemment des 4 dernières semaines"),
            ("distance de Jensen-Shannon ≥ 0,25 entre la répartition par réseau du jour et celle des 28 derniers jours", "les types de cartes utilisés sont très différents des 4 dernières semaines"),
            ("vu la moyenne des 28 derniers jours", "vu les 4 dernières semaines du GAB"), ("(écart ≥ 3)", "(très au-dessus de l'habitude)"), (" (écart ≥ 4)", ""),
            ("et écart ≥ 3", ", très au-dessus de l'habitude"), ("médiane des GAB-jours actifs", "valeur typique d'un jour de GAB"), ("valeurs médianes", "valeurs typiques"), ("(médiane des 4 précédents)", "(valeur typique des 4 précédents)"), ("(médiane)", "(valeur typique)"),
            ("médiane", "valeur typique"), ("GAB médian", "GAB typique"), ("volume médian", "volume typique"), ("distance ", "écart de répartition ")]


def en_clair(t):
    """Remplace le vocabulaire technique par des mots de tous les jours."""
    t = str(t)
    for a_, b_ in EN_CLAIR:
        t = t.replace(a_, b_)
    return t


def note(texte, genre="info"):
    """Encadré court : info (bleu), warn (orange), bad (rouge), ok (vert)."""
    display(HTML(CSS + '<div class="g-note {}">{}</div>'.format("" if genre == "info" else genre, texte)))


def bandeau(titre, sous_titre=""):
    display(HTML(CSS + '<div class="g-band"><b>{}</b><span class="sub">{}</span></div>'.format(en_clair(titre), en_clair(sous_titre))))


def retenir(points, titre="À retenir"):
    """Encadré « À retenir » : liste de phrases, calculées sur les données."""
    display(HTML(CSS + '<div class="g-ret"><b class="t">{}</b><ul>{}</ul></div>'.format(titre, "".join("<li>{}</li>".format(en_clair(p)) for p in points if p))))


def tuiles(items, retour=False):
    """Tuiles de chiffres : items = [(libellé, valeur, sous-texte, sens)] ; sens = 'up' / 'down' / '' colore le sous-texte."""
    h = ""
    for it in items:
        lib, val = it[0], it[1]
        sous = it[2] if len(it) > 2 else ""
        sens = it[3] if len(it) > 3 else ""
        h += '<div class="g-kpi"><div class="l">{}</div><div class="v">{}</div><div class="s {}">{}</div></div>'.format(en_clair(lib), val, sens, en_clair(sous))
    if retour:
        return '<div class="g-kpis">{}</div>'.format(h)
    display(HTML(CSS + '<div class="g-kpis">{}</div>'.format(h)))


def _html(x):
    return str(x).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")


def tableau(df, titre="", n=50, libelles=None, pills=("niveau",), longues=("diagnostic", "resume", "résumé", "comment on décide", "exemple"),
            petites=("voisins", "anomalies", "code technique", "codes"), index=False, dec=None):
    """Tableau HTML propre : en-têtes en français (dict `libelles`), nombres alignés à droite, niveau en pastille,
    textes longs sur une colonne large, colonnes secondaires en petit. Jamais plus de `n` lignes."""
    t = df.head(n).copy()
    if index:
        t = t.reset_index()
    libelles = libelles or {}
    cols = list(t.columns)
    th = "".join("<th>{}</th>".format(_html(en_clair(libelles.get(c, c)))) for c in cols)
    lignes = []
    for _, r in t.iterrows():
        cel = []
        for c in cols:
            x = r[c]
            lc = str(libelles.get(c, c)).lower()
            if c in pills or lc in pills:
                cel.append('<td><span class="pill {}">{}</span></td>'.format(CLASSE_NIVEAU.get(str(x), "i"), _html(x)))
            elif c in longues or lc in longues:
                cel.append('<td class="d">{}</td>'.format(_html("" if pd.isnull(x) else x)))
            elif c in petites or lc in petites:
                cel.append('<td class="v">{}</td>'.format(_html("" if pd.isnull(x) else x)))
            elif isinstance(x, (pd.Timestamp,)):
                cel.append("<td>{}</td>".format(jfr(x)))
            elif isinstance(x, (float, np.floating)):
                d_ = dec if dec is not None else (0 if abs(x) >= 100 or float(x).is_integer() else 1)
                cel.append('<td class="n">{}</td>'.format(nb(x, d_)))
            elif isinstance(x, (int, np.integer)) and not isinstance(x, (bool, np.bool_)):
                cel.append('<td class="n">{}</td>'.format(nb(x)))
            else:
                sx = "" if pd.isnull(x) else str(x)
                cel.append('<td{}>{}</td>'.format(' style="white-space:nowrap"' if len(sx) <= 28 else "", _html(sx)))
        lignes.append("<tr>{}</tr>".format("".join(cel)))
    cap = "<caption>{}</caption>".format(en_clair(titre)) if titre else ""
    suite = '<div class="g-leg">… {} lignes de plus (voir l\'Excel).</div>'.format(nb(len(df) - n)) if len(df) > n else ""
    return HTML(CSS + '<table class="g">{}<tr>{}</tr>{}</table>{}'.format(cap, th, "".join(lignes), suite))


def pastilles(niveaux):
    """« 3 Critique, 12 Élevé, 40 À surveiller » avec les couleurs."""
    vc = pd.Series(niveaux).value_counts()
    return " ".join('<span class="pill {}">{} {}</span>'.format(CLASSE_NIVEAU[k], nb(vc.get(k, 0)), k) for k in ORDRE_NIVEAUX if vc.get(k, 0))


# --- Graphiques : une fonction par forme ; Plotly si présent, sinon matplotlib
def _layout(titre, x_titre="", y_titre="", hauteur=360, **kw):
    lay = dict(title=dict(text=en_clair(titre), x=0.01, font=dict(size=15, color=NOIR)), height=hauteur, margin=dict(l=50, r=20, t=50, b=45),
               paper_bgcolor="#fff", plot_bgcolor="#fff", font=dict(family="Segoe UI, Helvetica, Arial, sans-serif", size=12, color=NOIR),
               xaxis=dict(title=x_titre, gridcolor="#eaeef2", zeroline=False, showline=True, linecolor=GRIS_CLAIR),
               yaxis=dict(title=y_titre, gridcolor="#eaeef2", zeroline=False, showline=False),
               legend=dict(orientation="h", y=1.02, yanchor="bottom", x=1, xanchor="right", font=dict(size=11)), hovermode="closest")
    lay.update(kw)
    return go.Layout(**lay)


def _est_date(x):
    try:
        return pd.api.types.is_datetime64_any_dtype(pd.Series(list(x)[:5]))
    except Exception:
        return False


def montrer(fig):
    iplot(fig, show_link=False)


def g_courbes(x, series, titre, y_titre="", x_titre="", hauteur=360, hover=None, marqueurs=None, bandes=None, lignes_v=None, y_zero=False):
    """Courbes (series = [(nom, valeurs, couleur, style)]) ; marqueurs = [(nom, x, y, couleur)] points ; bandes = [(x0, x1, couleur, nom)] ;
    lignes_v = [(x, nom)]."""
    if PLOTLY:
        data = []
        for i, s_ in enumerate(series):
            nom, y = s_[0], s_[1]
            coul = s_[2] if len(s_) > 2 and s_[2] else SERIE[i % len(SERIE)]
            style = s_[3] if len(s_) > 3 else "solid"
            data.append(go.Scatter(x=list(x), y=list(y), mode="lines", name=nom, line=dict(color=coul, width=1.8 if style == "solid" else 1.4, dash=style),
                                   hovertemplate="%{x|%d/%m/%Y}<br>" + nom + " : %{y:,.0f}<extra></extra>" if hover is None else hover))
        for m in (marqueurs or []):
            data.append(go.Scatter(x=list(m[1]), y=list(m[2]), mode="markers", name=m[0], marker=dict(color=m[3], size=9, line=dict(color="#fff", width=1)),
                                   text=list(m[4]) if len(m) > 4 else None, hovertemplate="%{x|%d/%m/%Y}<br>%{text}<extra>" + m[0] + "</extra>" if len(m) > 4 else None))
        shapes = [dict(type="rect", xref="x", yref="paper", x0=b[0], x1=b[1], y0=0, y1=1, fillcolor=b[2], opacity=0.12, line=dict(width=0)) for b in (bandes or [])]
        shapes += [dict(type="line", xref="x", yref="paper", x0=l[0], x1=l[0], y0=0, y1=1, line=dict(color=NOIR, width=1, dash="dot")) for l in (lignes_v or [])]
        annos = [dict(x=l[0], y=1, yref="paper", text=l[1], showarrow=False, yanchor="bottom", font=dict(size=11)) for l in (lignes_v or [])]
        lay = _layout(titre, x_titre, y_titre, hauteur, shapes=shapes, annotations=annos)
        if _est_date(x):
            lay.xaxis.update(tickformat="%d/%m/%Y", hoverformat="%d/%m/%Y")
        if y_zero:
            lay.yaxis.update(rangemode="tozero")
        montrer(go.Figure(data=data, layout=lay))
    else:
        fig, ax = plt.subplots(figsize=(13, hauteur / 80.0))
        for i, s_ in enumerate(series):
            ax.plot(x, s_[1], label=s_[0], color=s_[2] if len(s_) > 2 and s_[2] else SERIE[i % len(SERIE)], linewidth=1.4,
                    linestyle="--" if len(s_) > 3 and s_[3] != "solid" else "-")
        for m in (marqueurs or []):
            ax.scatter(m[1], m[2], color=m[3], s=30, zorder=3, label=m[0])
        for b in (bandes or []):
            ax.axvspan(b[0], b[1], color=b[2], alpha=0.12)
        for l in (lignes_v or []):
            ax.axvline(l[0], color=NOIR, linewidth=1, linestyle=":")
        ax.set_title(titre, loc="left"); ax.set_ylabel(y_titre); ax.set_xlabel(x_titre); ax.grid(alpha=0.3); ax.legend(fontsize=8, frameon=False)
        plt.tight_layout(); plt.show()


def g_barres(cats, series, titre, y_titre="", x_titre="", hauteur=340, empile=False, horizontal=False, hover_suffixe="", ligne_y=None):
    """Barres : series = [(nom, valeurs, couleur)] ; empile=True pour empiler ; ligne_y = repère horizontal."""
    cats = [str(c) for c in cats]
    if PLOTLY:
        data = []
        for i, s_ in enumerate(series):
            coul = s_[2] if len(s_) > 2 and s_[2] else SERIE[i % len(SERIE)]
            vals = list(s_[1])
            if horizontal:
                data.append(go.Bar(y=cats, x=vals, orientation="h", name=s_[0], marker=dict(color=coul), hovertemplate="%{y}<br>%{x:,.1f}" + hover_suffixe + "<extra>" + s_[0] + "</extra>"))
            else:
                data.append(go.Bar(x=cats, y=vals, name=s_[0], marker=dict(color=coul), hovertemplate="%{x}<br>%{y:,.1f}" + hover_suffixe + "<extra>" + s_[0] + "</extra>"))
        shapes = [dict(type="line", xref="paper", yref="y", x0=0, x1=1, y0=ligne_y, y1=ligne_y, line=dict(color=GRIS, width=1, dash="dash"))] if ligne_y is not None else []
        lay = _layout(titre, x_titre, y_titre, hauteur, barmode="stack" if empile else "group", bargap=0.25, shapes=shapes, showlegend=len(series) > 1)
        if len(series) > 3:          # beaucoup de séries : la légende passe sous le graphique, pour ne pas couvrir le titre
            lay.legend.update(y=-0.12, yanchor="top", x=0, xanchor="left")
            lay.margin.update(b=110)
            lay.height = hauteur + 60
        montrer(go.Figure(data=data, layout=lay))
    else:
        fig, ax = plt.subplots(figsize=(12, hauteur / 80.0))
        bas = np.zeros(len(cats))
        larg = 0.8 / (1 if empile else len(series))
        for i, s_ in enumerate(series):
            coul = s_[2] if len(s_) > 2 and s_[2] else SERIE[i % len(SERIE)]
            vals = np.asarray(list(s_[1]), dtype=float)
            if horizontal:
                ax.barh(cats, vals, left=bas if empile else None, color=coul, label=s_[0])
            else:
                pos = np.arange(len(cats)) + (0 if empile else (i - (len(series) - 1) / 2.0) * larg)
                ax.bar(pos, vals, width=larg, bottom=bas if empile else None, color=coul, label=s_[0])
                ax.set_xticks(np.arange(len(cats))); ax.set_xticklabels(cats, rotation=45 if len(cats) > 8 else 0, ha="right" if len(cats) > 8 else "center", fontsize=8)
            if empile:
                bas = bas + np.nan_to_num(vals)
        if ligne_y is not None:
            ax.axhline(ligne_y, color=GRIS, linestyle="--", linewidth=1)
        ax.set_title(titre, loc="left"); ax.set_ylabel(y_titre); ax.set_xlabel(x_titre); ax.grid(alpha=0.3, axis="x" if horizontal else "y")
        if len(series) > 1:
            ax.legend(fontsize=8, frameon=False)
        plt.tight_layout(); plt.show()


def g_histos(blocs, titre_general="", hauteur=320):
    """Plusieurs histogrammes côte à côte : blocs = [(titre, valeurs, libellé x)]."""
    n_ = len(blocs)
    if PLOTLY:
        try:
            from plotly.subplots import make_subplots
        except Exception:
            from plotly.tools import make_subplots
        fig = make_subplots(rows=1, cols=n_, subplot_titles=[b[0] for b in blocs])
        for i, b in enumerate(blocs):
            fig.add_trace(go.Histogram(x=list(pd.Series(b[1]).dropna()), nbinsx=40, marker=dict(color=BLEU), showlegend=False,
                                       hovertemplate=b[2] + " : %{x}<br>GAB : %{y}<extra></extra>"), row=1, col=i + 1)
            fig.layout["xaxis{}".format(i + 1 if i else "")].update(title=b[2], gridcolor="#eaeef2")
            fig.layout["yaxis{}".format(i + 1 if i else "")].update(title="nombre de GAB" if i == 0 else "", gridcolor="#eaeef2")
        fig.layout.update(height=hauteur, paper_bgcolor="#fff", plot_bgcolor="#fff", margin=dict(l=50, r=20, t=60, b=45), bargap=0.05,
                          font=dict(family="Segoe UI, Helvetica, Arial, sans-serif", size=12, color=NOIR), title=dict(text=titre_general, x=0.01))
        montrer(fig)
    else:
        fig, axes = plt.subplots(1, n_, figsize=(4.4 * n_, hauteur / 80.0))
        for ax, b in zip(np.atleast_1d(axes), blocs):
            ax.hist(pd.Series(b[1]).dropna(), bins=40, color=BLEU); ax.set_title(b[0], fontsize=10); ax.set_xlabel(b[2]); ax.grid(alpha=0.3)
        np.atleast_1d(axes)[0].set_ylabel("nombre de GAB")
        plt.tight_layout(); plt.show()


FRANCE = [(2.54, 51.09), (3.15, 50.78), (3.65, 50.48), (4.15, 50.25), (4.8, 50.15), (4.85, 49.8), (5.5, 49.55), (6.15, 49.5), (6.75, 49.15), (7.6, 49.05), (8.2, 48.95),
          (7.8, 48.5), (7.55, 47.85), (7.55, 47.55), (6.95, 47.5), (6.15, 46.95), (6.05, 46.4), (6.8, 46.4), (6.95, 45.95), (7.15, 45.85), (6.8, 45.6), (7.15, 45.25),
          (6.65, 45.0), (7.0, 44.7), (6.85, 44.3), (7.5, 43.85), (7.0, 43.55), (6.6, 43.15), (5.9, 43.1), (5.2, 43.35), (4.6, 43.4), (3.9, 43.5), (3.25, 43.25),
          (3.05, 42.9), (3.15, 42.45), (2.55, 42.35), (1.75, 42.55), (1.35, 42.7), (0.65, 42.7), (-0.3, 42.8), (-1.0, 43.05), (-1.75, 43.35), (-1.5, 43.6), (-1.25, 44.4),
          (-1.2, 45.0), (-1.05, 45.6), (-1.2, 46.15), (-2.1, 46.8), (-2.2, 47.15), (-2.5, 47.3), (-3.1, 47.5), (-4.15, 47.8), (-4.4, 47.95), (-4.75, 48.35), (-4.5, 48.65),
          (-3.6, 48.75), (-2.8, 48.55), (-2.0, 48.65), (-1.6, 48.65), (-1.6, 49.2), (-1.9, 49.7), (-1.25, 49.7), (-1.1, 49.35), (0.1, 49.45), (0.3, 49.75), (1.4, 50.1),
          (1.6, 50.4), (1.6, 50.85), (2.54, 51.09)]
CORSE = [(9.4, 43.0), (9.55, 42.6), (9.4, 41.9), (9.25, 41.4), (8.8, 41.55), (8.6, 41.9), (8.55, 42.35), (8.75, 42.6), (9.3, 42.7), (9.4, 43.0)]


def g_points(x, y, titre, textes=None, couleurs=None, tailles=None, x_titre="", y_titre="", hauteur=520, groupes=None, aspect=None, axes=True):
    """Nuage de points avec infobulle (textes). groupes = [(nom, x, y, textes, couleur, taille)] tracés par-dessus un fond gris."""
    if PLOTLY:
        data = [] if not len(list(x)) else [go.Scattergl(x=list(x), y=list(y), mode="markers", name="tous les GAB", text=list(textes) if textes is not None else None,
                             marker=dict(color=couleurs if couleurs is not None else GRIS_CLAIR, size=tailles if tailles is not None else 4, opacity=0.85,
                                         colorscale="Blues" if couleurs is not None and not isinstance(couleurs, str) else None,
                                         showscale=couleurs is not None and not isinstance(couleurs, str), colorbar=dict(title="", thickness=10, len=0.5)),
                             hovertemplate="%{text}<extra></extra>" if textes is not None else None)]
        carte = bool(aspect) and not axes
        if carte:                    # une carte : le contour de la France en fond, sous les points
            data = [go.Scatter(x=[p_[0] for p_ in poly], y=[p_[1] for p_ in poly], mode="lines", fill="toself", fillcolor="#f3f5f8", line=dict(color="#b9c2cc", width=1),
                               hoverinfo="skip", showlegend=False) for poly in (FRANCE, CORSE)] + data
        for gnom, gx, gy, gt, gc, gs in (groupes or []):
            data.append(go.Scattergl(x=list(gx), y=list(gy), mode="markers", name=gnom, text=list(gt), hovertemplate="%{text}<extra></extra>",
                                     marker=dict(color=gc, size=gs, line=dict(color="#fff", width=1))))
        lay = _layout(titre, x_titre, y_titre, hauteur, showlegend=bool(groupes))
        if aspect:
            lay.yaxis.update(scaleanchor="x", scaleratio=aspect)
        if not axes:
            lay.xaxis.update(visible=False)
            lay.yaxis.update(visible=False)
        if carte:
            lay.xaxis.update(range=[-5.3, 9.8])
            lay.yaxis.update(range=[41.2, 51.3])
            lay.plot_bgcolor = "#fff"
            lay.legend.update(bgcolor="rgba(255,255,255,0.8)")
        montrer(go.Figure(data=data, layout=lay))
    else:
        fig, ax = plt.subplots(figsize=(8, hauteur / 80.0))
        if aspect and not axes:
            for poly in (FRANCE, CORSE):
                ax.fill([p_[0] for p_ in poly], [p_[1] for p_ in poly], color="#f3f5f8", edgecolor="#b9c2cc", linewidth=0.8, zorder=0)
        if len(list(x)):
            ax.scatter(x, y, s=5, c=couleurs if couleurs is not None else GRIS_CLAIR, cmap="Blues" if couleurs is not None and not isinstance(couleurs, str) else None, alpha=0.8)
        for gnom, gx, gy, gt, gc, gs in (groupes or []):
            ax.scatter(gx, gy, s=gs * 6, c=gc, label=gnom, edgecolors="#fff", zorder=3)
        if aspect:
            ax.set_aspect(aspect)
        if not axes:
            ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(titre, loc="left"); ax.set_xlabel(x_titre); ax.set_ylabel(y_titre); ax.grid(alpha=0.3)
        if groupes:
            ax.legend(fontsize=8, frameon=False)
        plt.tight_layout(); plt.show()

<a id="essentiel"></a>
## L'essentiel pour le métier
**Tout ce qu'il faut savoir, sur une page**, pour le dernier jour complet de la fiche :

1. **le tableau de bord du jour** : GAB actifs, retraits, montant, retraits COS, retraits échoués, cartes avalées, cartes étrangères ;
2. **les 3 GAB en tête** : le plus et le moins de retraits, le plus d'argent retiré, le plus de retraits échoués et de cartes avalées, le
   plus de cartes UnionPay, JCB et Diners ;
3. **les 15 GAB les plus atypiques**, chacun avec son diagnostic, **les chiffres qui le justifient** et l'action conseillée ;
4. **la carte de France** : en rouge les GAB atypiques, en jaune ceux à surveiller, en gris les autres ;
5. **le fichier Excel** à télécharger.

Cette page se remplit automatiquement à la fin du calcul. Le reste du notebook explique et détaille.

In [ ]:
ESSENTIEL = display(HTML(CSS + '<div class="g-note warn">Calcul en cours : cette page se remplit à la fin du notebook.</div>'), display_id=True)

<a href="#sommaire" style="font-size:12px">↑ retour au sommaire</a>
<a id="s1"></a>
## 1. Réglages
**Pour la personne qui lance le notebook.** Les lecteurs du métier peuvent passer directement à la partie A.

On y indique la fiche à lire, les GAB et les dates des trois cas présentés, et le nom du fichier Excel produit. Rien d'autre n'est à modifier.

In [ ]:
DATASET_ENTREE = "fiche_gab_jour"        # fiche GAB x jour, ou "cassettes_gab_heure" (recette horaire)
DATASET_VOISINS = "voisins_gab"          # recette voisins_gab.sql (num_automate, voisin, distance_km) ; None = calcul ici
JOURS_A_EXAMINER = "tout"                # "tout" = toute la fiche (3 ans : compter 15 à 30 min) ; ou un nombre de jours (ex. 90)
ECARTER_FIN_INCOMPLETE = True            # écarter les derniers jours où la fiche n'a visiblement pas reçu toutes les opérations
DEBUT_JOURNEE_HEURE = 12                 # recette horaire seulement : heure de début d'une « journée » (0 = minuit)

TOP_LISTE = 15                           # GAB listés chaque jour (du plus au moins atypique)
TOP_PRIORITAIRE = 10                     # les N premiers sont marqués « prioritaire »
RAYON_VOISINS_KM = 3.0                   # si les voisins sont calculés ici : GAB à moins de N km
NB_VOISINS = 5                           # si les voisins sont calculés ici : au plus N voisins par GAB

# Sorties (le notebook les crée ; rien à fournir)
NOM_EXCEL_SORTIE = "photo_parc_gab.xlsx" # nom de l'Excel de synthèse généré en fin de notebook
JOURS_PHOTO_COMPLETE_EXCEL = "max"       # fichiers « photo GAB x jour » : jours par fichier ; "max" = le plus possible dans la limite Excel
NB_FICHIERS_PHOTO = 1                    # nombre de fichiers photo, du plus récent au plus ancien (7 pour couvrir 3 ans)
NOM_DOSSIER_DATAIKU = None               # ex. "exports_gab" : dossier géré Dataiku où déposer l'Excel et la photo complète
DATASET_SORTIE = None                    # ex. "photo_parc_jour" : la photo du dernier jour (une ligne par GAB)
DATASET_SORTIE_KPI = None                # ex. "kpi_parc_jour" : les indicateurs du dernier jour
DATASET_SORTIE_HISTO = None              # ex. "photo_parc_histo" : la photo GAB x jour de toute la période

# Les trois cas de la présentation (sections 14 à 16) et les records des réseaux étrangers (section 17).
# Les GAB du cas 2 viennent du suivi vandalisme : les saisir ici seulement, jamais dans un fichier partagé.
CAS_INVERSION = {"gab": "000000", "date": "2026-01-01"}                  # GAB et jour de l'inversion de cassettes confirmée
CAS_MANIFESTATION = {"date": "2026-01-01",                               # premier jour des manifestations
                     "jours_avant": 7, "jours_apres": 21,                 # période suivie autour
                     "gab": [],                                           # GAB dégradés connus, ex. ["123456", "234567"]
                     "codes_postaux": []}                                 # zone suivie (débuts de code postal), ex. ["75"] ; [] = toute la France
CAS_EXOTIQUE = {"date": "2026-01-01", "reseau": "cup"}                    # rafale de cartes étrangères : "cup" (UnionPay), "amex", "diners", "jcb", "trionis"
TOP_RECORDS = 10                                                          # nombre de records affichés par réseau étranger
JOUR_A_REJOUER = None                                                     # jour à rejouer en section 12 (ex. "AAAA-MM-JJ") ; None = jour de l'inversion

# Cas connus, utilisés uniquement pour VÉRIFIER la méthode (jamais pour la régler).
# "gab" : num_automate exact, ou début du code site si la colonne code_site_implantation existe ;
# "date" : jour de l'incident ; "avant" / "apres" : jours affichés autour (défaut 3 et 2) ; "quoi" : libellé.
CAS_TESTS = [
    {"gab": "000000", "date": "2026-01-01", "quoi": "inversion de cassettes 10 € / 20 €"},   # <- numéro et date du cas confirmé
    # {"gab": "......", "date": "AAAA-MM-JJ", "avant": 7, "apres": 21, "quoi": "GAB dégradé"},
    # {"gab": "......", "date": "AAAA-MM-JJ", "quoi": "afflux de cartes UnionPay"},
]
CAS_TEST = CAS_TESTS[0] if CAS_TESTS else None      # compatibilité : premier cas

# Partie A — Comprendre le parc avant de chercher les anomalies
On ne repère un comportement anormal qu'en sachant d'abord ce qui est normal. Cette partie décrit le parc sur les 3 ans : combien les GAB
servent, quand, avec quelles cartes, et comment chacun a son propre rythme.

<a href="#sommaire" style="font-size:12px">↑ retour au sommaire</a>
<a id="s2"></a>
## 2. Jusqu'à quelle date les données sont-elles fiables ?
**En bref** : avant d'analyser une journée, on vérifie qu'elle a été entièrement chargée.

**Pourquoi** : les opérations d'une journée arrivent progressivement. Si l'analyse tourne trop tôt, la dernière journée paraît creuse et des
centaines de GAB sembleraient à tort en panne.

**Ce que vous voyez** : la date du dernier jour complet, le retard des données sur aujourd'hui, et une courbe du nombre de GAB qui ont servi
chaque jour. Un jour où ce nombre chute nettement est marqué en orange : il n'est pas jugé.

<details><summary><i>Comment c'est calculé (pour les curieux)</i></summary>

Pour chaque jour, on compte les GAB qui ont fait au moins un retrait et on compare à la valeur typique des 4 mêmes jours de semaine précédents. Sous 90 %, le jour est déclaré incomplet. Les derniers jours incomplets sont écartés de l'analyse.

</details>

In [ ]:
# toute la fiche : toutes les colonnes (~190) et toutes les lignes (~5,5 millions de GAB-jours sur 3 ans, ~10 Go en mémoire)
_t0 = pd.Timestamp.now()
_ds = dataiku.Dataset(DATASET_ENTREE)
brut = _ds.get_dataframe()
brut.columns = [c.lower() for c in brut.columns]
_presentes = list(brut.columns)
EST_HORAIRE = "heure" in brut.columns
obligatoires = ["num_automate", "date_jour", "nb_retraits", "montant_total", "nb_retraits_repetes_10min"]
manquantes = [c for c in obligatoires if c not in brut.columns]
PAR_POSITION = all("nb_billets_pos{}".format(k) in brut.columns and "val_max_pos{}".format(k) in brut.columns for k in range(1, 5))
PAR_VALEUR = all("nb_billets_{}".format(v) in brut.columns for v in (10, 20, 50))
if not PAR_POSITION and not PAR_VALEUR:
    manquantes.append("nb_billets_10/20/50 (fiche) ou nb_billets_pos1..4 + val_max_pos1..4 (recette horaire)")
if manquantes:
    raise ValueError("Colonnes absentes de {} : {}".format(DATASET_ENTREE, manquantes))
if PAR_POSITION and not PAR_VALEUR:
    for k in range(1, 5):
        brut["nb_billets_pos{}".format(k)] = pd.to_numeric(brut["nb_billets_pos{}".format(k)], errors="coerce").fillna(0)
        brut["val_max_pos{}".format(k)] = pd.to_numeric(brut["val_max_pos{}".format(k)], errors="coerce")
    for v in (5, 10, 20, 50, 100):
        brut["nb_billets_{}".format(v)] = sum(np.where(brut["val_max_pos{}".format(k)] == v, brut["nb_billets_pos{}".format(k)], 0) for k in range(1, 5))
    brut["plan_cassettes"] = brut[["val_max_pos{}".format(k) for k in range(1, 5)]].fillna(0).astype(int).astype(str).apply("/".join, axis=1)
if "plan_cassettes" not in brut.columns:
    brut["plan_cassettes"] = "?"
for v in (5, 100):
    if "nb_billets_{}".format(v) not in brut.columns:
        brut["nb_billets_{}".format(v)] = 0

brut["date_jour"] = pd.to_datetime(brut["date_jour"].astype(str).str[:10], errors="coerce")
_avant = len(brut)
brut = brut[brut["date_jour"].notnull()].copy()
if len(brut) < _avant:
    note("{} ligne(s) écartée(s) : date illisible.".format(nb(_avant - len(brut))), "warn")
brut["num_automate"] = brut["num_automate"].astype(str)
NUMERIQUES = [c for c in brut.columns if c.startswith(("nb_", "montant_", "encaisse_", "somme_", "heure_")) or c in ("plus_long_trou_min", "baisse_encaisse_retraits")]
for c in NUMERIQUES:
    brut[c] = pd.to_numeric(brut[c], errors="coerce")
brut["nb_retraits"] = brut["nb_retraits"].fillna(0)
# heures HHMMSS (ex. 093000) -> heures décimales (9,5) ; les colonnes « heure_pic_* » sont déjà des heures entières
for c in ["heure_premier_retrait", "heure_dernier_retrait", "heure_premier_chargement"]:
    if c in brut.columns:
        _v = brut[c]
        brut[c] = (_v // 10000) + ((_v // 100) % 100) / 60.0
if EST_HORAIRE:
    brut["heure"] = pd.to_numeric(brut["heure"], errors="coerce")
    brut = brut[brut["heure"].notnull()].copy()
    ts = brut["date_jour"] + pd.to_timedelta(brut["heure"], unit="h")
    brut["date_jour"] = (ts - pd.Timedelta(hours=DEBUT_JOURNEE_HEURE)).dt.normalize()
    derniere_heure = ts.max()
    fin_complete = (derniere_heure - pd.Timedelta(hours=DEBUT_JOURNEE_HEURE + 23)).normalize()
    if brut["date_jour"].max() > fin_complete:
        note("Dernière journée incomplète écartée (données jusqu'au {}).".format(derniere_heure.strftime("%d/%m/%Y %Hh")), "warn")
        brut = brut[brut["date_jour"] <= fin_complete].copy()
if "code_site_implantation" in brut.columns:
    brut["code_site_implantation"] = brut["code_site_implantation"].astype(str).str.replace(r"\.0$", "", regex=True)

# --- Couverture : GAB actifs par jour face aux 4 mêmes jours de semaine précédents
FIN_FICHE = brut["date_jour"].max()
DEBUT_FICHE = brut["date_jour"].min()
AUJOURD_HUI = pd.Timestamp.today().normalize()
actifs_jour = brut[brut["nb_retraits"] > 0].groupby("date_jour")["num_automate"].nunique().reindex(pd.date_range(DEBUT_FICHE, FIN_FICHE, freq="D")).fillna(0)
ref_actifs = pd.Series({d_: np.nanmedian([actifs_jour.get(d_ - pd.Timedelta(days=7 * k_), np.nan) for k_ in range(1, 5)]) for d_ in actifs_jour.index})
taux_couverture = (actifs_jour / ref_actifs).where(ref_actifs.notnull())
jours_incomplets = taux_couverture[taux_couverture < 0.9].index
# derniers jours incomplets : on remonte depuis la fin tant que le jour est incomplet (au plus 7 jours)
FIN = FIN_FICHE
ecartes = []
if ECARTER_FIN_INCOMPLETE:
    while FIN in jours_incomplets and len(ecartes) < 7:
        ecartes.append(FIN)
        FIN = FIN - pd.Timedelta(days=1)
    if ecartes:
        brut = brut[brut["date_jour"] <= FIN].copy()
# la série longue (toute la fiche, quelques colonnes) sert au calendrier du parc et au profil de chaque GAB
COLONNES_LONGUES = ["num_automate", "date_jour", "nb_retraits", "montant_total", "nb_inaboutis", "nb_captures", "nb_ret_exotique", "nb_0h_6h", "jour_sans_retrait",
                    "statut_gab", "type_gab_e_i", "nb_ret_cup", "nb_ret_amex", "nb_ret_diners", "nb_ret_jcb", "nb_ret_trionis",
                    "montant_cup", "montant_amex", "montant_diners", "montant_jcb", "montant_trionis", "code_postal", "ville"]
longue = brut[[c for c in COLONNES_LONGUES if c in brut.columns]].copy()
if isinstance(JOURS_A_EXAMINER, str) and JOURS_A_EXAMINER.lower() == "tout":
    NB_JOURS_EXAMINES = max(1, (FIN - DEBUT_FICHE).days + 1 - 70)
else:
    NB_JOURS_EXAMINES = int(JOURS_A_EXAMINER)
    brut = brut[brut["date_jour"] >= FIN - pd.Timedelta(days=NB_JOURS_EXAMINES + 70)].copy()   # 10 semaines d'habitude avant la période
DEBUT_PERIODE = FIN - pd.Timedelta(days=NB_JOURS_EXAMINES - 1)
NB_GAB = brut["num_automate"].nunique()

bandeau("Les données", "fiche {} — {}".format(DATASET_ENTREE, "recette horaire regroupée par jour" if EST_HORAIRE else "une ligne par GAB et par jour"))
tuiles([("GAB dans la fiche", nb(NB_GAB), "numéros d'automate distincts"),
        ("Jours dans la fiche", nb((FIN_FICHE - DEBUT_FICHE).days + 1), "du {} au {}".format(DEBUT_FICHE.strftime("%d/%m/%Y"), FIN_FICHE.strftime("%d/%m/%Y"))),
        ("Dernier jour complet", FIN.strftime("%d/%m/%Y"), "{} jour(s) écarté(s) en fin de fiche".format(len(ecartes)) if ecartes else "le dernier jour de la fiche est complet", "down" if ecartes else "up"),
        ("Retard de la fiche", "{} j".format((AUJOURD_HUI - FIN).days), "entre le dernier jour complet et aujourd'hui ({})".format(AUJOURD_HUI.strftime("%d/%m/%Y"))),
        ("Période examinée", nb(NB_JOURS_EXAMINES) + " j", "du {} au {}".format(DEBUT_PERIODE.strftime("%d/%m/%Y"), FIN.strftime("%d/%m/%Y"))),
        ("Lignes lues", nb(len(brut)), "toutes les colonnes ({}), lues en {:.0f} s".format(len(_presentes), (pd.Timestamp.now() - _t0).total_seconds()))])
if ecartes:
    _t = pd.DataFrame({"jour": [jfr(d_) for d_ in sorted(ecartes)],
                       "GAB ayant retiré": [int(actifs_jour[d_]) for d_ in sorted(ecartes)],
                       "attendus (4 mêmes jours de semaine)": [int(round(ref_actifs[d_])) for d_ in sorted(ecartes)],
                       "couverture": [pct(taux_couverture[d_], 0) for d_ in sorted(ecartes)]})
    note("<b>{} journée(s) écartée(s) en fin de fiche</b> : la fiche n'y contient qu'une partie des opérations (moins de 90 % des GAB habituellement actifs). "
         "Ce n'est pas une anomalie du parc, c'est la fiche qui a été calculée avant que toutes les opérations soient arrivées. "
         "L'analyse s'arrête au <b>{}</b>, dernier jour complet.".format(len(ecartes), FIN.strftime("%d/%m/%Y")), "warn")
    display(tableau(_t))
_autres = [d_ for d_ in jours_incomplets if d_ <= FIN]
if _autres:
    note("{} autre(s) jour(s) incomplet(s) à l'intérieur de la période (trous de collecte) : les silences et baisses de ces jours seront "
         "traités comme de l'information, pas comme des alertes. Dates : {}.".format(len(_autres), ", ".join(d_.strftime("%d/%m/%Y") for d_ in _autres[-10:])), "warn")
# le graphique : GAB actifs par jour sur les 12 dernières semaines, jours incomplets marqués
_fen = actifs_jour[actifs_jour.index >= FIN_FICHE - pd.Timedelta(days=84)]
_inc = [d_ for d_ in jours_incomplets if d_ in _fen.index]
g_courbes(_fen.index, [("GAB ayant fait au moins un retrait", _fen.values, BLEU), ("attendus (4 mêmes jours de semaine)", ref_actifs.reindex(_fen.index).values, GRIS, "dash")],
          "Couverture de la fiche sur les 12 dernières semaines : combien de GAB ont retiré chaque jour", y_titre="GAB actifs", hauteur=320, y_zero=True,
          marqueurs=[("jour incomplet", _inc, [_fen[d_] for d_ in _inc], "#e36209", ["{} : {} GAB actifs, attendus {}".format(d_.strftime("%d/%m"), int(_fen[d_]), int(round(ref_actifs[d_]))) for d_ in _inc])] if _inc else None)

### Ce que contient la fiche
La fiche décrit **chaque GAB, chaque jour**, avec environ 190 informations rangées par thème : volumes, montants, retraits échoués, cartes
avalées, heures, billets, argent dans le GAB, types de cartes, qualité. Le tableau indique combien d'informations chaque thème apporte et
dans quelles sections du notebook elles servent. Toutes sont utilisées.

In [ ]:
GROUPES_FICHE = [("Identité et calendrier", ("num_automate", "date_jour", "jour_", "code_", "ville", "latitude", "longitude", "statut", "type_", "date_", "fin_donnees", "week_end", "geopoint"), "2, 6, 7, 9"),
                 ("Volumes", ("nb_lignes", "nb_retraits", "nb_cartes_distinctes"), "3, 4, 8, 10"),
                 ("Montants", ("montant_total", "montant_max", "montant_min", "montant_somme", "nb_montant_", "montant_retraits_"), "5, 10, 17"),
                 ("Inaboutis et billets incohérents", ("nb_inab", "nb_billets_inferieurs", "nb_billets_superieurs", "montant_non_distribue", "montant_sur_distribue", "nb_retraits_sans_info", "montant_billets_inab", "montant_inabouti", "somme_heures_inab"), "7 bis, 10, 14"),
                 ("Captures de carte", ("nb_cap", "nb_cartes_capturees", "somme_heures_capt"), "5, 10"),
                 ("Forme de la journée", ("nb_0h", "nb_6h", "nb_9h", "nb_12h", "nb_14h", "nb_18h", "nb_22h", "nb_nuit", "heure_", "nb_heures_actives", "plus_long_trou", "somme_heures_retraits", "nb_ope_heure", "montant_heure_pic"), "5, 7 bis, 10, 15"),
                 ("Pics horaires par type", ("_heure_pic",), "10"),
                 ("Billets et cassettes", ("nb_billets_", "plan_cassettes", "nb_plans", "nb_retraits_avec_", "montant_billets_distribues", "nb_retraits_100plus"), "5, 7 bis, 10, 14"),
                 ("Encaisse et chargements", ("encaisse", "nb_recharges", "montant_recharge", "montant_max_chargement", "baisse_encaisse", "nb_incoherences"), "7 bis, 10, 14, 15"),
                 ("Réseaux de carte", ("nb_ret_", "montant_interne", "montant_cb", "montant_visa", "montant_mastercard", "montant_cos", "montant_amex", "montant_cup", "montant_diners", "montant_jcb", "montant_autre", "montant_trionis", "montant_exotique", "montant_etranger"), "5, 7 bis, 10, 16, 17"),
                 ("Qualité des données", ("nb_retraits_montant_", "nb_heure_invalide", "nb_retraits_sans_carte", "nb_billets_valeur_inconnue"), "7 bis, 10")]
_reste = list(_presentes)
_lg = []
for nom_g, prefixes, sections in GROUPES_FICHE[::-1]:          # du plus spécifique au plus général
    cols_g = [c for c in _reste if any(c.startswith(p) or p in c for p in prefixes)]
    _reste = [c for c in _reste if c not in cols_g]
    _lg.append({"groupe": nom_g, "colonnes": len(cols_g), "utilisées en section": sections, "exemples": ", ".join(cols_g[:6]) + (" …" if len(cols_g) > 6 else "")})
_lg = pd.DataFrame(_lg[::-1])
if _reste:
    _lg = pd.concat([_lg, pd.DataFrame([{"groupe": "Autres", "colonnes": len(_reste), "utilisées en section": "description", "exemples": ", ".join(_reste[:6])}])], ignore_index=True)
display(tableau(_lg, "Les {} colonnes de la fiche, par groupe".format(len(_presentes)), petites=("exemples",)))

for c in longue.columns:
    if c.startswith("nb_") or c in ("montant_total", "jour_sans_retrait"):
        longue[c] = pd.to_numeric(longue[c], errors="coerce").fillna(0)
if "jour_sans_retrait" not in longue.columns:
    longue["jour_sans_retrait"] = (longue["nb_retraits"] == 0).astype(float)
if EST_HORAIRE:
    longue = longue.groupby(["num_automate", pd.Grouper(key="date_jour", freq="D")], as_index=False).sum(numeric_only=True)
longue["jour_semaine"] = longue["date_jour"].dt.dayofweek + 1
longue["mois"] = longue["date_jour"].dt.month
longue["annee"] = longue["date_jour"].dt.year
JOURS_FR = ["lundi", "mardi", "mercredi", "jeudi", "vendredi", "samedi", "dimanche"]
MOIS_FR = ["janv.", "févr.", "mars", "avr.", "mai", "juin", "juil.", "août", "sept.", "oct.", "nov.", "déc."]
note("Série longue : <b>{}</b> GAB, du {} au {}, soit <b>{}</b> GAB-jours (un GAB-jour = un GAB, un jour).".format(
    nb(longue["num_automate"].nunique()), longue["date_jour"].min().strftime("%d/%m/%Y"), longue["date_jour"].max().strftime("%d/%m/%Y"), nb(len(longue))))

<a href="#sommaire" style="font-size:12px">↑ retour au sommaire</a>
<a id="s3"></a>
## 3. Le parc en chiffres : combien de GAB, combien de retraits ?
**En bref** : les grands chiffres du parc sur toute la période.

**Ce que vous voyez** : des tuiles avec les chiffres clés, puis trois graphiques où **chaque barre compte des GAB**. Ils montrent que les
GAB sont très différents : certains font 30 retraits par jour, d'autres 300.

**À quoi ça sert** : c'est pour cela que chaque GAB est toujours comparé **à lui-même**, jamais à une moyenne du parc.

In [ ]:
par_gab = longue.groupby("num_automate").agg(jours=("date_jour", "size"), retraits_par_jour=("nb_retraits", "mean"),
                                             montant_par_jour=("montant_total", "mean"), jours_a_zero=("jour_sans_retrait", "mean"),
                                             inaboutis_par_jour=("nb_inaboutis", "mean"), captures_par_jour=("nb_captures", "mean"))
_tot_ret = longue["nb_retraits"].sum()
bandeau("Le parc en chiffres", "toute la fiche, du {} au {}".format(longue["date_jour"].min().strftime("%d/%m/%Y"), FIN.strftime("%d/%m/%Y")))
tuiles([("GAB", nb(longue["num_automate"].nunique()), "présents au moins un jour"),
        ("Retraits", nb(_tot_ret), "{} par jour pour le parc".format(nb(longue.groupby("date_jour")["nb_retraits"].sum().mean()))),
        ("Montant retiré", nb(longue["montant_total"].sum() / 1e6, 1) + " M€", "{} € par retrait en moyenne".format(nb(longue["montant_total"].sum() / max(1, _tot_ret), 1))),
        ("Retraits par GAB et par jour", nb(par_gab["retraits_par_jour"].median(), 1), "médiane ; 10 % des GAB sous {}, 10 % au-dessus de {}".format(
            nb(par_gab["retraits_par_jour"].quantile(0.1), 1), nb(par_gab["retraits_par_jour"].quantile(0.9), 1))),
        ("GAB-jours sans retrait", pct(longue["jour_sans_retrait"].mean()), "{} des GAB ne sont jamais à 0".format(pct((par_gab["jours_a_zero"] == 0).mean(), 0))),
        ("Inaboutis", nb(1000 * longue["nb_inaboutis"].sum() / max(1, _tot_ret), 1) + " ‰", "pour 1 000 retraits"),
        ("Cartes capturées", nb(longue["nb_captures"].sum() / max(1, len(longue)), 2), "par GAB et par jour")])
_rep = []
for col, lib in [("statut_gab", "statut"), ("type_gab_e_i", "type (I = intérieur, E = extérieur)")]:
    if col in longue.columns:
        _s = longue.drop_duplicates("num_automate")[col].fillna("inconnu").astype(str).value_counts()
        _rep += [{"critère": lib, "valeur": k, "GAB": int(v), "part du parc": pct(v / _s.sum(), 1)} for k, v in _s.items()]
if _rep:
    display(tableau(pd.DataFrame(_rep), "Répartition des GAB"))
g_histos([("Retraits par jour et par GAB", par_gab["retraits_par_jour"].clip(upper=par_gab["retraits_par_jour"].quantile(0.99)), "retraits / jour"),
          ("Part de jours sans retrait, par GAB", par_gab["jours_a_zero"] * 100, "% de jours à 0"),
          ("Montant par jour et par GAB", par_gab["montant_par_jour"].clip(upper=par_gab["montant_par_jour"].quantile(0.99)), "€ / jour")],
         "Les GAB ne se ressemblent pas : chaque barre compte des GAB")
retenir(["Le GAB médian fait <b>{}</b> retraits par jour ; entre les 10 % les moins actifs ({}) et les 10 % les plus actifs ({}), le rapport est de 1 à {}.".format(
             nb(par_gab["retraits_par_jour"].median(), 1), nb(par_gab["retraits_par_jour"].quantile(0.1), 1), nb(par_gab["retraits_par_jour"].quantile(0.9), 1),
             nb(par_gab["retraits_par_jour"].quantile(0.9) / max(0.1, par_gab["retraits_par_jour"].quantile(0.1)), 0)),
         "<b>{}</b> des GAB-jours sont sans retrait : un jour à 0 n'est pas rare en soi, tout dépend du GAB (certains ne sont jamais à 0, d'autres un jour sur trois).".format(pct(longue["jour_sans_retrait"].mean())),
         "Inaboutis et captures sont rares (<b>{}</b> inaboutis pour 1 000 retraits, <b>{}</b> capture par GAB et par jour) : on les jugera en probabilité, pas en écart.".format(
             nb(1000 * longue["nb_inaboutis"].sum() / max(1, _tot_ret), 1), nb(longue["nb_captures"].sum() / max(1, len(longue)), 2))])

<a href="#sommaire" style="font-size:12px">↑ retour au sommaire</a>
<a id="s4"></a>
## 4. Quand retire-t-on ? Les rythmes de la semaine et de l'année
**En bref** : le parc a des rythmes réguliers. Le vendredi et le samedi sont forts, le dimanche faible, l'été et les fêtes changent tout.

**Ce que vous voyez** : les retraits jour par jour sur toute la période, puis l'effet du jour de semaine, du mois et des jours fériés.
L'**indice** se lit simplement : 1 = un jour ordinaire, 1,2 = 20 % de retraits en plus, 0,7 = 30 % en moins.

**À quoi ça sert** : une baisse de 30 % un dimanche d'août n'a rien d'anormal. La détection tient compte de ces rythmes avant d'alerter.

In [ ]:
parc = longue.groupby("date_jour").agg(retraits=("nb_retraits", "sum"), gab_actifs=("nb_retraits", lambda s: (s > 0).sum()),
                                       inaboutis=("nb_inaboutis", "sum"), captures=("nb_captures", "sum")).reset_index()
parc["moyenne_28j"] = parc["retraits"].rolling(28, center=True, min_periods=10).mean()
parc["indice"] = parc["retraits"] / parc["moyenne_28j"]
g_courbes(parc["date_jour"], [("retraits du parc", parc["retraits"], BLEU_CLAIR), ("moyenne glissante 28 jours", parc["moyenne_28j"], BLEU)],
          "Retraits du parc, jour par jour, sur toute la fiche", y_titre="retraits", hauteur=360)
g_courbes(parc["date_jour"], [("GAB ayant fait au moins un retrait", parc["gab_actifs"], BLEU)], "Taille du parc actif, jour par jour", y_titre="GAB actifs", hauteur=260, y_zero=True)
extremes = pd.concat([parc.nsmallest(5, "indice").assign(sens="jours les plus creux"), parc.nlargest(5, "indice").assign(sens="jours les plus forts")])
extremes = extremes[["sens", "date_jour", "retraits", "gab_actifs", "indice"]].copy()
extremes["date_jour"] = extremes["date_jour"].map(jfr)
display(tableau(extremes, "Les 5 jours les plus creux et les 5 plus forts du parc (indice = retraits / moyenne 28 jours)",
                libelles={"sens": "", "date_jour": "jour", "retraits": "retraits", "gab_actifs": "GAB actifs", "indice": "indice"}, dec=2))

In [ ]:
moyenne_parc = parc["retraits"].mean()
jds = longue.groupby("jour_semaine")["nb_retraits"].sum() / longue.groupby("jour_semaine")["date_jour"].nunique() / moyenne_parc
mois_ = longue.groupby(["annee", "mois"])["nb_retraits"].sum() / longue.groupby(["annee", "mois"])["date_jour"].nunique() / moyenne_parc
g_barres([JOURS_FR[d - 1] for d in jds.index], [("indice", jds.values.round(2), BLEU)], "Effet du jour de semaine (1 = jour moyen)", y_titre="indice", hauteur=280, ligne_y=1)
g_courbes(pd.to_datetime(["{}-{:02d}-15".format(a, m) for a, m in mois_.index]), [("indice du mois", mois_.values.round(2), BLEU)],
          "Effet du mois (1 = jour moyen) : la saisonnalité du parc", y_titre="indice", hauteur=280,
          hover="%{x|%b %Y}<br>indice : %{y:.2f}<extra></extra>")


def feries_france(annees):
    """Les 11 jours fériés français, calculés sans bibliothèque (Pâques par l'algorithme de Meeus)."""
    out = {}
    for an in annees:
        a_, b_, c_ = an % 19, an // 100, an % 100
        d_, e_ = b_ // 4, b_ % 4
        f_ = (b_ + 8) // 25
        g_ = (b_ - f_ + 1) // 3
        h_ = (19 * a_ + b_ - d_ - g_ + 15) % 30
        i_, k_ = c_ // 4, c_ % 4
        l_ = (32 + 2 * e_ + 2 * i_ - h_ - k_) % 7
        m_ = (a_ + 11 * h_ + 22 * l_) // 451
        mois_p, jour_p = (h_ + l_ - 7 * m_ + 114) // 31, (h_ + l_ - 7 * m_ + 114) % 31 + 1
        paques = pd.Timestamp(an, mois_p, jour_p)
        out.update({pd.Timestamp(an, 1, 1): "jour de l'an", paques + pd.Timedelta(days=1): "lundi de Pâques", pd.Timestamp(an, 5, 1): "1er mai",
                    pd.Timestamp(an, 5, 8): "8 mai", paques + pd.Timedelta(days=39): "Ascension", paques + pd.Timedelta(days=50): "lundi de Pentecôte",
                    pd.Timestamp(an, 7, 14): "14 juillet", pd.Timestamp(an, 8, 15): "15 août", pd.Timestamp(an, 11, 1): "Toussaint",
                    pd.Timestamp(an, 11, 11): "11 novembre", pd.Timestamp(an, 12, 25): "Noël"})
    return out


FERIES = feries_france(sorted(set(longue["annee"].unique().tolist() + [int(longue["annee"].max()) + 1])))
try:
    import holidays as _hol
    FERIES.update({pd.Timestamp(d_): n_ for d_, n_ in _hol.France(years=sorted(longue["annee"].unique())).items()})
except Exception:
    pass
parc["ferie"] = parc["date_jour"].isin(list(FERIES))
parc["veille_ferie"] = (parc["date_jour"] + pd.Timedelta(days=1)).isin(list(FERIES))
eff = pd.DataFrame({"type de jour": ["jour férié", "veille de férié", "jour ordinaire"],
                    "indice (retraits / moyenne 28 jours)": [parc.loc[parc["ferie"], "indice"].mean(), parc.loc[parc["veille_ferie"], "indice"].mean(),
                                                            parc.loc[~parc["ferie"] & ~parc["veille_ferie"], "indice"].mean()],
                    "jours": [int(parc["ferie"].sum()), int(parc["veille_ferie"].sum()), int((~parc["ferie"] & ~parc["veille_ferie"]).sum())]})
display(tableau(eff, "Effet des jours fériés", dec=2))
_jmin, _jmax = jds.idxmin(), jds.idxmax()
_mmin, _mmax = mois_.idxmin(), mois_.idxmax()
retenir(["Jour de semaine : le <b>{}</b> est le jour le plus fort (indice {}), le <b>{}</b> le plus faible ({}).".format(JOURS_FR[_jmax - 1], nb(jds[_jmax], 2), JOURS_FR[_jmin - 1], nb(jds[_jmin], 2)),
         "Mois : le plus fort est <b>{} {}</b> ({}), le plus faible <b>{} {}</b> ({}).".format(MOIS_FR[_mmax[1] - 1], _mmax[0], nb(mois_[_mmax], 2), MOIS_FR[_mmin[1] - 1], _mmin[0], nb(mois_[_mmin], 2)),
         "Un jour férié vaut <b>{}</b> d'un jour moyen, une veille de férié <b>{}</b>.".format(nb(eff.iloc[0, 1], 2), nb(eff.iloc[1, 1], 2)) if parc["ferie"].any() else "",
         "Ces effets sont pris en compte dans la détection : l'habitude d'un GAB est celle du <b>même jour de semaine</b>, un jour férié "
         "abaisse le niveau d'une hausse, et une hausse déjà vue l'an dernier à la même date est signalée comme un rendez-vous annuel."])

<a href="#sommaire" style="font-size:12px">↑ retour au sommaire</a>
<a id="s5"></a>
## 5. À quoi ressemble un retrait ordinaire ?
**En bref** : à quelle heure on retire, avec quelle carte, quels billets sortent, et à quelle fréquence surviennent les incidents.

**Ce que vous voyez** : la part des retraits par tranche horaire, par type de carte et par billet ; les motifs de cartes avalées ; puis un
tableau qui montre que les incidents sont **rares** : la plupart des journées n'ont ni retrait échoué ni carte avalée.

**À quoi ça sert** : un incident rare ne se juge pas comme un volume. Trois cartes avalées dans la journée peuvent être tout à fait anormales
pour un GAB qui n'en avale jamais.

<details><summary><i>Comment c'est calculé (pour les curieux)</i></summary>

Pour les événements rares (retraits échoués, cartes avalées, cartes étrangères, retraits de nuit), on calcule la probabilité d'en voir autant vu les 4 dernières semaines du GAB (loi de Poisson). On alerte quand cette probabilité est inférieure à 1 sur 10 000.

</details>

In [ ]:
act = brut[brut["nb_retraits"] > 0]
TRANCHES = [c for c in ["nb_0h_6h", "nb_6h_9h", "nb_9h_12h", "nb_12h_14h", "nb_14h_18h", "nb_18h_22h", "nb_22h_24h"] if c in act.columns]
RESEAUX_D = [c for c in ["nb_ret_interne", "nb_ret_cb", "nb_ret_visa", "nb_ret_mastercard", "nb_ret_cos", "nb_ret_exotique"] if c in act.columns]
MOTIFS_D = [c for c in act.columns if c.startswith("nb_capture_")]
LIB_TRANCHE = {"nb_0h_6h": "0 h – 6 h", "nb_6h_9h": "6 h – 9 h", "nb_9h_12h": "9 h – 12 h", "nb_12h_14h": "12 h – 14 h", "nb_14h_18h": "14 h – 18 h", "nb_18h_22h": "18 h – 22 h", "nb_22h_24h": "22 h – 24 h"}
LIB_RESEAU = {"interne": "porteurs de la banque", "cb": "CB", "visa": "Visa", "mastercard": "Mastercard", "cos": "COS", "exotique": "étrangères (UnionPay, Amex, Diners, JCB)"}
if TRANCHES:
    parts = act[TRANCHES].sum() / act[TRANCHES].sum().sum() * 100
    g_barres([LIB_TRANCHE.get(c, c) for c in TRANCHES], [("part des retraits", parts.values.round(1), BLEU)],
             "Quand retire-t-on ? Part des retraits par tranche horaire", y_titre="%", hauteur=280, hover_suffixe=" %")
if RESEAUX_D:
    parts = act[RESEAUX_D].sum() / act[RESEAUX_D].sum().sum() * 100
    g_barres([LIB_RESEAU.get(c.replace("nb_ret_", ""), c) for c in RESEAUX_D], [("part des retraits", parts.values.round(2), BLEU)],
             "Avec quelle carte ? Part des retraits par réseau", y_titre="%", hauteur=280, hover_suffixe=" %")
bil = [c for c in ["nb_billets_5", "nb_billets_10", "nb_billets_20", "nb_billets_50", "nb_billets_100"] if c in act.columns]
if bil:
    parts = act[bil].sum() / max(1, act[bil].sum().sum()) * 100
    g_barres([c.replace("nb_billets_", "") + " €" for c in bil], [("part des billets", parts.values.round(1), BLEU)], "Quels billets ? Part des billets distribués par valeur",
             y_titre="%", hauteur=260, hover_suffixe=" %")
if MOTIFS_D:
    parts = act[MOTIFS_D].sum() / max(1, act[MOTIFS_D].sum().sum()) * 100
    g_barres([c.replace("nb_capture_", "").replace("_", " ") for c in MOTIFS_D], [("part des captures", parts.values.round(1), BLEU)], "Pourquoi une carte est capturée : part par motif",
             y_titre="%", hauteur=260, hover_suffixe=" %")
_mm = act["montant_total"] / act["nb_retraits"]
_items = [("Montant moyen d'un retrait", nb(_mm.median(), 0) + " €", "médiane des GAB-jours")]
if "nb_inaboutis" in act.columns:
    _items += [("Inaboutis pour 1 000 retraits", nb(1000 * act["nb_inaboutis"].sum() / act["nb_retraits"].sum(), 2), "{} des GAB-jours en ont au moins un".format(pct((act["nb_inaboutis"] > 0).mean(), 0)))]
if "nb_captures" in act.columns:
    _items += [("Captures pour 1 000 retraits", nb(1000 * act["nb_captures"].sum() / act["nb_retraits"].sum(), 2), "{} des GAB-jours en ont au moins une".format(pct((act["nb_captures"] > 0).mean(), 0)))]
if "nb_retraits_repetes_10min" in act.columns:
    _items += [("Retraits refaits en moins de 10 min", nb(1000 * act["nb_retraits_repetes_10min"].sum() / act["nb_retraits"].sum(), 1) + " ‰", "même carte, deux retraits rapprochés")]
if "nb_ret_exotique" in act.columns:
    _items += [("GAB-jours avec une carte étrangère", pct((act["nb_ret_exotique"] > 0).mean(), 0), "UnionPay, Amex, Diners ou JCB")]
if "plan_cassettes" in act.columns:
    _items += [("Plan de cassettes le plus courant", act["plan_cassettes"].value_counts().index[0], "{} des GAB-jours".format(pct(act["plan_cassettes"].value_counts(normalize=True).iloc[0], 0)))]
tuiles(_items)
if "nb_inaboutis" in act.columns and "nb_captures" in act.columns:
    _r = pd.DataFrame({"inaboutis": act["nb_inaboutis"].clip(upper=5).value_counts(normalize=True).sort_index(),
                       "captures": act["nb_captures"].clip(upper=5).value_counts(normalize=True).sort_index()}).fillna(0)
    _r.index = [str(int(i)) if i < 5 else "5 et plus" for i in _r.index]
    _r = (_r * 100).round(1).reset_index().rename(columns={"index": "nombre dans la journée", "inaboutis": "% des GAB-jours (inaboutis)", "captures": "% des GAB-jours (captures)"})
    display(tableau(_r, "Ce que vaut « rare » : la plupart des journées n'ont ni inabouti ni capture", dec=1))
    retenir(["<b>{}</b> des journées n'ont aucun inabouti et <b>{}</b> aucune capture : 3 captures dans une journée est un événement rare, "
             "qu'on juge avec une probabilité (loi de Poisson) et non avec un écart à la moyenne.".format(pct(1 - (act["nb_inaboutis"] > 0).mean(), 0), pct(1 - (act["nb_captures"] > 0).mean(), 0)),
             "Les retraits de nuit (0 h – 6 h) et les cartes étrangères sont marginaux à l'échelle du parc, mais concentrés sur quelques GAB : "
             "c'est l'habitude du GAB qui dit si 5 cartes UnionPay sont normales (gare, aéroport) ou non (quartier)."])

<a href="#sommaire" style="font-size:12px">↑ retour au sommaire</a>
<a id="s6"></a>
## 6. Où sont les GAB ?
**Ce que vous voyez** : la carte de France, un point par GAB, plus foncé quand le GAB fait beaucoup de retraits. Survoler un point affiche
le GAB et son volume. Le tableau donne, pour quelques indicateurs, la valeur d'un GAB typique et celle des 10 % de GAB les plus bas et les
plus hauts.

**À quoi ça sert** : voir où se concentre l'activité, et constater qu'une même valeur n'a pas le même sens d'un GAB à l'autre.

In [ ]:
pos = None
if "latitude" in brut.columns and "longitude" in brut.columns:
    pos = brut.groupby("num_automate").agg(lat=("latitude", "first"), lon=("longitude", "first"), ville=("ville", "first") if "ville" in brut.columns else ("latitude", "size")).join(par_gab["retraits_par_jour"])
    pos["lat"], pos["lon"] = pd.to_numeric(pos["lat"], errors="coerce"), pd.to_numeric(pos["lon"], errors="coerce")
    pos = pos[pos["lat"].between(41, 51.5) & pos["lon"].between(-5.5, 10)]
    g_points(pos["lon"], pos["lat"], "Les GAB de France métropolitaine (couleur = retraits par jour, échelle log)",
             textes=["GAB {}{}<br>{} retraits / jour".format(i, " — " + str(v) if "ville" in brut.columns and pd.notnull(v) else "", nb(r, 1))
                     for i, v, r in zip(pos.index, pos["ville"] if "ville" in brut.columns else [None] * len(pos), pos["retraits_par_jour"])],
             couleurs=np.log1p(pos["retraits_par_jour"].fillna(0)).values, tailles=5, hauteur=560, aspect=1.4)
    note("{} GAB hors de la France métropolitaine ou sans coordonnées (non tracés).".format(nb(brut["num_automate"].nunique() - len(pos))))
prof_gab = act.groupby("num_automate").agg(retraits=("nb_retraits", "median"))
_g = act.groupby("num_automate")
if TRANCHES:
    prof_gab["part_nuit"] = _g["nb_0h_6h"].sum() / _g["nb_retraits"].sum()
if "nb_ret_exotique" in act.columns:
    prof_gab["part_exotique"] = _g["nb_ret_exotique"].sum() / _g["nb_retraits"].sum()
if "nb_ret_porteur" in act.columns:
    prof_gab["part_porteurs"] = _g["nb_ret_porteur"].sum() / _g["nb_retraits"].sum()
prof_gab["montant_moyen"] = _g["montant_total"].sum() / _g["nb_retraits"].sum()
_d = prof_gab.describe(percentiles=[0.1, 0.5, 0.9]).T[["10%", "50%", "90%"]].reset_index()
_d.columns = ["indicateur (par GAB)", "10 % des GAB en dessous de", "GAB médian", "10 % des GAB au-dessus de"]
_d["indicateur (par GAB)"] = _d["indicateur (par GAB)"].map({"retraits": "retraits par jour (médiane)", "part_nuit": "part des retraits de nuit (0 h – 6 h)",
                                                             "part_exotique": "part des cartes étrangères", "part_porteurs": "part des porteurs de la banque", "montant_moyen": "montant moyen d'un retrait (€)"})
for c in _d.columns[1:]:
    _d[c] = [pct(v, 1) if "part" in str(i) else nb(v, 1) for i, v in zip(_d["indicateur (par GAB)"], _d[c])]
display(tableau(_d, "Hétérogénéité des GAB : la même valeur n'a pas le même sens d'un GAB à l'autre"))

<a href="#sommaire" style="font-size:12px">↑ retour au sommaire</a>
<a id="s7"></a>
## 7. Quels types de GAB ? Les familles
**En bref** : on range les GAB en six familles de comportement, à partir des 12 derniers mois : **étrangers / touristiques**, **saisonniers**,
**de nuit**, **très fréquentés**, **peu fréquentés**, **standards**.

**À quoi ça sert** : situer une alerte. Un afflux de cartes étrangères sur un GAB de gare n'a pas le même sens que sur un GAB de village.

<details><summary><i>Comment c'est calculé (pour les curieux)</i></summary>

Règles simples, dans cet ordre : étrangers / touristiques = part de cartes étrangères parmi les 10 % les plus hautes et au moins 3 fois la valeur typique ; saisonniers = mois le plus fort au moins 2 fois le plus faible ; de nuit = part des retraits entre 0 h et 6 h parmi les 10 % les plus hautes ; très fréquentés = les 10 % de GAB les plus actifs ; peu fréquentés = les 10 % les moins actifs ou plus de 30 % de jours sans retrait ; standards = les autres. Un GAB observé moins de 60 jours est classé « historique court ».

</details>

In [ ]:
_an = longue[longue["date_jour"] > FIN - pd.Timedelta(days=365)]
_g = _an.groupby("num_automate")
familles = pd.DataFrame({"jours_observes": _g.size(), "retraits_par_jour": _g["nb_retraits"].mean(), "part_jours_sans_retrait": _g["jour_sans_retrait"].mean()})
_ret = _g["nb_retraits"].sum().clip(lower=1)
familles["part_exotique"] = (_g["nb_ret_exotique"].sum() / _ret) if "nb_ret_exotique" in _an.columns else 0.0
familles["part_nuit_0h_6h"] = (_g["nb_0h_6h"].sum() / _ret) if "nb_0h_6h" in _an.columns else 0.0
_mois = _an.groupby(["num_automate", "mois"])["nb_retraits"].mean().unstack()
familles["mois_observes"] = _mois.notnull().sum(axis=1)
familles["amplitude_saison"] = (_mois.max(axis=1) / _mois.min(axis=1).clip(lower=1)).where(familles["mois_observes"] >= 10)
familles["mois_le_plus_fort"] = _mois.idxmax(axis=1)
_ok = familles["jours_observes"] >= 60
_q = lambda col, q: familles.loc[_ok, col].quantile(q)
_med = lambda col: familles.loc[_ok, col].median()
touristique = _ok & (familles["part_exotique"] >= _q("part_exotique", 0.9)) & (familles["part_exotique"] >= 3 * max(_med("part_exotique"), 1e-4))
saisonnier = _ok & (familles["amplitude_saison"] >= 2.0)
nuit = _ok & (familles["part_nuit_0h_6h"] >= _q("part_nuit_0h_6h", 0.9)) & (familles["part_nuit_0h_6h"] >= 2 * max(_med("part_nuit_0h_6h"), 1e-4))
tres_frequente = _ok & (familles["retraits_par_jour"] >= _q("retraits_par_jour", 0.9))
peu_frequente = _ok & ((familles["retraits_par_jour"] <= _q("retraits_par_jour", 0.1)) | (familles["part_jours_sans_retrait"] >= 0.3))
familles["famille"] = np.select([~_ok, touristique, saisonnier, nuit, tres_frequente, peu_frequente],
                                ["historique court", "étrangers / touristiques", "saisonniers", "de nuit", "très fréquentés", "peu fréquentés"], "standards")
FAMILLE = familles["famille"]
ORDRE_FAMILLES = ["étrangers / touristiques", "saisonniers", "de nuit", "très fréquentés", "peu fréquentés", "standards", "historique court"]
resume_familles = familles.groupby("famille").agg(nb_gab=("famille", "size"), retraits_par_jour=("retraits_par_jour", "median"),
                                                  part_exotique=("part_exotique", "median"), part_nuit_0h_6h=("part_nuit_0h_6h", "median"),
                                                  amplitude_saison=("amplitude_saison", "median"), part_jours_sans_retrait=("part_jours_sans_retrait", "median")).reindex(ORDRE_FAMILLES).dropna(how="all")
resume_familles["part_du_parc"] = resume_familles["nb_gab"] / resume_familles["nb_gab"].sum()
_t = resume_familles.reset_index()
for c in ("part_exotique", "part_nuit_0h_6h", "part_jours_sans_retrait", "part_du_parc"):
    _t[c] = _t[c].map(lambda v: pct(v, 1))
_t["retraits_par_jour"] = _t["retraits_par_jour"].map(lambda v: nb(v, 1))
_t["amplitude_saison"] = _t["amplitude_saison"].map(lambda v: nb(v, 1) if pd.notnull(v) else "-")
_t["nb_gab"] = _t["nb_gab"].astype(int)
display(tableau(_t, "Les familles (valeurs médianes de la famille)", libelles={"famille": "famille", "nb_gab": "GAB", "part_du_parc": "% du parc", "retraits_par_jour": "retraits / jour",
                                                                            "part_exotique": "cartes étrangères", "part_nuit_0h_6h": "retraits 0 h – 6 h", "amplitude_saison": "mois fort / mois faible",
                                                                            "part_jours_sans_retrait": "jours à 0"}))
g_barres(resume_familles.index, [("GAB", resume_familles["nb_gab"].values, BLEU)], "Nombre de GAB par famille", hauteur=300, horizontal=True)
if (familles["famille"] == "saisonniers").any():
    _s = familles.loc[familles["famille"] == "saisonniers", "mois_le_plus_fort"].value_counts().sort_index()
    g_barres([MOIS_FR[int(m) - 1] for m in _s.index], [("GAB saisonniers", _s.values, BLEU)], "GAB saisonniers : leur mois le plus fort", hauteur=260)
retenir(["<b>{}</b> GAB sur {} sont « standards » ({}) ; les autres familles concentrent les comportements particuliers que les règles doivent connaître.".format(
             nb(resume_familles["nb_gab"].get("standards", 0)), nb(resume_familles["nb_gab"].sum()), pct(resume_familles["part_du_parc"].get("standards", 0), 0))] +
        ["<b>{}</b> GAB {} ({} du parc).".format(nb(resume_familles.loc[f, "nb_gab"]), f, pct(resume_familles.loc[f, "part_du_parc"], 0)) for f in ORDRE_FAMILLES if f in resume_familles.index and f != "standards"])

<a href="#sommaire" style="font-size:12px">↑ retour au sommaire</a>
<a id="s7bis"></a>
## 7 bis. Ce que la fiche dit d'autre : billets, argent dans le GAB, cartes étrangères
**En bref** : les anomalies que le métier veut voir laissent des traces au-delà du nombre de retraits. Une inversion de cassettes se voit
dans les **billets sortis**. Un GAB vidé ou dégradé se voit dans l'**argent restant** et l'**heure du dernier retrait**. Une rafale de cartes
étrangères se voit **réseau par réseau**. Cette section montre ce qui est normal pour chacun de ces sujets.

### Les billets sortis correspondent-ils au montant demandé ?
Sur un retrait réussi, les billets sortis doivent faire exactement le montant demandé. **Moins** : le client n'a pas eu tout son argent
(cassette vide, bourrage). **Plus** : de l'argent est sorti en trop (erreur de chargement, cassettes inversées). Ces cas sont très rares :
quand ils se multiplient sur un GAB, c'est un signal fort.

In [ ]:
_an = brut.assign(annee=brut["date_jour"].dt.year)
_ga = _an.groupby("annee")
COL_BILLETS = [("nb_billets_inferieurs_montant", "retraits servis en moins"), ("nb_billets_superieurs_montant", "retraits servis en trop"),
               ("nb_retraits_sans_info_billets", "retraits sans info billets"), ("nb_inaboutis_avec_billets", "inaboutis où des billets sont sortis"),
               ("nb_retraits_100plus_sans_billet_50", "retraits ≥ 100 € servis sans billet de 50 €"), ("nb_montant_10_ou_30", "retraits de 10 € ou 30 €")]
COL_BILLETS = [(c, l) for c, l in COL_BILLETS if c in brut.columns]
if COL_BILLETS:
    _t = pd.DataFrame({l: _ga[c].sum() for c, l in COL_BILLETS})
    _t.insert(0, "retraits aboutis", _ga["nb_retraits"].sum())
    for c, l in [("montant_non_distribue", "€ non distribués"), ("montant_sur_distribue", "€ sortis en trop"), ("montant_billets_inaboutis", "€ sortis sur des inaboutis")]:
        if c in brut.columns:
            _t[l] = _ga[c].sum().round(0)
    _t = _t.T.reset_index().rename(columns={"index": "indicateur"})
    _t.columns = [str(c) for c in _t.columns]
    display(tableau(_t, "Billets et montant demandé, année par année"))
    _items = []
    for c, l in COL_BILLETS[:2]:
        _gj = (brut[c] > 0)
        _items.append((l.capitalize(), pct(brut[c].sum() / max(1, brut["nb_retraits"].sum()), 3), "des retraits ; {} des GAB-jours en ont au moins un".format(pct(_gj.mean(), 2))))
    if "nb_retraits_sans_info_billets" in brut.columns:
        _si = brut.groupby("num_automate")[["nb_retraits_sans_info_billets", "nb_retraits"]].sum()
        _items.append(("GAB aux billets peu fiables", nb(int((_si["nb_retraits_sans_info_billets"] > 0.2 * _si["nb_retraits"].clip(lower=1)).sum())),
                       "plus de 20 % de retraits sans info billets : leurs anomalies de billets sont ignorées"))
    tuiles(_items)
    _gab_b = brut.groupby("num_automate")[[c for c, _ in COL_BILLETS[:2]] + (["montant_non_distribue"] if "montant_non_distribue" in brut.columns else [])].sum()
    _gab_b["total"] = _gab_b[[c for c, _ in COL_BILLETS[:2]]].sum(axis=1)
    _gab_b = _gab_b.sort_values("total", ascending=False).head(10).reset_index().rename(columns=dict({"num_automate": "GAB", "total": "total incohérents", "montant_non_distribue": "€ non distribués"}, **dict(COL_BILLETS)))
    display(tableau(_gab_b, "Les 10 GAB qui cumulent le plus de retraits aux billets incohérents sur la période"))

### Combien d'argent reste dans les GAB, et quand sont-ils rechargés ?
L'**encaisse** est l'argent présent dans le GAB. Elle baisse à chaque retrait et remonte à chaque chargement. Un GAB presque vide en fin de
journée explique un arrêt sans panne ni dégradation. Une baisse d'encaisse différente du montant retiré signale une erreur de comptage.

In [ ]:
if "encaisse_fin" in brut.columns and brut["encaisse_fin"].notnull().any():
    _e = brut[brut["nb_retraits"] > 0]
    _rech = brut.groupby("num_automate")["nb_recharges"].sum() / brut.groupby("num_automate").size() * 7 if "nb_recharges" in brut.columns else pd.Series(dtype=float)
    _items = [("Encaisse en fin de journée", nb(_e["encaisse_fin"].median()) + " €", "médiane des GAB-jours actifs ; 10 % sous {} €".format(nb(_e["encaisse_fin"].quantile(0.1))))]
    if len(_rech):
        _items.append(("Chargements par GAB et par semaine", nb(_rech.median(), 1), "médiane ; 10 % des GAB plus de {}".format(nb(_rech.quantile(0.9), 1))))
    if "encaisse_avant_chargement" in brut.columns:
        _items.append(("Encaisse juste avant un chargement", nb(brut["encaisse_avant_chargement"].median()) + " €", "médiane : le niveau auquel on recharge"))
    if "nb_incoherences_encaisse" in brut.columns:
        _items.append(("GAB-jours avec une incohérence d'encaisse", pct((brut["nb_incoherences_encaisse"] > 0).mean(), 2), "baisse d'encaisse différente du montant retiré"))
    _vide = (_e["encaisse_fin"] < 500).mean()
    _items.append(("GAB-jours finissant presque vides", pct(_vide, 2), "encaisse de fin de journée sous 500 €"))
    tuiles(_items)
    g_histos([("Encaisse en fin de journée", _e["encaisse_fin"].clip(upper=_e["encaisse_fin"].quantile(0.99)), "€"),
              ("Chargements par semaine, par GAB", _rech.clip(upper=_rech.quantile(0.99)) if len(_rech) else pd.Series([0]), "chargements / semaine")],
             "Encaisse et chargements (chaque barre compte des GAB-jours, puis des GAB)")
else:
    note("Pas de colonnes d'encaisse renseignées dans la fiche.", "warn")

### Avec quelles cartes retire-t-on ?
Les clients de la banque et les cartes CB font l'essentiel des retraits. Les **cartes étrangères** sont rares : UnionPay et Diners (passant
par **Trionis**), **Amex** (réseau à part) et JCB. Elles sont concentrées sur quelques GAB, et c'est là que se repèrent les rafales.

In [ ]:
RESEAUX_TOUS = [("interne", "porteurs de la banque"), ("cb", "CB"), ("visa", "Visa"), ("mastercard", "Mastercard"), ("cos", "COS"),
                ("cup", "UnionPay (Trionis)"), ("diners", "Diners (Trionis)"), ("jcb", "JCB"), ("amex", "Amex"), ("trionis", "Trionis (UnionPay + Diners)"), ("autre", "autre / inconnu")]
_lr = []
for r, lib in RESEAUX_TOUS:
    if "nb_ret_" + r in brut.columns:
        n_ = brut["nb_ret_" + r].sum()
        m_ = brut["montant_" + r].sum() if "montant_" + r in brut.columns else np.nan
        _lr.append({"réseau": lib, "retraits": n_, "part des retraits": pct(n_ / max(1, brut["nb_retraits"].sum()), 3), "montant retiré (€)": m_,
                    "montant moyen par retrait (€)": m_ / n_ if n_ else np.nan, "GAB qui en ont vu au moins un": int((brut.groupby("num_automate")["nb_ret_" + r].sum() > 0).sum()),
                    "jours où le parc en a vu": int((brut.groupby("date_jour")["nb_ret_" + r].sum() > 0).sum())})
if _lr:
    display(tableau(pd.DataFrame(_lr), "Les réseaux de cartes sur toute la période", dec=0))
    _exo = [(r, lib) for r, lib in RESEAUX_TOUS if r in ("cup", "diners", "jcb", "amex") and "nb_ret_" + r in longue.columns]
    if _exo:
        _m = longue.assign(mois=longue["date_jour"].dt.to_period("M").dt.to_timestamp()).groupby("mois")[["nb_ret_" + r for r, _ in _exo]].sum()
        g_courbes(_m.index, [(lib, _m["nb_ret_" + r].values, SERIE[i]) for i, (r, lib) in enumerate(_exo)], "Retraits par réseau étranger, mois par mois, sur toute la fiche",
                  y_titre="retraits par mois", hauteur=320, hover="%{x|%m/%Y}<br>%{y:,.0f} retraits<extra>%{fullData.name}</extra>")
if "nb_retraits_cos" in brut.columns and "nb_captures_cos" in brut.columns:
    tuiles([("Retraits COS", pct(brut["nb_retraits_cos"].sum() / max(1, brut["nb_retraits"].sum()), 1), "des retraits"),
            ("Captures COS ou internes", pct(brut["nb_captures_cos"].sum() / max(1, brut["nb_captures"].sum()), 0), "des captures : la carte COS est avalée normalement, ses captures brouillent le signal"),
            ("Captures des autres cartes", nb(brut["nb_captures_horscos"].sum() / max(1, len(brut)), 3) if "nb_captures_horscos" in brut.columns else "-", "par GAB et par jour (CB, Visa, Mastercard, étrangères)")])

### À quelle heure les GAB commencent et arrêtent de servir
Un GAB ordinaire sert du matin au soir. Quand un GAB qui sert d'habitude jusqu'au soir fait son dernier retrait en fin de matinée, il
s'est **arrêté en cours de journée** : panne, argent épuisé ou dégradation.

In [ ]:
if "heure_dernier_retrait" in brut.columns and brut["heure_dernier_retrait"].notnull().any():
    _h = brut[brut["nb_retraits"] > 0]
    g_histos([("Heure du premier retrait", _h["heure_premier_retrait"].dropna(), "heure"), ("Heure du dernier retrait", _h["heure_dernier_retrait"].dropna(), "heure")],
             "À quelle heure les GAB commencent et finissent de servir (GAB-jours actifs)")
    tuiles([("Premier retrait", "{} h {:02d}".format(int(_h["heure_premier_retrait"].median()), int(_h["heure_premier_retrait"].median() % 1 * 60)), "médiane"),
            ("Dernier retrait", "{} h {:02d}".format(int(_h["heure_dernier_retrait"].median()), int(_h["heure_dernier_retrait"].median() % 1 * 60)), "médiane"),
            ("Dernier retrait avant 14 h", pct((_h["heure_dernier_retrait"] < 14).mean(), 2), "des GAB-jours actifs")])

### Les données sont-elles propres ?
Ces compteurs doivent rester à zéro ou presque : montants illisibles, heures illisibles, retraits sans information sur les billets. Une
journée aux données douteuses est signalée pour information, jamais en alerte, pour ne pas confondre une erreur d'enregistrement avec un
problème de GAB.

In [ ]:
COL_QUALITE = [("nb_retraits_montant_zero", "retraits tentés à 0 €"), ("nb_retraits_montant_invalide", "montants négatifs ou vides"), ("nb_heure_invalide", "heures illisibles"),
               ("nb_retraits_sans_carte", "retraits sans numéro de carte"), ("nb_billets_valeur_inconnue", "billets de valeur inconnue"), ("nb_retraits_sans_info_billets", "retraits sans info billets")]
COL_QUALITE = [(c, l) for c, l in COL_QUALITE if c in brut.columns]
if COL_QUALITE:
    _q = pd.DataFrame({l: _ga[c].sum() for c, l in COL_QUALITE})
    _q["GAB-jours concernés"] = _an.assign(_x=(_an[[c for c, _ in COL_QUALITE]].sum(axis=1) > 0).astype(int)).groupby("annee")["_x"].sum()
    _q = _q.T.reset_index().rename(columns={"index": "compteur"})
    _q.columns = [str(c) for c in _q.columns]
    display(tableau(_q, "Qualité des données, année par année"))
del _an, _ga

<a href="#sommaire" style="font-size:12px">↑ retour au sommaire</a>
<a id="s7ter"></a>
## 7 ter. Chaque GAB a son propre rythme
**En bref** : pour chaque GAB, son jour le plus fort et le plus faible, son mois le plus fort et le plus faible, et sa **tendance** (les 12
derniers mois face aux 12 précédents).

**À quoi ça sert** : la détection compare chaque GAB à ses propres habitudes. Ces profils sont dans l'onglet **GAB** de l'Excel. Pour voir le
portrait complet d'un GAB : `portrait_gab("numéro du GAB")`.

In [ ]:
_l = longue[["num_automate", "date_jour", "nb_retraits"]].copy()
_l["js"] = _l["date_jour"].dt.dayofweek
_l["mois"] = _l["date_jour"].dt.month
_moy = _l.groupby("num_automate")["nb_retraits"].mean().replace(0, np.nan)
_ijs = _l.groupby(["num_automate", "js"])["nb_retraits"].mean().unstack().div(_moy, axis=0)
_im = _l.groupby(["num_automate", "mois"])["nb_retraits"].mean().unstack().div(_moy, axis=0)
PROFILS_GAB = pd.DataFrame({"retraits par jour (3 ans)": _moy.round(1),
                            "jour le plus fort": _ijs.idxmax(axis=1).map(lambda d: JOURS_FR[int(d)] if pd.notnull(d) else ""),
                            "jour le plus faible": _ijs.idxmin(axis=1).map(lambda d: JOURS_FR[int(d)] if pd.notnull(d) else ""),
                            "écart semaine (fort ÷ faible)": (_ijs.max(axis=1) / _ijs.min(axis=1).replace(0, np.nan)).round(2),
                            "mois le plus fort": _im.idxmax(axis=1).map(lambda m: MOIS_FR[int(m) - 1] if pd.notnull(m) else ""),
                            "mois le plus faible": _im.idxmin(axis=1).map(lambda m: MOIS_FR[int(m) - 1] if pd.notnull(m) else ""),
                            "amplitude saison (fort ÷ faible)": (_im.max(axis=1) / _im.min(axis=1).replace(0, np.nan)).round(2)})
_fin12 = _l[_l["date_jour"] > FIN - pd.Timedelta(days=365)].groupby("num_automate")["nb_retraits"].mean()
_prec12 = _l[(_l["date_jour"] <= FIN - pd.Timedelta(days=365)) & (_l["date_jour"] > FIN - pd.Timedelta(days=730))].groupby("num_automate")["nb_retraits"].mean()
PROFILS_GAB["tendance sur un an (%)"] = ((_fin12 / _prec12.replace(0, np.nan) - 1) * 100).round(1)
PROFILS_GAB["tendance"] = pd.cut(PROFILS_GAB["tendance sur un an (%)"], [-1e9, -20, -5, 5, 20, 1e9], labels=["forte baisse", "baisse", "stable", "hausse", "forte hausse"]).astype(str).replace("nan", "historique < 2 ans")
_tc = PROFILS_GAB["tendance"].value_counts().reindex(["forte baisse", "baisse", "stable", "hausse", "forte hausse", "historique < 2 ans"]).fillna(0).astype(int)
tuiles([(k, nb(v), "GAB") for k, v in _tc.items()])
g_barres(_ijs.columns.map(lambda d: JOURS_FR[int(d)]), [("indice médian des GAB", _ijs.median().round(2).values, BLEU)], "Jour de semaine : l'indice médian des GAB (1 = leur jour moyen)", hauteur=260, ligne_y=1)
g_histos([("Écart semaine (jour fort ÷ jour faible)", PROFILS_GAB["écart semaine (fort ÷ faible)"].clip(upper=5), "rapport"),
          ("Amplitude de la saison (mois fort ÷ mois faible)", PROFILS_GAB["amplitude saison (fort ÷ faible)"].clip(upper=5), "rapport"),
          ("Tendance sur un an", PROFILS_GAB["tendance sur un an (%)"].clip(-60, 60), "%")], "Chaque GAB a son propre rythme (chaque barre compte des GAB)")
_ex_jf = PROFILS_GAB["jour le plus fort"].value_counts()
retenir(["Le jour le plus fort n'est pas le même pour tous : <b>{}</b> pour {} GAB, <b>{}</b> pour {}.".format(_ex_jf.index[0], nb(_ex_jf.iloc[0]), _ex_jf.index[1] if len(_ex_jf) > 1 else "-", nb(_ex_jf.iloc[1]) if len(_ex_jf) > 1 else "-"),
         "<b>{}</b> GAB ont une saison marquée (mois fort au moins 1,5 fois le mois faible).".format(nb(int((PROFILS_GAB["amplitude saison (fort ÷ faible)"] >= 1.5).sum()))),
         "<b>{}</b> GAB sont en forte baisse sur un an (moins 20 % ou plus) et <b>{}</b> en forte hausse : à rapprocher des alertes de dérive lente.".format(nb(_tc.get("forte baisse", 0)), nb(_tc.get("forte hausse", 0)))])


def portrait_gab(gab, debut=None, fin=None):
    """Portrait d'un GAB : retraits jour par jour (moyenne 28 jours), profil par jour de semaine et par mois."""
    s_ = longue[longue["num_automate"] == str(gab)].sort_values("date_jour")
    if debut is not None:
        s_ = s_[s_["date_jour"] >= pd.Timestamp(debut)]
    if fin is not None:
        s_ = s_[s_["date_jour"] <= pd.Timestamp(fin)]
    if s_.empty:
        note("GAB {} absent de la fiche.".format(gab), "warn")
        return
    g_courbes(s_["date_jour"], [("retraits du jour", s_["nb_retraits"], BLEU_CLAIR), ("moyenne glissante 28 jours", s_["nb_retraits"].rolling(28, min_periods=7).mean().round(1), BLEU)],
              "GAB {} : retraits jour par jour".format(gab), y_titre="retraits", hauteur=300)
    if str(gab) in PROFILS_GAB.index:
        p_ = PROFILS_GAB.loc[str(gab)]
        tuiles([("Retraits par jour", nb(p_["retraits par jour (3 ans)"], 1), "moyenne sur la fiche"), ("Jour le plus fort", p_["jour le plus fort"], "le plus faible : " + p_["jour le plus faible"]),
                ("Mois le plus fort", p_["mois le plus fort"], "le plus faible : " + p_["mois le plus faible"]), ("Tendance sur un an", p_["tendance"], "{} %".format(nb(p_["tendance sur un an (%)"], 1)))])


_ex = (PROFILS_GAB["retraits par jour (3 ans)"] - PROFILS_GAB["retraits par jour (3 ans)"].median()).abs().idxmin()
portrait_gab(_ex)

# Partie B — Comment on repère un GAB anormal
Le principe en une phrase : **chaque GAB est comparé à ses propres habitudes**, jamais à la moyenne du parc.

<a href="#sommaire" style="font-size:12px">↑ retour au sommaire</a>
<a id="s8"></a>
## 8. L'habitude de chaque GAB
**En bref** : pour chaque GAB et chaque jour, on calcule ce qu'il fait d'ordinaire ce jour-là de la semaine (pour un lundi : ses 4 derniers
lundis). Une journée est anormale quand elle s'en éloigne beaucoup plus que d'habitude.

**Ce que vous voyez** : un GAB ordinaire pris en exemple. En haut, ses retraits (trait bleu) et son habitude (pointillés gris). En bas,
l'écart de chaque jour : au-delà de 3 (barres rouges), la journée sort vraiment de l'ordinaire pour ce GAB.

<details><summary><i>Comment c'est calculé (pour les curieux)</i></summary>

Habitude = valeur typique (médiane) du même jour de semaine sur les 4 semaines précédentes, sinon des 28 jours précédents. Écart = distance à l'habitude divisée par la variation habituelle du GAB sur 28 jours : 3 veut dire « trois fois plus loin que d'ordinaire ». Le jour examiné n'entre jamais dans sa propre habitude, et l'effet commun à tout le parc ce jour-là (jour férié, panne générale) est retiré.

</details>

In [ ]:
# toutes les colonnes numériques de la fiche sont gardées. Les comptes valent 0 un jour sans opération ; les valeurs « continues »
# (encaisse, heures, montant max, plus long trou) restent vides ce jour-là.
CONTINUS = [c for c in NUMERIQUES if c.startswith(("encaisse_", "heure_")) or c in ("montant_max", "montant_min", "plus_long_trou_min", "montant_max_chargement")]
COMPTES = [c for c in NUMERIQUES if c not in CONTINUS and not c.startswith(("nb_billets_pos", "val_max_pos"))]
GEO = [c for c in ["code_site_implantation", "latitude", "longitude", "ville", "code_postal", "statut_gab", "type_gab_e_i"] if c in brut.columns]
if EST_HORAIRE or brut.duplicated(["num_automate", "date_jour"]).any():
    agg = {c: "sum" for c in COMPTES}
    for c in CONTINUS:
        agg[c] = "min" if ("premier" in c or c.endswith("_min") or c == "encaisse_debut") else "max"
    agg["plan_cassettes"] = "last"
    for c in GEO:
        agg[c] = "first"
    jour = brut.groupby(["num_automate", "date_jour"], as_index=False).agg(agg)
else:
    jour = brut                                   # la fiche a déjà une ligne par GAB et par jour : pas de regroupement (et pas de copie)

# calendrier complet de chaque GAB (de son premier jour à la fin) : les jours sans aucune ligne deviennent des jours à 0
bornes = jour.groupby("num_automate")["date_jour"].min()
cal = pd.concat([pd.DataFrame({"num_automate": g_, "date_jour": pd.date_range(d_, FIN, freq="D")}) for g_, d_ in bornes.items()], ignore_index=True)
if len(cal) != len(jour):
    jour = cal.merge(jour, on=["num_automate", "date_jour"], how="left")
del cal
jour[COMPTES] = jour[COMPTES].fillna(0)
jour = jour.sort_values(["num_automate", "date_jour"]).reset_index(drop=True)
for c in GEO:
    jour[c] = jour.groupby("num_automate")[c].ffill()
    jour[c] = jour.groupby("num_automate")[c].bfill()
jour["jour_semaine"] = jour["date_jour"].dt.dayofweek + 1
jour["plan"] = jour.groupby("num_automate")["plan_cassettes"].ffill()
jour["plan"] = jour.groupby("num_automate")["plan"].bfill().fillna("?")
for v in (10, 20, 50):
    jour["billets_{}".format(v)] = jour["nb_billets_{}".format(v)]
jour["billets_total"] = jour[["nb_billets_{}".format(v) for v in (5, 10, 20, 50, 100)]].sum(axis=1)
for v in (10, 20, 50):
    jour["part_{}".format(v)] = np.where(jour["billets_total"] > 0, jour["billets_{}".format(v)] / jour["billets_total"].clip(lower=1), np.nan)
jour["montant_moyen"] = np.where(jour["nb_retraits"] > 0, jour["montant_total"] / jour["nb_retraits"].clip(lower=1), np.nan)


def glisse(valeur, cles, fenetre, mini, fonction="median", decalage=1):
    """Statistique glissante par groupe (`cles` = colonnes de `jour` ou Series) sur les `fenetre` lignes précédentes, décalées
    de `decalage` (1 = le jour examiné est exclu). Vectorisé : indispensable sur 3 ans de fiche."""
    cles = [jour[c] if isinstance(c, str) else c for c in cles]
    s_ = valeur.groupby(cles).shift(decalage) if decalage else valeur
    r_ = getattr(s_.groupby(cles).rolling(fenetre, min_periods=mini), fonction)()
    return r_.reset_index(level=list(range(len(cles))), drop=True).reindex(valeur.index)


INDICATEURS = {   # nom : (colonne, type, plancher de variation, libellé)
    "retraits": ("nb_retraits", "compte", 1.0, "retraits"),
    "montant_moyen": ("montant_moyen", "ratio", 0.05, "montant moyen par retrait (€)"),
    "part_10": ("part_10", "part", 0.02, "part des billets de 10 €"),
    "part_20": ("part_20", "part", 0.02, "part des billets de 20 €"),
    "part_50": ("part_50", "part", 0.02, "part des billets de 50 €"),
    "repetes": ("nb_retraits_repetes_10min", "compte", 0.8, "retraits refaits par la même carte < 10 min"),
    "inaboutis": ("nb_inaboutis", "compte", 0.8, "retraits inaboutis"),
    "captures": ("nb_captures", "compte", 0.8, "cartes capturées"),
}
INDICATEURS = {k: v for k, v in INDICATEURS.items() if v[0] in jour.columns}


def habitude(valeur, cles):
    """Médiane du même jour de semaine sur 4 semaines (>= 3 valeurs), sinon médiane des 28 jours précédents (>= 10)."""
    return glisse(valeur, cles + ["jour_semaine"], 4, 3).fillna(glisse(valeur, cles, 28, 10))


actif = jour["nb_retraits"] > 0
for nom, (col, typ, plancher, _) in INDICATEURS.items():
    cles = ["num_automate", "plan"] if typ == "part" else ["num_automate"]
    jour["hab_" + nom] = habitude(jour[col].where(actif), cles)
    if typ == "compte":
        ecart = 2 * np.sqrt(jour[col] + 0.375) - 2 * np.sqrt(jour["hab_" + nom] + 0.375)
    elif typ == "ratio":
        ecart = np.log(jour[col] / jour["hab_" + nom])
    else:
        ecart = jour[col] - jour["hab_" + nom]
    ecart = ecart.where(actif)
    variation = glisse(ecart.abs(), cles, 28, 10) * 1.4826
    z = (ecart / variation.clip(lower=plancher)).clip(-6, 6)
    jour["z_" + nom] = z - z.groupby(jour["date_jour"]).transform("median")   # effet du jour retiré
Z = ["z_" + n for n in INDICATEURS]
jour["habitude_ok"] = jour["z_retraits"].notnull()
jour["plan_sans_habitude"] = actif & jour["habitude_ok"] & jour["z_part_10"].isnull()
periode_m = jour["date_jour"] >= DEBUT_PERIODE

tuiles([("GAB-jours", nb(len(jour)), "{} GAB × calendrier complet (jours à 0 compris)".format(nb(jour["num_automate"].nunique()))),
        ("Indicateurs suivis", str(len(INDICATEURS)), ", ".join(v[3] for v in INDICATEURS.values())[:90] + "…"),
        ("Habitude calculable", pct(jour.loc[actif & periode_m, "habitude_ok"].mean(), 1), "des GAB-jours actifs de la période ; le reste = historique trop court"),
        ("Plans de cassettes", nb(jour["plan"].nunique()), "{} GAB-jours avec un plan trop récent (parts de billets non jugées)".format(nb(jour["plan_sans_habitude"].sum())))])
# exemple : un GAB « ordinaire » (volume médian) sur les 16 dernières semaines
_cand = jour[periode_m & actif].groupby("num_automate")["nb_retraits"].median()
_ex = (_cand - _cand.median()).abs().idxmin() if len(_cand) else None
if _ex is not None:
    _s = jour[(jour["num_automate"] == _ex) & (jour["date_jour"] > FIN - pd.Timedelta(days=112))]
    g_courbes(_s["date_jour"], [("retraits du jour", _s["nb_retraits"], BLEU), ("habitude (même jour de semaine, 4 semaines)", _s["hab_retraits"], GRIS, "dash")],
              "Exemple : GAB {} (volume médian du parc) — ses retraits face à son habitude, 16 dernières semaines".format(_ex), y_titre="retraits", hauteur=300)
    _z = _s["z_retraits"]
    g_barres(_s["date_jour"].dt.strftime("%d/%m"), [("écart normalisé", _z.round(2).values, [COULEURS_NIVEAU["Critique"] if abs(v) >= 3 else BLEU_CLAIR for v in _z.fillna(0)])],
             "Le même GAB : l'écart du jour (en variations habituelles). Au-delà de ±3 (rouge), la journée sort de l'ordinaire", y_titre="écart", hauteur=260)

<a href="#sommaire" style="font-size:12px">↑ retour au sommaire</a>
<a id="s9"></a>
## 9. Que se passe-t-il autour du GAB ? Les voisins
**En bref** : chaque GAB est rapproché de ses voisins à moins de 3 km.

**À quoi ça sert** : si tous les GAB d'un quartier baissent ensemble, c'est un événement de zone (coupure, travaux, manifestation), pas une
panne isolée. Si un GAB monte quand son voisin est arrêté, les clients se sont simplement reportés.

In [ ]:
jour["z_voisins"] = np.nan
jour["nb_voisins"] = 0
jour["voisin_arrete"] = None
jour["voisins_en_baisse"] = ""
jour["voisins_en_hausse"] = ""
VOISINS = {}
source_voisins = None
if DATASET_VOISINS:
    try:
        vois = dataiku.Dataset(DATASET_VOISINS).get_dataframe()
        vois.columns = [c.lower() for c in vois.columns]
        vois["num_automate"] = vois["num_automate"].astype(str)
        vois["voisin"] = vois["voisin"].astype(str)
        vois = vois.sort_values(["num_automate", "distance_km"])
        for gab, grp in vois.groupby("num_automate"):
            VOISINS[gab] = [(v, round(float(d), 2)) for v, d in zip(grp["voisin"], grp["distance_km"])][:NB_VOISINS]
        source_voisins = "recette " + DATASET_VOISINS
    except Exception as e:
        note("Dataset {} illisible ({}) : voisins calculés ici si latitude / longitude existent.".format(DATASET_VOISINS, e), "warn")
if not VOISINS and "latitude" in jour.columns and "longitude" in jour.columns:
    pos_ = jour.groupby("num_automate")[["latitude", "longitude"]].first()
    pos_ = pos_[pd.to_numeric(pos_["latitude"], errors="coerce").notnull() & pd.to_numeric(pos_["longitude"], errors="coerce").notnull()]
    noms = pos_.index.values
    lat, lon = np.radians(pos_["latitude"].astype(float).values), np.radians(pos_["longitude"].astype(float).values)
    for i0 in range(0, len(noms), 500):
        i1 = min(i0 + 500, len(noms))
        dlat = lat[i0:i1, None] - lat[None, :]
        dlon = lon[i0:i1, None] - lon[None, :]
        a_ = np.sin(dlat / 2) ** 2 + np.cos(lat[i0:i1, None]) * np.cos(lat[None, :]) * np.sin(dlon / 2) ** 2
        dist = 2 * 6371.0 * np.arcsin(np.sqrt(np.clip(a_, 0, 1)))
        for k, i in enumerate(range(i0, i1)):
            d = dist[k].copy()
            d[i] = np.inf
            proches = np.argsort(d)[:NB_VOISINS]
            proches = [j for j in proches if d[j] <= RAYON_VOISINS_KM]
            if proches:
                VOISINS[noms[i]] = [(noms[j], round(float(d[j]), 2)) for j in proches]
    source_voisins = "calculés ici, à moins de {} km".format(RAYON_VOISINS_KM)
if VOISINS:
    piv_z = jour.pivot(index="num_automate", columns="date_jour", values="z_retraits")
    piv_n = jour.pivot(index="num_automate", columns="date_jour", values="nb_retraits")
    piv_h = jour.pivot(index="num_automate", columns="date_jour", values="hab_retraits")
    res_z, res_v, res_n, res_b, res_h = {}, {}, {}, {}, {}
    for gab, vs in VOISINS.items():
        v_noms = [v for v, _ in vs if v in piv_z.index]
        if not v_noms or gab not in piv_z.index:
            continue
        zn = piv_z.loc[v_noms].values
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            res_z[gab] = np.nanmedian(zn, axis=0)
        arrete = (piv_n.loc[v_noms].values == 0) & (piv_h.loc[v_noms].fillna(0).values >= 10)
        res_v[gab] = [v_noms[int(np.argmax(col))] if col.any() else None for col in arrete.T]
        res_n[gab] = len(v_noms)
        zn0 = np.where(np.isnan(zn), 0.0, zn)
        res_b[gab] = [", ".join(v_noms[i] for i in np.where((col <= -3) | arr)[0]) for col, arr in zip(zn0.T, arrete.T)]
        res_h[gab] = [", ".join(v_noms[i] for i in np.where(col >= 3)[0]) for col in zn0.T]
    if res_z:
        def _long(dic, nom, dropna=True):
            t_ = pd.DataFrame(dic, index=piv_z.columns).T.stack(dropna=dropna).rename(nom).reset_index()
            t_.columns = ["num_automate", "date_jour", nom]
            return t_
        jour = jour.drop(columns=["z_voisins", "voisin_arrete", "voisins_en_baisse", "voisins_en_hausse"])
        for dic, nom, dn in [(res_z, "z_voisins", True), (res_v, "voisin_arrete", False), (res_b, "voisins_en_baisse", True), (res_h, "voisins_en_hausse", True)]:
            jour = jour.merge(_long(dic, nom, dn), on=["num_automate", "date_jour"], how="left")
        jour["voisins_en_baisse"] = jour["voisins_en_baisse"].fillna("")
        jour["voisins_en_hausse"] = jour["voisins_en_hausse"].fillna("")
        jour["nb_voisins"] = jour["num_automate"].map(res_n).fillna(0).astype(int)
    _nv = pd.Series([len(v) for v in VOISINS.values()])
    tuiles([("GAB avec au moins un voisin", nb(len(VOISINS)), "sur {} ({})".format(nb(jour["num_automate"].nunique()), source_voisins)),
            ("Voisins par GAB", nb(_nv.median()), "médiane ; au plus {}".format(NB_VOISINS)),
            ("GAB isolés", nb(jour["num_automate"].nunique() - len(VOISINS)), "aucun voisin : pas de contexte de zone pour eux")])
else:
    note("Aucun voisin disponible (ni recette voisins_gab, ni latitude / longitude) : pas de contexte de voisinage.", "warn")

<a href="#sommaire" style="font-size:12px">↑ retour au sommaire</a>
<a id="s10"></a>
## 10. La liste des anomalies recherchées
**En bref** : une cinquantaine d'anomalies, chacune avec un nom clair, rangées en familles : arrêt, baisse, hausse, montants, billets et
cassettes, retraits échoués et cartes avalées, cartes étrangères, horaires. Chacune a un **niveau** et produit une phrase avec les chiffres
du jour face à l'habitude du GAB.

**Ce que vous voyez** : le tableau de toutes les anomalies, avec leur niveau, comment on décide, et combien de fois elles se sont
déclenchées : en **GAB-jours** (un GAB un jour donné) et en **GAB différents**. Un GAB à l'arrêt 10 jours compte 10 GAB-jours mais 1 GAB.

<details><summary><i>Comment c'est calculé (pour les curieux)</i></summary>

Trois façons de décider. Pour les volumes (retraits, montants) : écart d'au moins 3 fois la variation habituelle ET un rapport minimal (×1,5, ×2, ×3, ou moins de 40 % pour une baisse). Pour les événements rares : probabilité d'en voir autant inférieure à 1 sur 10 000 vu les 4 dernières semaines (loi de Poisson), avec un minimum de 3 à 5 événements. Pour la répartition dans la journée ou entre réseaux : distance entre la répartition du jour et l'habituelle (Jensen-Shannon).

</details>

In [ ]:
from scipy.stats import poisson

REGLES = []          # (code, nom, gravité, masque, textes)
FAMILLE_ANOMALIE, COMMENT = {}, {}


def regle(code, nom, gravite, masque, textes, famille="", comment=""):
    """Enregistre une anomalie : masque booléen et phrases (Series alignées sur `jour`)."""
    masque = masque.fillna(False).astype(bool) if isinstance(masque, pd.Series) else pd.Series(masque, index=jour.index)
    if isinstance(textes, pd.Series):
        textes = textes.reindex(masque.index)[masque]
    REGLES.append((code, nom, gravite, masque, textes))
    FAMILLE_ANOMALIE[code], COMMENT[code] = famille, comment


def moyenne28(col):
    return glisse(jour[col], ["num_automate"], 28, 10, "mean")


def p_poisson(x, lam):
    """Probabilité d'observer au moins x événements quand on en attend lam (lam plancher 0,05)."""
    return pd.Series(poisson.sf(np.asarray(x, dtype=float) - 1, np.clip(np.asarray(lam, dtype=float), 0.05, None)), index=jour.index)


def chances(p):
    """« 1 chance sur N » lisible : 2 chiffres significatifs, plafonné à un million."""
    def _c(v):
        if pd.isnull(v) or v <= 1e-6:
            return "moins d'1 chance sur 1 million"
        return "1 chance sur {:,.0f}".format(float("{:.2g}".format(1.0 / v))).replace(",", " ")
    return p.map(_c)


def n0(x):
    return x.fillna(0).map(lambda v: "{:,.0f}".format(v).replace(",", " "))


def n1(x):
    return x.fillna(0).map(lambda v: "{:,.1f}".format(v).replace(",", " ").replace(".", ","))


def hab_jour(lam):
    """« habitude 2,3 par jour », ou « n'en voit presque jamais » quand l'habitude est quasi nulle."""
    return pd.Series(np.where(lam.fillna(0) < 0.05, "ce GAB n'en voit presque jamais", "habitude " + n1(lam) + " par jour"), index=lam.index)


def js_divergence(P, Q):
    """Distance de Jensen-Shannon (0 = mêmes répartitions, 1 = disjointes) entre deux tableaux de parts (lignes)."""
    P = np.where(np.isnan(P), 0, P)
    Q = np.where(np.isnan(Q), 0, Q)
    M = (P + Q) / 2.0
    with np.errstate(divide="ignore", invalid="ignore"):
        kl_p = np.where(P > 0, P * np.log2(P / M), 0).sum(axis=1)
        kl_q = np.where(Q > 0, Q * np.log2(Q / M), 0).sum(axis=1)
    return 0.5 * kl_p + 0.5 * kl_q


J = jour
a = lambda c: c in J.columns
g = lambda n: J["z_" + n].fillna(0) if ("z_" + n) in J.columns else pd.Series(0.0, index=J.index)
ratio = lambda c, h: J[c] / J[h].clip(lower=1)

F_ARRET, F_HAUSSE, F_BAISSE, F_MONTANT, F_CASSETTES, F_INCIDENTS, F_ETRANGER, F_FORME = (
    "Arrêt ou silence", "Activité en hausse", "Activité en baisse", "Montants", "Cassettes et billets", "Incidents cartes", "Cartes étrangères", "Horaires et clientèle")
ORDRE_FAMILLES_ANOMALIES = [F_ARRET, F_BAISSE, F_HAUSSE, F_MONTANT, F_CASSETTES, F_INCIDENTS, F_ETRANGER, F_FORME, "Qualité des données"]

# --- jours de données incomplètes (mesurés en section 2) : silences et baisses dégradés en information ces jours-là
J["jour_incomplet"] = J["date_jour"].isin(jours_incomplets)
J["gab_actifs_jour"] = J["date_jour"].map(actifs_jour)
J["gab_actifs_attendus"] = J["date_jour"].map(ref_actifs)
complet = ~J["jour_incomplet"]
INCOMPLET_TXT = " (données du jour probablement incomplètes : " + n0(J["gab_actifs_jour"]) + " GAB actifs au lieu de " + n0(J["gab_actifs_attendus"]) + ")"

# --- Arrêt ou silence : journée sans retrait improbable
J["est_zero"] = (J["nb_retraits"] == 0).astype(float)
k_wd = glisse(J["est_zero"], ["num_automate", "jour_semaine"], 8, 4, "sum")
n_wd = glisse(J["est_zero"], ["num_automate", "jour_semaine"], 8, 4, "count")
k_56 = glisse(J["est_zero"], ["num_automate"], 56, 28, "sum")
n_56 = glisse(J["est_zero"], ["num_automate"], 56, 28, "count")
J["part_zero"] = pd.concat([(k_wd + 0.5) / (n_wd + 1), (k_56 + 0.5) / (n_56 + 1)], axis=1).min(axis=1).clip(upper=0.98)
J["jamais_a_zero"] = (k_56 == 0) & n_56.notnull()
_run = ((J["est_zero"] == 0) | (J["num_automate"] != J["num_automate"].shift())).cumsum()
J["jours_a_zero"] = J["est_zero"].groupby(_run).cumsum()
for c in ("part_zero", "jamais_a_zero"):
    J[c] = J[c].where(J["jours_a_zero"] <= 1)
    J[c] = J.groupby("num_automate")[c].ffill()
J["jamais_a_zero"] = J["jamais_a_zero"].fillna(False).astype(bool)
J["p_silence"] = np.where(J["nb_retraits"] == 0, J["part_zero"] ** J["jours_a_zero"], 1.0)
JOURS_FR_PL = ["lundis", "mardis", "mercredis", "jeudis", "vendredis", "samedis", "dimanches"]
txt = ("aucun retrait depuis " + n0(J["jours_a_zero"]) + " jour(s) ; d'habitude à 0 " + (J["part_zero"] * 100).round(0).fillna(0).astype(int).astype(str)
       + " % des " + J["jour_semaine"].map(lambda d: JOURS_FR_PL[int(d) - 1]) + " (probabilité " + (J["p_silence"] * 100).round(2).fillna(100).astype(str) + " %)")
actif_zero = (J["nb_retraits"] == 0) & J["jamais_a_zero"] & (J["hab_retraits"].fillna(0) >= 50)


def hfmt(h):
    """Heures décimales -> « 11 h 05 »."""
    return h.map(lambda v: "" if pd.isnull(v) else "{} h {:02d}".format(int(v), int(round(v % 1 * 60)) % 60))


# dernier retrait connu avant l'arrêt (jour, heure, encaisse restante) : c'est ce qui date une dégradation ou une encaisse vide
_act = J["nb_retraits"] > 0
J["dernier_jour_actif"] = J["date_jour"].where(_act).groupby(J["num_automate"]).ffill()
DERNIER_TXT = pd.Series("", index=J.index)
if a("heure_dernier_retrait"):
    J["derniere_heure_active"] = J["heure_dernier_retrait"].where(_act).groupby(J["num_automate"]).ffill()
    DERNIER_TXT = ", dernier retrait le " + J["dernier_jour_actif"].dt.strftime("%d/%m").fillna("?") + " à " + hfmt(J["derniere_heure_active"])
if a("encaisse_fin"):
    J["derniere_encaisse"] = J["encaisse_fin"].where(_act).groupby(J["num_automate"]).ffill()
    DERNIER_TXT = DERNIER_TXT + pd.Series(np.where(J["derniere_encaisse"].notnull(), ", encaisse restante " + n0(J["derniere_encaisse"]) + " €" +
                                                   np.where(J["derniere_encaisse"] < 1000, " (GAB vide d'argent probable)", ""), ""), index=J.index)
txt_a = "aucun retrait depuis " + n0(J["jours_a_zero"]) + " jour(s) sur un GAB actif (" + n0(J["hab_retraits"]) + " retraits par jour d'habitude), jamais à 0 depuis 8 semaines" + DERNIER_TXT
silence_imp = (J["nb_retraits"] == 0) & ~actif_zero & (J["p_silence"] <= 0.001) & J["part_zero"].notnull()
silence_peu = (J["nb_retraits"] == 0) & ~actif_zero & (J["p_silence"] > 0.001) & (J["p_silence"] <= 0.01) & J["part_zero"].notnull()
regle("R02a", "GAB actif sans aucun retrait", 3, actif_zero & complet, txt_a, F_ARRET,
      "GAB à 50 retraits par jour ou plus, jamais à 0 depuis 8 semaines, et 0 retrait aujourd'hui : signalé dès le premier jour")
regle("R02", "Journée(s) sans retrait très improbable", 3, silence_imp & complet, txt, F_ARRET,
      "part habituelle de jours à 0 (même jour de semaine, 8 semaines) élevée à la puissance du nombre de jours consécutifs à 0 : moins de 1 chance sur 1 000")
regle("R02b", "Journée(s) sans retrait peu probable", 2, silence_peu & complet, txt, F_ARRET, "même calcul, entre 1 chance sur 1 000 et 1 chance sur 100")
regle("R02i", "Sans retrait un jour de données incomplètes", 0, (actif_zero | silence_imp | silence_peu) & ~complet, "aucun retrait" + INCOMPLET_TXT, F_ARRET,
      "le silence serait une alerte, mais la fiche n'a pas reçu toutes les opérations ce jour-là : information seulement")

# --- Billets distribués face au montant demandé (colonnes « inaboutis cachés » de la fiche)
#     un GAB dont plus de 20 % des retraits n'ont pas d'info billets n'est pas jugé sur ces colonnes ce jour-là
fiable_billets = (J["nb_retraits_sans_info_billets"] <= 0.2 * J["nb_retraits"].clip(lower=1)) if a("nb_retraits_sans_info_billets") else pd.Series(True, index=J.index)
billets_incoh = pd.Series(False, index=J.index)
BILLETS_TXT = pd.Series("", index=J.index)
for code, col, mcol, nom, lib_, sens in [("R30", "nb_billets_inferieurs_montant", "montant_non_distribue", "Billets sortis inférieurs au montant demandé", "retraits servis en moins", "non distribués"),
                                         ("R31", "nb_billets_superieurs_montant", "montant_sur_distribue", "Billets sortis supérieurs au montant demandé", "retraits servis en trop", "sortis en trop")]:
    if a(col):
        lam = moyenne28(col)
        p = p_poisson(J[col], lam)
        ok = (J[col] >= 5) & (p < 1e-4) & lam.notnull() & fiable_billets
        t_ = lib_ + " : " + n0(J[col]) + " (" + hab_jour(lam) + ", " + chances(p) + ")" + ((" ; " + n0(J[mcol]) + " € " + sens) if a(mcol) else "")
        regle(code, nom, 2, ok, t_, F_CASSETTES,
              "au moins 5 retraits aboutis dont les billets sortis ne font pas le montant demandé, et moins d'1 chance sur 10 000 vu l'habitude du GAB (billets jugés fiables ce jour-là)")
        billets_incoh = billets_incoh | ok
        BILLETS_TXT = BILLETS_TXT.where(~ok, BILLETS_TXT + " ; " + n0(J[col]) + " " + lib_)
if a("nb_inaboutis_avec_billets"):
    lam = moyenne28("nb_inaboutis_avec_billets")
    p = p_poisson(J["nb_inaboutis_avec_billets"], lam)
    regle("R32", "Billets sortis sur des retraits inaboutis", 2, (J["nb_inaboutis_avec_billets"] >= 3) & (p < 1e-4) & lam.notnull(),
          "billets sortis sur " + n0(J["nb_inaboutis_avec_billets"]) + " retraits inaboutis (habitude " + n1(lam) + " par jour)" +
          ((" ; " + n0(J["montant_billets_inaboutis"]) + " € sortis sans retrait abouti") if a("montant_billets_inaboutis") else ""), F_CASSETTES,
          "au moins 3 retraits inaboutis où des billets sont quand même sortis, improbable vu l'habitude : argent sorti sans retrait enregistré")
if a("nb_retraits_100plus_sans_billet_50") and a("nb_montant_100_200"):
    _gros = J["nb_montant_100_200"] + (J["nb_montant_plus_200"] if a("nb_montant_plus_200") else 0)
    lam = moyenne28("nb_retraits_100plus_sans_billet_50")
    p = p_poisson(J["nb_retraits_100plus_sans_billet_50"], lam)
    regle("R33", "Cassette de 50 € vide ou bloquée", 1, (J["nb_retraits_100plus_sans_billet_50"] >= 10) & (p < 1e-4) & (J["nb_retraits_100plus_sans_billet_50"] >= 0.6 * _gros.clip(lower=1)) & lam.notnull(),
          "retraits de 100 € ou plus servis sans billet de 50 € : " + n0(J["nb_retraits_100plus_sans_billet_50"]) + " sur " + n0(_gros) + " (" + hab_jour(lam) + ")", F_CASSETTES,
          "au moins 10 retraits de 100 € ou plus servis sans billet de 50 € (60 % ou plus des gros retraits du jour), alors que le GAB a une cassette de 50 €")
if a("nb_incoherences_encaisse"):
    lam = moyenne28("nb_incoherences_encaisse")
    p = p_poisson(J["nb_incoherences_encaisse"], lam)
    _baisse = ((" ; baisse d'encaisse " + n0(J["baisse_encaisse_retraits"]) + " € pour " + n0(J["montant_total"]) + " € retirés") if a("baisse_encaisse_retraits") else "")
    regle("R34", "Encaisse incohérente avec les retraits", 1, (J["nb_incoherences_encaisse"] >= 5) & (p < 1e-4) & lam.notnull(),
          "baisses d'encaisse différentes du montant retiré : " + n0(J["nb_incoherences_encaisse"]) + " (" + hab_jour(lam) + ")" + _baisse, F_CASSETTES,
          "au moins 5 retraits où l'encaisse ne baisse pas du montant retiré, improbable vu l'habitude : erreur de comptage ou cassette mal déclarée")

# --- Cassettes et billets : inversion 10/20 (signature du cas confirmé), bascule, retraits refaits, plan modifié
if a("hab_part_10"):
    part_txt = lambda v: ("part des billets de {} € ".format(v) + (J["part_" + str(v)] * 100).round(0).fillna(0).astype(int).astype(str)
                          + " % au lieu de " + (J["hab_part_" + str(v)] * 100).round(0).fillna(0).astype(int).astype(str) + " %")
    p10, p20 = part_txt(10), part_txt(20)
    rep_txt = "retraits refaits par la même carte " + n0(J["nb_retraits_repetes_10min"]) + " au lieu de " + n1(J["hab_repetes"])
    _repetes = (g("repetes") >= 3) & (J["nb_retraits_repetes_10min"] >= 10)
    inv = (np.maximum(g("part_10"), g("part_20")) >= 3) & (_repetes | billets_incoh)
    regle("R18", "Inversion de cassettes 10 € / 20 € possible", 3, inv,
          "inversion 10 € / 20 € possible : " + pd.Series(np.where(g("part_10") >= g("part_20"), p10, p20), index=J.index) +
          pd.Series(np.where(_repetes, " et " + rep_txt, ""), index=J.index) + BILLETS_TXT, F_CASSETTES,
          "part des billets de 10 € ou de 20 € très au-dessus de l'habitude (écart ≥ 3) ET, le même jour, au moins 10 retraits refaits par la même carte en moins de 10 min "
          "(le client reçoit la mauvaise somme et recommence) ou des billets sortis qui ne font pas le montant demandé")
    k_max = pd.DataFrame({v: g("part_" + str(v)).abs() for v in (10, 20, 50)}).idxmax(axis=1)
    ecart_pts = pd.DataFrame({v: (J["part_" + str(v)] - J["hab_part_" + str(v)]).abs() for v in (10, 20, 50)}).max(axis=1)
    bascule = (~inv) & (pd.DataFrame({v: g("part_" + str(v)).abs() for v in (10, 20, 50)}).max(axis=1) >= 3) & (ecart_pts >= 0.10) & (J["nb_retraits"] >= 20)
    PT = {v: part_txt(v) for v in (10, 20, 50)}
    bascule_txt = pd.Series(np.select([k_max == 10, k_max == 20], [PT[10].values, PT[20].values], PT[50].values), index=J.index)
    grosse = bascule & (ecart_pts >= 0.25) & (J["nb_retraits"] >= 50)
    regle("R18b", "Bascule de cassette marquée", 1, grosse, "bascule de cassette : " + bascule_txt, F_CASSETTES,
          "part d'une valeur de billet décalée de 25 points ou plus, sur un GAB à 50 retraits ou plus : cassette vide, bloquée ou rechargée")
    regle("R18i", "Bascule de cassette légère", 0, bascule & ~grosse, "bascule de cassette : " + bascule_txt, F_CASSETTES, "décalage de 10 à 25 points : information")
    regle("R18c", "Retraits refaits par la même carte", 2, (~inv) & (g("repetes") >= 3) & (J["nb_retraits_repetes_10min"] >= 10), rep_txt, F_CASSETTES,
          "au moins 10 retraits refaits en moins de 10 min par la même carte et écart ≥ 3, sans décalage des billets")
if a("nb_plans_cassettes"):
    regle("R18d", "Valeur déclarée d'une cassette changée dans la journée", 1, J["nb_plans_cassettes"] >= 2,
          "plan de cassettes modifié dans la journée (" + n0(J["nb_plans_cassettes"]) + " plans vus : " + J["plan_cassettes"].astype(str) + ")", F_CASSETTES,
          "deux paramétrages de cassettes différents le même jour")

# --- Calendrier : jours fériés (calculés en section 4), veilles et lendemains
J["ferie"] = J["date_jour"].map(lambda d_: "jour férié : " + FERIES[d_] if d_ in FERIES else
                                ("veille de férié" if (d_ + pd.Timedelta(days=1)) in FERIES else ("lendemain de férié" if (d_ - pd.Timedelta(days=1)) in FERIES else "")))
CTX = J["ferie"].map(lambda t_: " (" + t_ + ")" if t_ else "")
ferie_ok = J["ferie"] != ""

# --- Même date l'an dernier (même jour de semaine, 52 semaines plus tôt, moyenne ± 1 semaine) : rendez-vous annuels
J["cle_n1"] = J["date_jour"] - pd.Timedelta(days=364)
_n1 = longue[["num_automate", "date_jour", "nb_retraits"]].rename(columns={"date_jour": "cle_n1", "nb_retraits": "ret_n1"})
_n1 = pd.concat([_n1, _n1.assign(cle_n1=_n1["cle_n1"] + pd.Timedelta(days=7)), _n1.assign(cle_n1=_n1["cle_n1"] - pd.Timedelta(days=7))])
_n1 = _n1.groupby(["num_automate", "cle_n1"])["ret_n1"].mean().reset_index()
J = J.merge(_n1, on=["num_automate", "cle_n1"], how="left")
_L = longue[["num_automate", "date_jour", "nb_retraits"]].sort_values(["num_automate", "date_jour"]).copy()
_L["m_autour"] = _L.groupby("num_automate")["nb_retraits"].rolling(29, center=True, min_periods=15).mean().reset_index(level=0, drop=True)
_L["m_avant"] = _L.groupby("num_automate")["nb_retraits"].shift(15).groupby(_L["num_automate"]).rolling(28, min_periods=15).mean().reset_index(level=0, drop=True)
_L["indice_saison_n1"] = (_L["m_autour"] / _L["m_avant"].clip(lower=5)).clip(0.5, 2.5)
_L = _L.loc[_L["m_avant"] >= 20, ["num_automate", "date_jour", "indice_saison_n1"]].rename(columns={"date_jour": "cle_n1"})
J = J.merge(_L, on=["num_automate", "cle_n1"], how="left")
J["indice_saison_n1"] = J["indice_saison_n1"].fillna(1.0)
J["attendu_retraits"] = J["hab_retraits"] * J["indice_saison_n1"]
jour = J
A_N1 = J["ret_n1"].notnull()
meme_niveau_n1 = A_N1 & (J["nb_retraits"] / J["ret_n1"].clip(lower=1)).between(0.7, 1.4) & (J["ret_n1"] >= 20)

# --- Activité en hausse / en baisse, avec le contexte des voisins
saison_txt = pd.Series(np.where((J["indice_saison_n1"] - 1).abs() >= 0.10, ", attendu avec la saison " + n0(J["attendu_retraits"]), ""), index=J.index)
vol_txt = "retraits " + n0(J["nb_retraits"]) + " (habitude " + n0(J["hab_retraits"]) + saison_txt + ")"
ref_haut = pd.concat([J["hab_retraits"], J["attendu_retraits"]], axis=1).max(axis=1)
ref_bas = pd.concat([J["hab_retraits"], J["attendu_retraits"]], axis=1).min(axis=1)
zv = J["z_voisins"] if a("z_voisins") else pd.Series(np.nan, index=J.index)
voisin_arrete = J["voisin_arrete"].notnull() & (J["voisin_arrete"].astype(str) != "") & (J["voisin_arrete"].astype(str) != "None") if a("voisin_arrete") else pd.Series(False, index=J.index)
r_h = J["nb_retraits"] / ref_haut.clip(lower=1)
r_b = J["nb_retraits"] / ref_bas.clip(lower=1)
hausse = (g("retraits") >= 3) & (r_h >= 1.5) & (J["hab_retraits"] >= 10) & (J["nb_retraits"] - ref_haut >= 30)
baisse = (g("retraits") <= -3) & (r_b <= 0.4) & (J["hab_retraits"] >= 30) & (J["nb_retraits"] > 0) & complet
partage_h = zv.fillna(0) >= 2
partage_b = zv.fillna(0) <= -2
forte = r_h >= 2
tres_forte = r_h >= 3
h_base = hausse & ~voisin_arrete & ~partage_h
C_HAUSSE = "écart ≥ 3 variations habituelles, au moins 30 retraits de plus que l'habitude (ou que l'attendu saisonnier si plus haut), GAB à 10 retraits par jour ou plus"
regle("R26s", "Hausse déjà vue l'an dernier à la même date", 0, h_base & meme_niveau_n1,
      "hausse déjà vue l'an dernier à la même date : " + vol_txt + ", l'an dernier " + n0(J["ret_n1"]) + " : rendez-vous annuel probable", F_HAUSSE,
      "la hausse remplit les conditions, mais le GAB faisait entre 0,7 et 1,4 fois ce volume le même jour de semaine un an plus tôt : rendez-vous annuel, information")
h_base = h_base & ~meme_niveau_n1
regle("R10f", "Hausse d'activité brutale (×3 ou plus)", 3, h_base & tres_forte & ~ferie_ok, "hausse d'activité brutale : " + vol_txt, F_HAUSSE, C_HAUSSE + " ; retraits ≥ 3 fois la référence")
regle("R10", "Hausse d'activité forte (×2 à ×3)", 2, h_base & forte & ~tres_forte & ~ferie_ok, "hausse d'activité : " + vol_txt, F_HAUSSE, C_HAUSSE + " ; retraits entre 2 et 3 fois la référence")
regle("R10m", "Hausse d'activité modérée (×1,5 à ×2)", 1, h_base & ~forte & ~ferie_ok, "hausse d'activité : " + vol_txt, F_HAUSSE, C_HAUSSE + " ; retraits entre 1,5 et 2 fois la référence")
regle("R10j", "Hausse d'activité un jour férié ou autour", 1, h_base & forte & ferie_ok, "hausse d'activité : " + vol_txt + CTX, F_HAUSSE, "hausse ×2 ou plus, mais le jour est férié, veille ou lendemain de férié : un niveau de moins")
regle("R10jm", "Hausse modérée un jour férié ou autour", 0, h_base & ~forte & ferie_ok, "hausse d'activité : " + vol_txt + CTX, F_HAUSSE, "hausse ×1,5 à ×2 un jour férié ou autour : information")
regle("R10r", "Hausse : report depuis un voisin à l'arrêt", 1, hausse & voisin_arrete,
      "hausse d'activité, " + vol_txt + " : report probable depuis le GAB voisin " + J["voisin_arrete"].astype(str) + ", sans retrait ce jour-là", F_HAUSSE,
      "la hausse coïncide avec un voisin (habitude ≥ 10 retraits) à 0 retrait ce jour-là : la clientèle s'est reportée")
regle("R14", "Hausse partagée par les voisins (événement local)", 0, hausse & ~voisin_arrete & partage_h,
      "hausse d'activité, " + vol_txt + ", partagée par les voisins " + J["voisins_en_hausse"].astype(str) + " : événement local probable", F_HAUSSE,
      "les voisins sont eux aussi en hausse (écart médian ≥ 2) : marché, fête, événement ; information")
C_BAISSE = "écart ≤ −3 variations habituelles, retraits ≤ 40 % de l'habitude (ou de l'attendu saisonnier si plus bas), GAB à 30 retraits par jour ou plus, jour de données complet"
regle("R10b", "Baisse d'activité", 2, baisse & ~partage_b & ~ferie_ok & ~meme_niveau_n1, "baisse d'activité : " + vol_txt, F_BAISSE, C_BAISSE)
regle("R10bj", "Baisse d'activité un jour férié ou autour", 1, baisse & ~partage_b & ferie_ok & ~meme_niveau_n1, "baisse d'activité : " + vol_txt + CTX, F_BAISSE, C_BAISSE + " ; jour férié ou autour : un niveau de moins")
regle("R26b", "Baisse déjà vue l'an dernier à la même date", 0, baisse & ~partage_b & meme_niveau_n1,
      "baisse déjà vue l'an dernier à la même date : " + vol_txt + ", l'an dernier " + n0(J["ret_n1"]), F_BAISSE, "même volume le même jour de semaine un an plus tôt : information")
regle("R19", "Baisse partagée par les voisins : incident de zone possible", 3, baisse & partage_b,
      "baisse d'activité, " + vol_txt + ", partagée par les voisins " + J["voisins_en_baisse"].astype(str) + " : incident de zone possible", F_BAISSE,
      "la baisse est aussi chez les voisins (écart médian ≤ −2) : coupure, travaux, manifestation ; Critique car plusieurs GAB touchés")

# --- Montants
J["hab_montant_total"] = habitude(J["montant_total"].where(J["nb_retraits"] > 0), ["num_automate"])
J["attendu_montant"] = J["hab_montant_total"] * J["indice_saison_n1"]
ref_m = pd.concat([J["hab_montant_total"], J["attendu_montant"]], axis=1).max(axis=1)
rm = J["montant_total"] / ref_m.clip(lower=1)
mont_txt = "montant retiré " + n0(J["montant_total"]) + " € (habitude " + n0(J["hab_montant_total"]) + " €" + pd.Series(
    np.where((J["indice_saison_n1"] - 1).abs() >= 0.10, ", attendu avec la saison " + n0(J["attendu_montant"]) + " €", ""), index=J.index) + ")"
m_ok = (J["hab_montant_total"] >= 2000) & (J["montant_total"] - ref_m >= 10000) & (J["nb_retraits"] >= 20) & ~partage_h
_m3 = m_ok & (rm >= 3) & (J["montant_total"] - ref_m >= 15000)
C_MONT = "GAB à 2 000 € par jour ou plus, au moins 10 000 € de plus que l'habitude, 20 retraits ou plus, hausse non partagée par les voisins"
regle("R10M3", "Montant retiré triplé", 3, _m3 & ~ferie_ok & ~meme_niveau_n1, "montant retiré triplé : " + mont_txt, F_MONTANT, C_MONT + " ; ×3 et au moins 15 000 € de plus")
regle("R10M2", "Montant retiré doublé", 1, m_ok & (rm >= 2) & ~_m3 & ~ferie_ok & ~meme_niveau_n1, "montant retiré doublé : " + mont_txt, F_MONTANT, C_MONT + " ; ×2 à ×3")
regle("R10Mj", "Montant retiré doublé un jour férié ou autour", 1, m_ok & (rm >= 2) & ferie_ok & ~meme_niveau_n1, "montant retiré en hausse : " + mont_txt + CTX, F_MONTANT, C_MONT + " ; jour férié ou autour")
if a("hab_montant_moyen"):
    regle("R11", "Montant moyen par retrait anormal", 1, (g("montant_moyen").abs() >= 4) & (J["nb_retraits"] >= 30)
          & ((ratio("montant_moyen", "hab_montant_moyen") >= 2) | (ratio("montant_moyen", "hab_montant_moyen") <= 0.5)),
          "montant moyen par retrait " + n0(J["montant_moyen"]) + " € au lieu de " + n0(J["hab_montant_moyen"]) + " €", F_MONTANT,
          "montant moyen d'un retrait doublé ou divisé par deux (écart ≥ 4), sur 30 retraits ou plus")

# --- Incidents cartes : inaboutis et captures
for code, col, pic, lib in [("R05", "nb_inaboutis", "nb_inaboutis_heure_pic", "retraits inaboutis"), ("R06", "nb_captures", "nb_captures_heure_pic", "cartes capturées")]:
    if a(col):
        lam = moyenne28(col)
        p = p_poisson(J[col], lam)
        mini = 4 if col == "nb_captures" else 3
        regle(code, lib.capitalize() + " anormalement nombreux", 2, (J[col] >= mini) & (p < 1e-4) & lam.notnull(),
              lib + " : " + n0(J[col]) + " (" + hab_jour(lam) + ", " + chances(p) + ")", F_INCIDENTS,
              "au moins {} dans la journée et moins d'1 chance sur 10 000 vu la moyenne des 28 derniers jours (loi de Poisson)".format(mini))
        if a(pic):
            regle(code + "s", lib.capitalize() + " en série dans la même heure", 2, (J[pic] >= mini), n0(J[pic]) + " " + lib + " dans la même heure", F_INCIDENTS,
                  "au moins {} dans une même heure : panne de lecteur ou fraude en cours".format(mini))
MOT = [c for c in J.columns if c.startswith("nb_capture_") and c != "nb_captures"]
if MOT:
    cond, txt21 = pd.Series(False, index=J.index), pd.Series("", index=J.index)
    for c in MOT:
        lam = moyenne28(c)
        p = p_poisson(J[c], lam)
        c_ok = (J[c] >= 3) & (p < 1e-4) & lam.notnull()
        txt21 = txt21.where(~c_ok | (txt21 != ""), "captures « " + c.replace("nb_capture_", "").replace("_", " ") + " » : " + n0(J[c]) + " (" + hab_jour(lam) + ", " + chances(p) + ")")
        cond = cond | c_ok
    regle("R21", "Motif de capture inhabituel", 2, cond, txt21, F_INCIDENTS, "au moins 3 captures d'un même motif (carte volée, carte perdue…), improbable vu l'habitude")
RES = [r for r in ["interne", "cb", "visa", "mastercard", "cos", "exotique"] if a("nb_ret_" + r) and a("nb_inab_" + r)]
if RES:
    tot_inab = J[["nb_inab_" + r for r in RES]].sum(axis=1)
    tot_ret = J[["nb_ret_" + r for r in RES]].sum(axis=1)
    cond, txt22 = pd.Series(False, index=J.index), pd.Series("", index=J.index)
    for r in RES:
        taux_r = J["nb_inab_" + r] / (J["nb_inab_" + r] + J["nb_ret_" + r]).replace(0, np.nan)
        autres = (tot_inab - J["nb_inab_" + r]) / (tot_inab - J["nb_inab_" + r] + tot_ret - J["nb_ret_" + r]).replace(0, np.nan)
        c_ok = (J["nb_inab_" + r] >= 3) & (taux_r >= 0.3) & (autres.fillna(0) < 0.1)
        txt22 = txt22.where(~c_ok | (txt22 != ""), "inaboutis concentrés sur " + LIB_RESEAU.get(r, r) + " : " + n0(J["nb_inab_" + r]) + " échecs sur " + n0(J["nb_inab_" + r] + J["nb_ret_" + r]) + " tentatives, les autres réseaux normaux")
        cond = cond | c_ok
    regle("R22", "Inaboutis concentrés sur un réseau de cartes", 2, cond, txt22, F_INCIDENTS, "un réseau échoue 30 % du temps ou plus (3 échecs minimum) pendant que les autres échouent moins de 10 % : problème de routage")
    if all(a("nb_cap_" + r) for r in RES):
        cond, txt28 = pd.Series(False, index=J.index), pd.Series("", index=J.index)
        for r in RES:
            lam = moyenne28("nb_cap_" + r)
            p = p_poisson(J["nb_cap_" + r], lam)
            c_ok = (J["nb_cap_" + r] >= 3) & (p < 1e-4) & lam.notnull()
            txt28 = txt28.where(~c_ok | (txt28 != ""), "captures de cartes " + LIB_RESEAU.get(r, r) + " : " + n0(J["nb_cap_" + r]) + " (" + hab_jour(lam) + ", " + chances(p) + ")")
            cond = cond | c_ok
        regle("R28", "Captures inhabituelles sur un réseau de cartes", 1, cond, txt28, F_INCIDENTS, "au moins 3 captures sur un même réseau, improbable vu l'habitude")

# --- Cartes étrangères : par réseau (UnionPay, Amex, Diners, JCB) quand la fiche le permet, sinon en bloc
EXO = [(r, lib, mini) for r, lib, mini in [("cup", "UnionPay (UPI)", 5), ("amex", "American Express", 5), ("diners", "Diners", 3), ("jcb", "JCB", 3)] if a("nb_ret_" + r)]
if EXO:
    cond, massif, txt07 = pd.Series(False, index=J.index), pd.Series(False, index=J.index), pd.Series("", index=J.index)
    reseau07 = pd.Series("", index=J.index)
    for r, lib, mini in EXO:
        lam = moyenne28("nb_ret_" + r)
        p = p_poisson(J["nb_ret_" + r], lam)
        c_ok = (J["nb_ret_" + r] >= mini) & (p < 1e-4) & lam.notnull()
        t_ = "retraits " + lib + " : " + n0(J["nb_ret_" + r]) + " (" + hab_jour(lam) + ", " + chances(p) + ")"
        txt07 = txt07.where(~c_ok, np.where(txt07 == "", t_, txt07 + " ; " + t_))
        reseau07 = reseau07.where(~c_ok | (reseau07 != ""), lib)
        cond = cond | c_ok
        massif = massif | (c_ok & (J["nb_ret_" + r] >= 10))
    dept = J["code_postal"].astype(str).str.zfill(5).str[:2] if a("code_postal") else pd.Series("?", index=J.index)
    cle = pd.DataFrame({"d": J["date_jour"], "dep": dept, "res": reseau07})
    nb_meme_jour = cle[cond].groupby(["d", "dep", "res"])["res"].transform("size").reindex(J.index).fillna(0)
    coord = cond & (nb_meme_jour >= 3) & (dept != "?")
    C_EXO = "au moins 5 retraits UnionPay ou Amex (3 pour Diners ou JCB) et moins d'1 chance sur 10 000 vu l'habitude du GAB : un GAB de gare en voit tous les jours, un GAB de quartier jamais"
    regle("R20", "Même réseau étranger sur plusieurs GAB du département le même jour", 3, coord,
          txt07 + ", et " + n0(nb_meme_jour - 1) + " autre(s) GAB du département " + dept + " avec la même rafale ce jour-là", F_ETRANGER,
          C_EXO + " ; au moins 3 GAB du même département le même jour : attaque coordonnée possible")
    regle("R07f", "Afflux massif de cartes étrangères (10 ou plus)", 3, cond & massif & ~coord, "afflux massif : " + txt07, F_ETRANGER, C_EXO + " ; 10 retraits ou plus")
    regle("R07", "Afflux de cartes étrangères", 2, cond & ~massif & ~coord, txt07, F_ETRANGER, C_EXO)
elif a("nb_ret_exotique"):
    lam = moyenne28("nb_ret_exotique")
    p = p_poisson(J["nb_ret_exotique"], lam)
    regle("R07", "Afflux de cartes étrangères", 2, (J["nb_ret_exotique"] >= 5) & (p < 1e-4) & lam.notnull(),
          "cartes étrangères (Amex, UnionPay, Diners, JCB) : " + n0(J["nb_ret_exotique"]) + " retraits (habitude " + n1(lam) + " par jour, " + chances(p) + ")", F_ETRANGER,
          "au moins 5 retraits par carte étrangère et moins d'1 chance sur 10 000 vu l'habitude du GAB")

# --- Horaires et clientèle
if a("nb_0h_6h"):
    lam = moyenne28("nb_0h_6h")
    tot56 = glisse(J["nb_0h_6h"], ["num_automate"], 56, 28, "sum")
    p8 = p_poisson(J["nb_0h_6h"], lam)
    regle("R08", "Retraits de nuit sur un GAB qui n'en fait jamais", 1, (((J["nb_0h_6h"] >= 4) & (tot56 == 0)) | ((J["nb_0h_6h"] >= 6) & (p8 < 1e-5))) & tot56.notnull(),
          "retraits entre 0 h et 6 h : " + n0(J["nb_0h_6h"]) + " (habitude " + n1(lam) + " par nuit)" + CTX, F_FORME,
          "au moins 4 retraits entre 0 h et 6 h sur un GAB qui n'en a eu aucun depuis 8 semaines, ou au moins 6 et moins d'1 chance sur 100 000")
TR = [c for c in ["nb_0h_6h", "nb_6h_9h", "nb_9h_12h", "nb_12h_14h", "nb_14h_18h", "nb_18h_22h", "nb_22h_24h"] if a(c)]
if len(TR) == 7:
    parts = pd.DataFrame({c: J[c] / J[TR].sum(axis=1).replace(0, np.nan) for c in TR})
    habs = pd.DataFrame({c: glisse(parts[c], ["num_automate"], 28, 10, "mean") for c in TR})
    J["js_heures"] = js_divergence(parts.values, habs.values)
    pic_tr = parts.idxmax(axis=1).astype(str).map(lambda c_: LIB_TRANCHE.get(c_, c_))
    _ix = parts.fillna(-1).values.argmax(axis=1)
    J["part_tranche_max"] = parts.values[np.arange(len(parts)), _ix]                      # chiffres de preuve : la tranche la plus chargée du jour
    J["hab_part_tranche_max"] = habs.values[np.arange(len(habs)), _ix]
    regle("R12", "Répartition dans la journée inhabituelle", 1, (J["js_heures"] >= 0.25) & (J["nb_retraits"] >= 50) & habs[TR[0]].notnull(),
          "répartition des retraits dans la journée inhabituelle : le plus de retraits entre " + pic_tr
          + " (" + (parts.max(axis=1) * 100).round(0).fillna(0).astype(int).astype(str) + " %)", F_FORME,
          "distance de Jensen-Shannon ≥ 0,25 entre la répartition horaire du jour et celle des 28 derniers jours, sur 50 retraits ou plus")
RS = [c for c in ["nb_ret_interne", "nb_ret_cb", "nb_ret_visa", "nb_ret_mastercard", "nb_ret_cos", "nb_ret_exotique"] if a(c)]
if len(RS) >= 4:
    parts_r = pd.DataFrame({c: J[c] / J[RS].sum(axis=1).replace(0, np.nan) for c in RS})
    habs_r = pd.DataFrame({c: glisse(parts_r[c], ["num_automate"], 28, 10, "mean") for c in RS})
    J["js_reseaux"] = js_divergence(parts_r.values, habs_r.values)
    dom = parts_r.idxmax(axis=1).astype(str).str.replace("nb_ret_", "").map(lambda r_: LIB_RESEAU.get(r_, r_))
    _ix = parts_r.fillna(-1).values.argmax(axis=1)
    J["part_reseau_dominant"] = parts_r.values[np.arange(len(parts_r)), _ix]               # chiffres de preuve : le réseau dominant du jour
    J["hab_part_reseau_dominant"] = habs_r.values[np.arange(len(habs_r)), _ix]
    regle("R13", "Mélange de réseaux de cartes inhabituel", 1, (J["js_reseaux"] >= 0.25) & (J["nb_retraits"] >= 50) & habs_r[RS[0]].notnull(),
          "mélange de cartes inhabituel : " + dom + " = "
          + (parts_r.max(axis=1) * 100).round(0).fillna(0).astype(int).astype(str) + " % des retraits", F_FORME,
          "distance de Jensen-Shannon ≥ 0,25 entre la répartition par réseau du jour et celle des 28 derniers jours, sur 50 retraits ou plus")
if a("nb_ret_porteur"):
    J["part_porteur"] = J["nb_ret_porteur"] / J["nb_retraits"].replace(0, np.nan)
    hab_pp = glisse(J["part_porteur"], ["num_automate"], 28, 10, "mean")
    regle("R29", "Afflux de clients d'autres banques", 1, (hab_pp - J["part_porteur"] >= 0.35) & (J["nb_retraits"] >= 20),
          "part de porteurs de la banque " + (J["part_porteur"] * 100).round(0).fillna(0).astype(int).astype(str) + " % au lieu de " + (hab_pp * 100).round(0).fillna(0).astype(int).astype(str) + " % : afflux de clients d'autres banques",
          F_FORME, "la part des porteurs de la banque chute de 35 points ou plus (20 retraits minimum)")

# --- Dérives sur plusieurs semaines (classées avec l'activité en baisse)
m28 = glisse(J["nb_retraits"], ["num_automate"], 28, 20, "mean", decalage=0)
m56 = glisse(J["nb_retraits"], ["num_automate"], 56, 40, "mean", decalage=28)
mn28 = glisse(J["nb_retraits"], ["num_automate"], 28, 20, "min", decalage=0)
J["retraits_4_semaines"], J["retraits_8_semaines_avant"] = m28, m56          # chiffres de preuve de la dérive lente
regle("R15", "Dérive lente de l'activité", 2, (m28 < 0.6 * m56) & (m56 >= 20) & (mn28 > 0),
      "dérive lente : " + n0(m28) + " retraits par jour sur 4 semaines, contre " + n0(m56) + " sur les 8 semaines d'avant", F_BAISSE,
      "les 4 dernières semaines font moins de 60 % des 8 semaines précédentes, sans jour à 0 (GAB à 20 retraits par jour ou plus)")
if len(TR) == 7:
    cond = pd.Series(False, index=J.index)
    txt16 = pd.Series("", index=J.index)
    J["part_jours_zero_tranche"] = np.nan
    for c in TR:
        zero28 = glisse((J[c] == 0).astype(float), ["num_automate"], 28, 20, "mean", decalage=0)
        hab_part = glisse(parts[c], ["num_automate"], 56, 30, "mean", decalage=28)
        c_ok = (zero28 > 0.5) & (hab_part >= 0.10) & (J["nb_retraits"] > 0)
        J["part_jours_zero_tranche"] = J["part_jours_zero_tranche"].where(~c_ok | J["part_jours_zero_tranche"].notnull(), zero28)
        txt16 = txt16.where(~c_ok | (txt16 != ""), "tranche " + LIB_TRANCHE.get(c, c) + " (d'habitude " + (hab_part * 100).round(0).fillna(0).astype(int).astype(str)
                            + " % des retraits) à 0 " + (zero28 * 100).round(0).fillna(0).astype(int).astype(str) + " % des jours depuis 4 semaines")
        cond = cond | c_ok
    regle("R16", "Trou horaire récurrent", 1, cond, "trou horaire récurrent : " + txt16, F_BAISSE,
          "une tranche horaire qui pesait 10 % des retraits ou plus est à 0 plus d'un jour sur deux depuis 4 semaines : panne partielle, horaires d'accès changés")
if a("plus_long_trou_min") and a("nb_heures_actives"):
    hab_trou = glisse(J["plus_long_trou_min"], ["num_automate"], 28, 10)
    hab_ha = glisse(J["nb_heures_actives"], ["num_automate"], 28, 10)
    exces_h = (J["plus_long_trou_min"] - hab_trou) / 60.0
    rythme = J["hab_retraits"] / hab_ha.clip(lower=1)
    p1 = np.exp(-rythme.fillna(0) * exces_h.clip(lower=0))
    regle("R01", "Silence anormal en journée", 1, (exces_h >= 3) & (p1 < 1e-4) & (J["nb_retraits"] >= 10) & (J["hab_retraits"] >= 30) & (hab_trou <= 8 * 60) & hab_trou.notnull() & complet,
          "silence de " + n0(J["plus_long_trou_min"] / 60.0) + " h sans opération (habitude " + n0(hab_trou / 60.0) + " h, la nuit comprise) : " + chances(pd.Series(p1, index=J.index)), F_ARRET,
          "le plus long trou sans opération dépasse de 3 h ou plus l'habitude du GAB, sur un GAB actif en journée (30 retraits ou plus, trou habituel ≤ 8 h) : moins d'1 chance sur 10 000 au rythme du GAB")

# --- Arrêt en cours de journée : le GAB cesse de servir des heures plus tôt que d'habitude (dégradation, panne, encaisse vide)
if a("heure_dernier_retrait"):
    J["hab_dernier_retrait"] = glisse(J["heure_dernier_retrait"].where(J["nb_retraits"] > 0), ["num_automate"], 28, 10)
    _vide = (J["encaisse_fin"] < 1000) if a("encaisse_fin") else pd.Series(False, index=J.index)
    arret_j = (J["heure_dernier_retrait"] <= J["hab_dernier_retrait"] - 4) & (J["hab_dernier_retrait"] >= 17) & (J["nb_retraits"] > 0) \
              & (J["nb_retraits"] <= 0.7 * J["hab_retraits"]) & (J["hab_retraits"] >= 30) & complet
    _t = ("dernier retrait à " + hfmt(J["heure_dernier_retrait"]) + " au lieu de " + hfmt(J["hab_dernier_retrait"]) + " d'habitude ; " + vol_txt +
          ((", encaisse restante " + n0(J["encaisse_fin"]) + " €") if a("encaisse_fin") else ""))
    regle("R35z", "Arrêt en cours de journée partagé par les voisins", 3, arret_j & ~_vide & (partage_b | voisin_arrete),
          _t + ", voisins touchés : " + J["voisins_en_baisse"].astype(str) + " : incident de zone possible (dégradation, coupure)", F_ARRET,
          "dernier retrait au moins 4 h plus tôt que d'habitude (GAB qui sert d'habitude jusqu'à 17 h ou plus), 30 % de retraits en moins, ET les voisins en baisse ou à l'arrêt le même jour")
    regle("R35", "Arrêt en cours de journée", 2, arret_j & ~_vide & ~(partage_b | voisin_arrete), _t, F_ARRET,
          "dernier retrait au moins 4 h plus tôt que d'habitude (GAB qui sert d'habitude jusqu'à 17 h ou plus) et au moins 30 % de retraits en moins : panne ou dégradation")
    regle("R35v", "GAB vide d'argent en cours de journée", 1, arret_j & _vide, _t + " : encaisse épuisée", F_ARRET,
          "même arrêt en cours de journée, mais l'encaisse de fin est sous 1 000 € : le GAB est vide, ce n'est ni une panne ni une dégradation")

# --- Qualité des données : signalée en information, jamais en alerte
_q = [c for c in ["nb_retraits_montant_invalide", "nb_heure_invalide", "nb_billets_valeur_inconnue"] if a(c)]
if _q or a("nb_retraits_sans_info_billets"):
    _dq = (J[_q].sum(axis=1) > 0) if _q else pd.Series(False, index=J.index)
    if a("nb_retraits_sans_info_billets"):
        _dq = _dq | ((J["nb_retraits_sans_info_billets"] > 0.2 * J["nb_retraits"]) & (J["nb_retraits"] >= 10))
    _cq = _q + (["nb_retraits_sans_info_billets"] if a("nb_retraits_sans_info_billets") else [])
    _tq = J.loc[_dq, _cq].fillna(0).astype(int).apply(lambda r_: ", ".join("{} {}".format(v_, k_.replace("nb_", "").replace("_", " ")) for k_, v_ in r_.items() if v_), axis=1) if _dq.any() else pd.Series(dtype=object)
    regle("R36i", "Données du jour douteuses", 0, _dq, "données douteuses ce jour-là : " + _tq.astype(str), "Qualité des données",
          "montants ou heures illisibles, billets de valeur inconnue, ou plus de 20 % de retraits sans info billets : information, pour ne pas confondre une erreur d'enregistrement avec une anomalie")

# --- Le catalogue, avec le compte de la période
NOM_REGLE = {c_: n_ for c_, n_, _, _, _ in REGLES}
GRAVITE_REGLE = {c_: g_ for c_, _, g_, _, _ in REGLES}
NIVEAU_GRAVITE = {3: "Critique", 2: "Élevé", 1: "À surveiller", 0: "information"}
_p = J["date_jour"] >= DEBUT_PERIODE
catalogue = pd.DataFrame([{"famille": FAMILLE_ANOMALIE[c_], "anomalie": n_, "niveau si seule": NIVEAU_GRAVITE[g_], "gravite": g_,
                           "comment on décide": COMMENT[c_], "GAB-jours": int((m_ & _p).sum()), "GAB distincts": int(J.loc[m_ & _p, "num_automate"].nunique()), "code technique": c_}
                          for c_, n_, g_, m_, _ in REGLES])
catalogue["comment on décide"] = catalogue["comment on décide"].map(en_clair)
ACTION_FAMILLE = {F_ARRET: "Vérifier l'état du GAB (panne ou dégradation) et envoyer un technicien si besoin",
                  F_BAISSE: "Vérifier le GAB et son environnement (travaux, accès, concurrence)",
                  F_HAUSSE: "Vérifier que l'encaisse suffira ; chercher un événement local",
                  F_MONTANT: "Vérifier les gros retraits (fraude possible) et l'encaisse",
                  F_CASSETTES: "Faire contrôler le chargement des cassettes et l'encaisse",
                  F_INCIDENTS: "Faire vérifier le lecteur de carte ; prévenir l'équipe fraude si des cartes volées sont avalées",
                  F_ETRANGER: "Prévenir l'équipe fraude : rafale de cartes étrangères",
                  F_FORME: "Vérifier les horaires d'accès et l'environnement du GAB",
                  "Qualité des données": "Signaler à l'équipe données"}
ACTION_CODE = {"R02a": "Envoyer un technicien : GAB à l'arrêt", "R02": "Envoyer un technicien : GAB à l'arrêt", "R02b": "Vérifier si le GAB est en panne",
               "R35": "Envoyer un technicien : GAB arrêté en pleine journée", "R35z": "Vérifier un incident de zone (coupure, travaux, manifestation) et l'état des GAB",
               "R35v": "Recharger le GAB et revoir la fréquence de chargement", "R19": "Vérifier un incident de zone (coupure, travaux, manifestation)",
               "R18": "Faire contrôler les cassettes au plus vite : clients mal servis, écart de caisse", "R33": "Recharger ou débloquer la cassette de 50 €",
               "R30": "Contrôler la distribution des billets : des clients n'ont pas eu tout leur argent", "R31": "Contrôler la distribution et l'encaisse : de l'argent est sorti en trop",
               "R32": "Contrôler l'encaisse : billets sortis sans retrait enregistré", "R20": "Prévenir l'équipe fraude : attaque coordonnée possible sur le département",
               "R07f": "Prévenir l'équipe fraude : afflux massif de cartes étrangères", "R15": "Comprendre la baisse durable (travaux, concurrence, emplacement)",
               "R10r": "Aucune action sur ce GAB : il reçoit les clients d'un voisin arrêté (voir le voisin)"}


def action_conseillee(code_, gravite_=1):
    if gravite_ == 0:
        return "Aucune action : information"
    return ACTION_CODE.get(code_, ACTION_FAMILLE.get(FAMILLE_ANOMALIE.get(code_, ""), "Vérifier le GAB"))


catalogue["que faire"] = [action_conseillee(c_, g_) for c_, g_ in zip(catalogue["code technique"], catalogue["gravite"])]
catalogue["ordre"] = catalogue["famille"].map({f: i for i, f in enumerate(ORDRE_FAMILLES_ANOMALIES)})
catalogue = catalogue.sort_values(["ordre", "gravite", "GAB-jours"], ascending=[True, False, False]).drop(columns="ordre").reset_index(drop=True)
bandeau("Le catalogue : {} anomalies dans {} familles".format(len(catalogue), catalogue["famille"].nunique()),
        "déclenchements sur la période du {} au {} ({} jours, {} GAB)".format(DEBUT_PERIODE.strftime("%d/%m/%Y"), FIN.strftime("%d/%m/%Y"), nb(NB_JOURS_EXAMINES), nb(J["num_automate"].nunique())))
display(tableau(catalogue.drop(columns=["gravite", "code technique"]), n=100, pills=("niveau si seule",)))
_cf = catalogue[catalogue["gravite"] >= 1].groupby("famille")[["GAB-jours", "GAB distincts"]].sum().reindex(ORDRE_FAMILLES_ANOMALIES).fillna(0)
g_barres(_cf.index, [("GAB-jours en alerte", _cf["GAB-jours"].values, BLEU_CLAIR), ("GAB distincts concernés", _cf["GAB distincts"].values, BLEU)],
         "Par famille d'anomalie : combien de GAB-jours, et combien de GAB distincts (alertes de niveau À surveiller ou plus)", hauteur=340)

### Les chiffres qui prouvent chaque diagnostic
**En bref** : chaque GAB signalé porte, à côté de son diagnostic, **le chiffre qui le justifie** : la valeur du jour et la valeur
habituelle du GAB. On peut ainsi vérifier d'un coup d'œil que le diagnostic est vrai, sans refaire le calcul. Pour aller plus loin,
`verifier_gab("numéro", "date")` affiche la **fiche de preuve** complète d'un GAB un jour donné (voir section 12).

In [ ]:
INDIC_PREUVE = {   # anomalie -> (colonne, colonne d'habitude si elle existe, libellé)
    "R05": ("nb_inaboutis", None, "retraits échoués"), "R05s": ("nb_inaboutis_heure_pic", None, "retraits échoués dans la même heure"),
    "R06": ("nb_captures", None, "cartes avalées"), "R06s": ("nb_captures_heure_pic", None, "cartes avalées dans la même heure"),
    "R21": ("nb_captures", None, "cartes avalées"), "R22": ("nb_inaboutis", None, "retraits échoués"), "R28": ("nb_captures", None, "cartes avalées"),
    "R07": ("nb_ret_exotique", None, "retraits cartes étrangères"), "R07f": ("nb_ret_exotique", None, "retraits cartes étrangères"),
    "R20": ("nb_ret_exotique", None, "retraits cartes étrangères"), "R08": ("nb_0h_6h", None, "retraits entre 0 h et 6 h"),
    "R18": ("part_10", "hab_part_10", "part des billets de 10 €"), "R18b": ("part_10", "hab_part_10", "part des billets de 10 €"),
    "R18i": ("part_10", "hab_part_10", "part des billets de 10 €"), "R18c": ("nb_retraits_repetes_10min", "hab_repetes", "retraits refaits < 10 min"),
    "R18d": ("nb_plans_cassettes", None, "paramétrages de cassettes dans la journée"),
    "R30": ("nb_billets_inferieurs_montant", None, "retraits servis en moins"), "R31": ("nb_billets_superieurs_montant", None, "retraits servis en trop"),
    "R32": ("nb_inaboutis_avec_billets", None, "billets sortis sur des retraits échoués"), "R33": ("nb_retraits_100plus_sans_billet_50", None, "retraits ≥ 100 € sans billet de 50 €"),
    "R34": ("nb_incoherences_encaisse", None, "incohérences d'encaisse"), "R35": ("heure_dernier_retrait", "hab_dernier_retrait", "heure du dernier retrait"),
    "R35z": ("heure_dernier_retrait", "hab_dernier_retrait", "heure du dernier retrait"), "R35v": ("heure_dernier_retrait", "hab_dernier_retrait", "heure du dernier retrait"),
    "R11": ("montant_moyen", "hab_montant_moyen", "montant moyen par retrait"), "R10M3": ("montant_total", "hab_montant_total", "montant retiré"),
    "R10M2": ("montant_total", "hab_montant_total", "montant retiré"), "R10Mj": ("montant_total", "hab_montant_total", "montant retiré"),
    "R01": ("plus_long_trou_min", None, "plus long moment sans opération (min)"), "R29": ("part_porteur", None, "part des clients de la banque"),
    "R15": ("retraits_4_semaines", "retraits_8_semaines_avant", "retraits par jour sur 4 semaines (habituel = 8 semaines d'avant)"),
    "R16": ("part_jours_zero_tranche", "-", "part des jours à zéro sur la tranche horaire, depuis 4 semaines"),
    "R12": ("part_tranche_max", "hab_part_tranche_max", "part de la tranche horaire la plus chargée"),
    "R13": ("part_reseau_dominant", "hab_part_reseau_dominant", "part du type de carte dominant")}
INDIC_PREUVE = {k: v for k, v in INDIC_PREUVE.items() if v[0] in jour.columns}


def fmt_valeur(col, v):
    """Une valeur lisible selon sa nature : %, heure, euros, nombre."""
    if v is None or pd.isnull(v):
        return "-"
    if col.startswith("part_"):
        return pct(v, 0)
    if col.startswith("heure_"):
        return "{} h {:02d}".format(int(v), int(round(v % 1 * 60)) % 60)
    if col.startswith(("montant", "encaisse")):
        return nb(v) + " €"
    return nb(v, 0 if abs(v) >= 10 or float(v).is_integer() else 1)


# valeur habituelle de chaque indicateur clé : l'habitude déjà calculée, sinon la moyenne des 28 jours précédents du GAB
for col_, hab_, _ in set(INDIC_PREUVE.values()):
    if hab_ != "-" and (hab_ is None or hab_ not in jour.columns) and "ref_" + col_ not in jour.columns:
        jour["ref_" + col_] = glisse(jour[col_], ["num_automate"], 28, 10, "mean")


def ajouter_chiffres_cles(df):
    """Pour chaque GAB-jour en alerte : l'indicateur de l'anomalie la plus grave, sa valeur du jour et sa valeur habituelle."""
    code1 = df["codes"].str.split(" ").str[0].fillna("")
    df["indicateur clé"], df["valeur du jour"], df["valeur habituelle"] = "", "", ""
    zp = [v for v in (10, 20, 50) if "z_part_{}".format(v) in df.columns]
    for code_ in code1[code1 != ""].unique():
        m0_ = code1 == code_
        if code_ in ("R18", "R18b", "R18i") and zp:
            # la coupure dont la part s'écarte le plus de l'habitude ce jour-là (10 €, 20 € ou 50 €)
            choix = df.loc[m0_, ["z_part_{}".format(v) for v in zp]].abs().fillna(0).idxmax(axis=1).str.replace("z_part_", "")
            for v_ in choix.unique():
                m_ = m0_ & (choix.reindex(df.index) == v_)
                col_, hab_ = "part_" + v_, "hab_part_" + v_
                df.loc[m_, "indicateur clé"] = "part des billets de {} €".format(v_)
                df.loc[m_, "valeur du jour"] = [fmt_valeur(col_, v) for v in df.loc[m_, col_]]
                df.loc[m_, "valeur habituelle"] = [fmt_valeur(col_, v) for v in df.loc[m_, hab_]]
            continue
        col_, hab_, lib_ = INDIC_PREUVE.get(code_, ("nb_retraits", "hab_retraits", "retraits"))
        m_ = m0_
        ref_ = df.loc[m_, hab_] if hab_ and hab_ in df.columns else df.loc[m_, "ref_" + col_] if (hab_ != "-" and "ref_" + col_ in df.columns) else pd.Series(np.nan, index=df.index[m_])
        df.loc[m_, "indicateur clé"] = lib_
        df.loc[m_, "valeur du jour"] = [fmt_valeur(col_, v) for v in df.loc[m_, col_]]
        df.loc[m_, "valeur habituelle"] = [fmt_valeur(col_, v) for v in ref_]
    df["action conseillée"] = ""
    for code_ in code1[code1 != ""].unique():
        df.loc[code1 == code_, "action conseillée"] = action_conseillee(code_, GRAVITE_REGLE.get(code_, 1))
    df["le chiffre qui le prouve"] = np.where(df["indicateur clé"] != "", df["indicateur clé"] + " : " + df["valeur du jour"] + " ce jour-là, " + df["valeur habituelle"] + " d'habitude", "")
    return df


# --- la fiche de preuve d'un GAB un jour donné
INDIC_FICHE = [("nb_retraits", "retraits"), ("montant_total", "montant retiré"), ("montant_moyen", "montant moyen par retrait"), ("heure_premier_retrait", "heure du premier retrait"),
               ("heure_dernier_retrait", "heure du dernier retrait"), ("nb_inaboutis", "retraits échoués"), ("nb_captures", "cartes avalées"),
               ("nb_retraits_repetes_10min", "retraits refaits < 10 min"), ("part_10", "part des billets de 10 €"), ("part_20", "part des billets de 20 €"), ("part_50", "part des billets de 50 €"),
               ("nb_billets_inferieurs_montant", "retraits servis en moins"), ("nb_billets_superieurs_montant", "retraits servis en trop"), ("nb_ret_exotique", "cartes étrangères"),
               ("nb_ret_cup", "cartes UnionPay"), ("nb_ret_amex", "cartes Amex"), ("nb_0h_6h", "retraits entre 0 h et 6 h"), ("encaisse_fin", "encaisse en fin de journée"),
               ("nb_recharges", "chargements"), ("nb_incoherences_encaisse", "incohérences d'encaisse")]
INDIC_FICHE = [x for x in INDIC_FICHE if x[0] in jour.columns]


def verifier_gab(gab, date_, graphique=True):
    """Fiche de preuve : chaque indicateur du jour, les 4 mêmes jours de semaine précédents, l'habitude, les 8 semaines d'avant."""
    d_ = pd.Timestamp(date_)
    g_ = jour[jour["num_automate"] == str(gab)].set_index("date_jour").sort_index()
    if d_ not in g_.index:
        note("GAB {} absent de la fiche le {}.".format(gab, d_.strftime("%d/%m/%Y")), "bad")
        return None
    r_ = g_.loc[d_]
    # si l'alerte dure depuis plusieurs jours (GAB à l'arrêt, baisse qui dure), on compare aux jours d'avant son début
    debut_ = d_
    while (debut_ - pd.Timedelta(days=1)) in g_.index and g_.loc[debut_ - pd.Timedelta(days=1), "niveau"] != "Normal" and (d_ - debut_).days < 120:
        debut_ = debut_ - pd.Timedelta(days=1)
    if r_["nb_retraits"] == 0 and pd.notnull(r_.get("dernier_jour_actif")):
        debut_ = min(debut_, pd.Timestamp(r_["dernier_jour_actif"]) + pd.Timedelta(days=1))
    x_ = debut_ - pd.Timedelta(days=1)
    while x_.dayofweek != d_.dayofweek:
        x_ = x_ - pd.Timedelta(days=1)
    memes = [x_ - pd.Timedelta(days=7 * k) for k in range(3, -1, -1)]          # les 4 mêmes jours de semaine juste avant le début de l'alerte
    huit = g_.loc[debut_ - pd.Timedelta(days=56): debut_ - pd.Timedelta(days=1)]
    cols_alerte = {INDIC_PREUVE.get(c_, ("nb_retraits",))[0] for c_ in str(r_["codes"]).split() if c_}
    lignes = []
    for col_, lib_ in INDIC_FICHE:
        v_ = r_[col_]
        prec = g_.reindex(memes)[col_]
        h_ = prec.median()
        mx_, mn_ = huit[col_].max(), huit[col_].min()
        if pd.isnull(v_) or huit[col_].notnull().sum() < 5:
            verdict = ""
        elif v_ > mx_ and v_ > 0:
            verdict = "▲ plus haut des 8 dernières semaines"
        elif v_ < mn_:
            verdict = "▼ plus bas des 8 dernières semaines"
        else:
            verdict = "dans ses habitudes"
        lignes.append({"indicateur": ("● " if col_ in cols_alerte else "") + lib_, "ce jour-là": fmt_valeur(col_, v_),
                       "les 4 mêmes jours de semaine avant ({})".format(JOURS_FR[d_.dayofweek] + "s"): " · ".join(fmt_valeur(col_, x) for x in prec.values),
                       "habitude (valeur typique de ces 4 jours)": fmt_valeur(col_, h_), "plus bas / plus haut des 8 semaines d'avant": "{} / {}".format(fmt_valeur(col_, mn_), fmt_valeur(col_, mx_)),
                       "verdict": verdict})
    bandeau("Vérifier le diagnostic : GAB {} le {}".format(gab, jfr(d_)), "niveau {} ; rang {} du jour".format(r_["niveau"], int(r_["rang_jour"])) if pd.notnull(r_.get("rang_jour")) else "")
    note("<b>Diagnostic</b> : {}".format(_html(r_["diagnostic"]) or "rien d'anormal"), "info")
    if debut_ < d_:
        note("L'alerte dure depuis le <b>{}</b> ({} jours) : le GAB est comparé à ses jours ordinaires <b>avant</b> cette date.".format(jfr(debut_), (d_ - debut_).days + 1), "warn")
    display(tableau(pd.DataFrame(lignes), "Les chiffres du GAB ce jour-là, face à ses jours ordinaires (● = indicateur de l'anomalie)", n=40))
    if graphique:
        s_ = g_.loc[d_ - pd.Timedelta(days=56): min(FIN, d_ + pd.Timedelta(days=7))].reset_index()
        g_courbes(s_["date_jour"], [("retraits", s_["nb_retraits"], BLEU), ("habitude", s_["hab_retraits"], GRIS, "dash")], "GAB {} : retraits des 8 semaines autour du {}".format(gab, d_.strftime("%d/%m")),
                  y_titre="retraits", hauteur=240, lignes_v=[(d_, "jour vérifié")])
    return pd.DataFrame(lignes)


# --- les chiffres du jour : le parc, et les 3 GAB en tête pour chaque indicateur
PODIUM = [("nb_retraits", "Le plus de retraits", "retraits", "hab_retraits"), ("montant_total", "Le plus d'argent retiré", "€", "hab_montant_total"),
          ("nb_captures", "Le plus de cartes avalées", "cartes", "hab_captures"), ("nb_inaboutis", "Le plus de retraits échoués", "retraits échoués", "hab_inaboutis"),
          ("nb_ret_exotique", "Le plus de cartes étrangères", "cartes", None), ("nb_ret_cup", "Le plus de cartes UnionPay", "cartes", None),
          ("nb_ret_amex", "Le plus de cartes Amex", "cartes", None), ("nb_ret_diners", "Le plus de cartes Diners", "cartes", None), ("nb_ret_jcb", "Le plus de cartes JCB", "cartes", None),
          ("nb_0h_6h", "Le plus de retraits de nuit (0 h – 6 h)", "retraits", None), ("nb_retraits_repetes_10min", "Le plus de retraits refaits < 10 min", "retraits", "hab_repetes")]
PODIUM = [p_ for p_ in PODIUM if p_[0] in jour.columns]
for col_, _, _, hab_ in PODIUM:
    if (hab_ is None or hab_ not in jour.columns) and "ref_" + col_ not in jour.columns:
        jour["ref_" + col_] = glisse(jour[col_], ["num_automate"], 28, 10, "mean")
MEDAILLES = ["#d4a017", "#8a8f98", "#b0703c"]


def podiums_du_jour(d_, n=3):
    """Pour un jour : les n GAB en tête de chaque indicateur, avec leur montant et leur habitude."""
    j_ = jour[jour["date_jour"] == d_]
    out = []
    for col_, titre_, unite_, hab_ in PODIUM:
        t_ = j_[j_[col_] > 0].nlargest(n, col_)
        for k_, (_, x) in enumerate(t_.iterrows()):
            ref_ = x[hab_] if hab_ and hab_ in x.index else x.get("ref_" + col_, np.nan)
            out.append({"jour": d_, "indicateur": titre_, "rang": k_ + 1, "GAB": x["num_automate"], "ville": x.get("ville", ""), "valeur": x[col_], "unité": unite_,
                        "habitude du GAB": ref_, "retraits ce jour-là": x["nb_retraits"], "montant retiré (€)": x["montant_total"], "niveau": x["niveau"]})
    # plus fortes hausses et baisses face à l'habitude (GAB d'au moins 30 retraits par jour d'habitude)
    b_ = j_[j_["hab_retraits"] >= 30].assign(_r=lambda t: t["nb_retraits"] / t["hab_retraits"])
    for titre_, t_ in [("Plus forte hausse face à son habitude", b_.nlargest(n, "_r")), ("Plus forte baisse face à son habitude (hors arrêt)", b_[b_["nb_retraits"] > 0].nsmallest(n, "_r"))]:
        for k_, (_, x) in enumerate(t_.iterrows()):
            out.append({"jour": d_, "indicateur": titre_, "rang": k_ + 1, "GAB": x["num_automate"], "ville": x.get("ville", ""), "valeur": x["nb_retraits"], "unité": "retraits",
                        "habitude du GAB": x["hab_retraits"], "retraits ce jour-là": x["nb_retraits"], "montant retiré (€)": x["montant_total"], "niveau": x["niveau"]})
    return pd.DataFrame(out)


def chiffres_du_jour(date_, n=3):
    """Les chiffres du parc un jour donné (face aux 4 mêmes jours de semaine précédents) et les podiums par indicateur."""
    d_ = pd.Timestamp(date_)
    j_ = jour[jour["date_jour"] == d_]
    if j_.empty:
        note("Le {} n'est pas dans la fiche.".format(d_.strftime("%d/%m/%Y")), "bad")
        return None
    ref_ = jour[(jour["date_jour"] < d_) & (jour["date_jour"] >= d_ - pd.Timedelta(days=28)) & (jour["jour_semaine"] == d_.dayofweek + 1)]
    nref = max(1, ref_["date_jour"].nunique())
    sans = j_["nb_retraits"] == 0
    arret = sans & (j_["part_zero"].fillna(0) <= 0.15) & (j_["hab_retraits"].fillna(0) >= 10)
    items = []
    if "statut_gab" in j_.columns:
        st_ = j_["statut_gab"].astype(str).str.upper()
        items.append(("GAB actifs au référentiel", nb((st_ == "ACTIF").sum()), "{} non actifs (fermés au référentiel)".format(nb((st_ != "ACTIF").sum()))))
    items += [("GAB qui ont servi", nb((~sans).sum()), "au moins un retrait ce jour-là"),
              ("GAB sans aucun retrait", nb(sans.sum()), "dont {} à l'arrêt alors qu'ils servent d'habitude ce jour-là".format(nb(arret.sum())), "down" if arret.sum() else "")]
    for lib_, col_ in [("Retraits", "nb_retraits"), ("Montant retiré", "montant_total"), ("Cartes avalées", "nb_captures"), ("Retraits échoués", "nb_inaboutis"), ("Cartes étrangères", "nb_ret_exotique")]:
        if col_ in j_.columns:
            v_, r_ = j_[col_].sum(), ref_[col_].sum() / nref if len(ref_) else np.nan
            e_ = (v_ / r_ - 1) * 100 if r_ and pd.notnull(r_) else np.nan
            items.append((lib_, nb(v_) + (" €" if col_ == "montant_total" else ""), "{}{} % face aux 4 {}s précédents".format("+" if e_ >= 0 else "−", nb(abs(e_), 0), JOURS_FR[d_.dayofweek]) if pd.notnull(e_) else "pas de référence",
                          "" if pd.isnull(e_) or abs(e_) < 15 or (r_ or 0) < 10 else "down"))
    tuiles(items)
    pod = podiums_du_jour(d_, n)
    html = ['<div class="g-kpis">']
    for titre_ in pod["indicateur"].unique() if len(pod) else []:
        t_ = pod[pod["indicateur"] == titre_]
        lignes = ""
        for _, x in t_.iterrows():
            hab_ = "d'habitude : " + (fmt_valeur("montant", x["habitude du GAB"]) if x["unité"] == "€" else nb(x["habitude du GAB"], 1 if pd.notnull(x["habitude du GAB"]) and x["habitude du GAB"] < 10 else 0)) if pd.notnull(x["habitude du GAB"]) else ""
            u_ = x["unité"] if x["valeur"] > 1 else x["unité"].replace("retraits", "retrait").replace("cartes", "carte").replace("échoués", "échoué")
            val_ = fmt_valeur("montant", x["valeur"]) if x["unité"] == "€" else "{} {}".format(nb(x["valeur"]), u_)
            lignes += ('<tr><td style="padding:3px 6px;border:0"><span style="display:inline-block;width:18px;height:18px;border-radius:9px;background:{};color:#fff;font-size:11px;text-align:center;line-height:18px">{}</span></td>'
                       '<td style="padding:3px 6px;border:0;text-align:left"><b>{}</b><div style="font-size:11px;color:#57606a">{} · {} retraits · {}</div></td>'
                       '<td style="padding:3px 6px;border:0;text-align:right;white-space:nowrap"><b>{}</b><div style="font-size:11px;color:#57606a">{}</div></td></tr>').format(
                MEDAILLES[min(int(x["rang"]) - 1, 2)], int(x["rang"]), x["GAB"], _html(x["ville"]) or "", nb(x["retraits ce jour-là"]), fmt_valeur("montant", x["montant retiré (€)"]), val_, hab_)
        html.append('<div class="g-kpi" style="flex:1 1 330px;max-width:390px"><div class="l" style="text-transform:none;font-size:12.5px;font-weight:600;color:#1f2328">{}</div>'
                    '<table style="border-collapse:collapse;width:100%;font-size:12.5px">{}</table></div>'.format(titre_, lignes))
    html.append("</div>")
    display(HTML(CSS + '<div style="font-weight:600;margin:10px 0 4px 0">Les {} GAB en tête, indicateur par indicateur (avec leur nombre de retraits, leur montant et leur habitude)</div>'.format(n) + "".join(html)))
    return pod

<a href="#sommaire" style="font-size:12px">↑ retour au sommaire</a>
<a id="s11"></a>
## 11. Que trouve la détection sur toute la période ?
**En bref** : chaque GAB reçoit chaque jour un **niveau**, celui de son anomalie la plus grave :

- **Critique** : à traiter tout de suite ;
- **Élevé** : à traiter ;
- **À surveiller** : à garder à l'œil ;
- **Normal** : rien à signaler.

Chaque jour, les GAB sont aussi **classés** du plus au moins atypique : le **rang du jour** 1 est le GAB le plus atypique du parc ce jour-là.

**Comment lire les nombres** : un **GAB-jour** est un GAB un jour donné. On compte d'un côté les GAB-jours en alerte (le travail de chaque
jour), de l'autre les **GAB différents** concernés sur la période.

In [ ]:
jour["score"] = jour[Z].abs().sum(axis=1).fillna(0)        # écart total : départage les GAB à score de règles égal
ev = []
for code, nom, grav, m, textes in REGLES:
    m = m.reindex(J.index).fillna(False)
    if m.any():
        e = J.loc[m, ["num_automate", "date_jour"]].copy()
        e["code"], e["anomalie"], e["gravite"] = code, nom, grav
        e["texte"] = textes.reindex(J.index)[m].astype(str).values
        ev.append(e)
evenements = pd.concat(ev, ignore_index=True) if ev else pd.DataFrame(columns=["num_automate", "date_jour", "code", "anomalie", "gravite", "texte"])
evenements["famille_anomalie"] = evenements["code"].map(FAMILLE_ANOMALIE)
evenements = evenements.sort_values(["num_automate", "date_jour", "gravite"], ascending=[True, True, False])
resume = evenements.groupby(["num_automate", "date_jour"]).agg(
    score_regles=("gravite", "sum"), gravite_max=("gravite", "max"), nb_regles=("code", "size"),
    codes=("code", lambda s: " ".join(s)), anomalies=("anomalie", lambda s: " + ".join(s)), diagnostic=("texte", lambda s: " + ".join(s))).reset_index()
jour = jour.drop(columns=[c for c in ["score_regles", "gravite_max", "nb_regles", "codes", "anomalies", "diagnostic"] if c in jour.columns]) \
           .merge(resume, on=["num_automate", "date_jour"], how="left")
for c, v in [("score_regles", 0), ("gravite_max", 0), ("nb_regles", 0), ("codes", ""), ("anomalies", ""), ("diagnostic", "")]:
    jour[c] = jour[c].fillna(v)
    if v == 0:
        jour[c] = jour[c].astype(int)
jour["niveau"] = np.select([jour["gravite_max"] >= 3, jour["gravite_max"] >= 2, jour["gravite_max"] >= 1], ["Critique", "Élevé", "À surveiller"], "Normal")
jour["resume"] = jour["diagnostic"].str.split(" + ").str[0].fillna("")
jour["nb_causes"] = jour["nb_regles"]
jour = ajouter_chiffres_cles(jour)
if "FAMILLE" in globals():
    jour["famille"] = jour["num_automate"].map(FAMILLE).fillna("historique court")
J = jour
ident = ["num_automate"] + (["code_site_implantation"] if "code_site_implantation" in jour.columns else []) + (["famille"] if "famille" in jour.columns else [])
# rang de chaque GAB dans sa journée (1 = le plus atypique), pour tous les jours
_ordre = jour.sort_values(["date_jour", "score_regles", "gravite_max", "score"], ascending=[True, False, False, False], na_position="last")
jour["rang_jour"] = (_ordre.groupby("date_jour").cumcount() + 1).reindex(jour.index)

P = jour[jour["date_jour"] >= DEBUT_PERIODE]                 # la période examinée
EV = evenements[evenements["date_jour"] >= DEBUT_PERIODE]
alertes = P[P["niveau"] != "Normal"].copy()
_n_ce = int(P["niveau"].isin(["Critique", "Élevé"]).sum())
_g_ce = P.loc[P["niveau"].isin(["Critique", "Élevé"]), "num_automate"].nunique()
bandeau("Résultat de la détection sur la période", "du {} au {} : {} jours, {} GAB, {} GAB-jours".format(
    DEBUT_PERIODE.strftime("%d/%m/%Y"), FIN.strftime("%d/%m/%Y"), nb(P["date_jour"].nunique()), nb(P["num_automate"].nunique()), nb(len(P))))
tuiles([("GAB-jours Critique", nb((P["niveau"] == "Critique").sum()), "{} GAB distincts".format(nb(P.loc[P["niveau"] == "Critique", "num_automate"].nunique()))),
        ("GAB-jours Élevé", nb((P["niveau"] == "Élevé").sum()), "{} GAB distincts".format(nb(P.loc[P["niveau"] == "Élevé", "num_automate"].nunique()))),
        ("GAB-jours À surveiller", nb((P["niveau"] == "À surveiller").sum()), "{} GAB distincts".format(nb(P.loc[P["niveau"] == "À surveiller", "num_automate"].nunique()))),
        ("Alertes par jour", nb(_n_ce / max(1, P["date_jour"].nunique()), 1), "GAB Critique ou Élevé un jour moyen"),
        ("Part du parc en alerte", pct(_n_ce / max(1, len(P)), 2), "des GAB-jours sont Critique ou Élevé"),
        ("GAB jamais en alerte", nb(P["num_automate"].nunique() - _g_ce), "sur {} : {} des GAB n'ont jamais été Critique ni Élevé".format(nb(P["num_automate"].nunique()), pct(1 - _g_ce / max(1, P["num_automate"].nunique()), 0)))])

# la chronologie : chaque jour, combien de GAB par niveau
chrono = P.groupby(["date_jour", "niveau"]).size().unstack(fill_value=0).reindex(columns=ORDRE_NIVEAUX, fill_value=0)
g_barres(chrono.index.strftime("%d/%m/%Y"), [(niv, chrono[niv].values, COULEURS_NIVEAU[niv]) for niv in ["Critique", "Élevé", "À surveiller"]],
         "Chaque jour de la période : nombre de GAB en alerte, par niveau", y_titre="GAB", hauteur=340, empile=True)
if P["date_jour"].nunique() > 90:
    _m = P.assign(mois=P["date_jour"].dt.to_period("M").astype(str))
    _mm = pd.DataFrame({"GAB distincts Critique ou Élevé": _m[_m["niveau"].isin(["Critique", "Élevé"])].groupby("mois")["num_automate"].nunique(),
                        "GAB-jours Critique ou Élevé": _m[_m["niveau"].isin(["Critique", "Élevé"])].groupby("mois").size()}).fillna(0)
    g_barres(_mm.index, [("GAB-jours", _mm["GAB-jours Critique ou Élevé"].values, BLEU_CLAIR), ("GAB distincts", _mm["GAB distincts Critique ou Élevé"].values, BLEU)],
             "Par mois : GAB-jours en alerte (Critique ou Élevé) et GAB distincts concernés", y_titre="", hauteur=320)
# combien de jours un GAB en alerte l'est-il ? (persistance)
_jpg = P[P["niveau"].isin(["Critique", "Élevé"])].groupby("num_automate").size()
if len(_jpg):
    _b = pd.cut(_jpg, [0, 1, 2, 5, 10, 30, 10000], labels=["1 jour", "2 jours", "3 à 5", "6 à 10", "11 à 30", "plus de 30"]).value_counts().reindex(["1 jour", "2 jours", "3 à 5", "6 à 10", "11 à 30", "plus de 30"]).fillna(0)
    g_barres(_b.index, [("GAB", _b.values, BLEU)], "Les GAB en alerte le sont combien de jours sur la période ? (nombre de GAB par durée)", y_titre="GAB", hauteur=280)
# par famille de GAB
if "famille" in P.columns:
    _pf = P.groupby("famille").agg(gab=("num_automate", "nunique"), gab_jours=("niveau", "size"))
    _pf["GAB-jours Critique ou Élevé"] = P[P["niveau"].isin(["Critique", "Élevé"])].groupby("famille").size().reindex(_pf.index).fillna(0).astype(int)
    _pf["GAB distincts en alerte"] = P[P["niveau"].isin(["Critique", "Élevé"])].groupby("famille")["num_automate"].nunique().reindex(_pf.index).fillna(0).astype(int)
    _pf["part des GAB-jours en alerte"] = (_pf["GAB-jours Critique ou Élevé"] / _pf["gab_jours"]).map(lambda v: pct(v, 2))
    _pf = _pf.reindex([f for f in ORDRE_FAMILLES if f in _pf.index]).reset_index().rename(columns={"gab": "GAB", "gab_jours": "GAB-jours"})
    display(tableau(_pf, "Les alertes selon la famille de GAB"))
retenir(["Sur <b>{}</b> jours et <b>{}</b> GAB, <b>{}</b> GAB-jours sont Critique ou Élevé, soit <b>{}</b> par jour en moyenne et <b>{}</b> des GAB-jours.".format(
             nb(P["date_jour"].nunique()), nb(P["num_automate"].nunique()), nb(_n_ce), nb(_n_ce / max(1, P["date_jour"].nunique()), 1), pct(_n_ce / max(1, len(P)), 2)),
         "Ces alertes concernent <b>{}</b> GAB distincts ; <b>{}</b> GAB ({}) n'ont jamais été en alerte Critique ou Élevé sur la période.".format(
             nb(_g_ce), nb(P["num_automate"].nunique() - _g_ce), pct(1 - _g_ce / max(1, P["num_automate"].nunique()), 0)),
         "La famille d'anomalie la plus fréquente est <b>{}</b> ({} GAB-jours).".format(*[(f, nb(n)) for f, n in EV[EV["gravite"] >= 2].groupby("famille_anomalie").size().sort_values(ascending=False).head(1).items()][0]) if len(EV[EV["gravite"] >= 2]) else "",
         "Le nombre de jours en alerte par GAB (graphique ci-dessus) distingue les incidents ponctuels (1 ou 2 jours) des GAB durablement en difficulté."])

### Toutes les alertes Critique et Élevé
Une ligne par GAB et par jour, les plus récentes en premier. Un arrêt ou une baisse qui dure est regroupé sur une seule ligne avec le
nombre de jours. Le diagnostic est abrégé ; l'Excel le donne en entier.

In [ ]:
def court(texte, n=2):
    parts_ = [p_ for p_ in str(texte).split(" + ") if p_]
    return " + ".join(parts_[:n]) + (" (+{} autre{})".format(len(parts_) - n, "s" if len(parts_) - n > 1 else "") if len(parts_) > n else "")


LIBELLES = {"num_automate": "GAB", "code_site_implantation": "code site", "date_jour": "jour", "niveau": "niveau", "score_regles": "score", "anomalies": "anomalies",
            "diagnostic": "diagnostic", "voisins": "voisins", "rang": "rang", "rang_jour": "rang du jour", "prioritaire": "prioritaire", "famille": "famille", "nb_causes": "causes",
            "nb_retraits": "retraits", "hab_retraits": "habitude", "ville": "ville", "nb_jours": "jours"}
alertes = alertes.sort_values(["date_jour", "score_regles", "score"], ascending=[False, False, False])
persist = alertes["codes"].str.contains(r"R15|R16|R02") & ~alertes["codes"].str.contains(r"R0[1,5-9]|R1[0-4]|R1[7-9]|R2")
dedoublon = alertes[persist].groupby("num_automate").agg(date_jour=("date_jour", "min"), niveau=("niveau", "first"), score_regles=("score_regles", "max"),
                                                          anomalies=("anomalies", "first"), codes=("codes", "first"), diagnostic=("diagnostic", "first"), nb_jours=("date_jour", "size")).reset_index()
for c in ident:
    if c != "num_automate":
        dedoublon = dedoublon.merge(alertes[["num_automate", c]].drop_duplicates("num_automate"), on="num_automate", how="left")
dedoublon["diagnostic"] = dedoublon["diagnostic"] + " (" + dedoublon["nb_jours"].astype(str) + " jour(s) sur la période)"
vue_alertes = pd.concat([alertes[~persist], dedoublon], ignore_index=True, sort=False).sort_values(["date_jour", "score_regles"], ascending=[False, False])
_v = vue_alertes[vue_alertes["niveau"] != "À surveiller"].copy()
_v["diagnostic"] = _v["diagnostic"].map(court)
note("<b>{}</b> lignes Critique ou Élevé sur la période ({} GAB-jours, {} GAB distincts). Les {} plus récentes :".format(
    nb(len(_v)), nb(_n_ce), nb(_g_ce), min(60, len(_v))))
display(tableau(_v[["date_jour", "niveau"] + ident + ["anomalies", "le chiffre qui le prouve", "diagnostic"]], n=60, libelles=LIBELLES, longues=("diagnostic", "le chiffre qui le prouve")))

### Ce qui s'explique sans être une alerte
Certains écarts ont une explication connue : hausse partagée par les voisins (marché, fête), hausse déjà vue l'an dernier à la même date,
journée aux données incomplètes. Ils figurent dans le diagnostic pour éclairer le lecteur, sans compter comme une alerte.

In [ ]:
info = EV[EV["gravite"] == 0]
if len(info):
    _i = info.groupby("anomalie").agg(**{"GAB-jours": ("num_automate", "size"), "GAB distincts": ("num_automate", "nunique")}).reset_index().sort_values("GAB-jours", ascending=False)
    display(tableau(_i, "Informations sur la période"))
    display(tableau(info.sort_values("date_jour", ascending=False)[["date_jour", "num_automate", "anomalie", "texte"]].rename(columns={"texte": "diagnostic"}), "Les 20 plus récentes", n=20, libelles=LIBELLES))
else:
    note("Aucune information de gravité 0 sur la période.")

### Comment on choisit les GAB les plus atypiques
**En bref** : un GAB n'entre dans la liste du jour **que s'il a au moins une anomalie de niveau À surveiller ou plus**. Les GAB en
alerte sont ensuite classés : d'abord le **niveau** de leur anomalie la plus grave (Critique, puis Élevé, puis À surveiller), puis le
**nombre et la gravité** de leurs anomalies, puis l'ampleur de leurs écarts. La liste garde au plus 15 GAB ; les 10 premiers sont
**prioritaires**.

**Et un jour sans GAB atypique ?** La liste est vide, et le notebook le dit. On n'invente pas d'anomalie pour remplir la liste.

**Ce que vous voyez** : l'entonnoir du dernier jour (du parc entier à la liste prioritaire), puis, sur toute la période, combien de GAB
sont en alerte chaque jour et combien de jours n'en ont aucun.

In [ ]:
_d = jour[jour["date_jour"] == FIN]
_ent = pd.Series({"GAB dans la fiche": len(_d), "avec au moins un écart expliqué ou une anomalie": int((_d["nb_causes"] > 0).sum()),
                  "en alerte (À surveiller ou plus)": int((_d["niveau"] != "Normal").sum()), "Critique ou Élevé": int(_d["niveau"].isin(["Critique", "Élevé"]).sum()),
                  "dans la liste du jour (au plus {})".format(TOP_LISTE): int(min(TOP_LISTE, (_d["niveau"] != "Normal").sum())),
                  "prioritaires (au plus {})".format(TOP_PRIORITAIRE): int(min(TOP_PRIORITAIRE, (_d["niveau"] != "Normal").sum()))})
g_barres(list(_ent.index[::-1]), [("GAB", _ent.values[::-1], [GRIS_CLAIR, BLEU_CLAIR, COULEURS_NIVEAU["À surveiller"], COULEURS_NIVEAU["Élevé"], COULEURS_NIVEAU["Critique"], COULEURS_NIVEAU["Critique"]][::-1])],
         "De tout le parc à la liste prioritaire : l'entonnoir du {}".format(FIN.strftime("%d/%m/%Y")), hauteur=320, horizontal=True)
_npj = P.assign(_a=(P["niveau"] != "Normal").astype(int)).groupby("date_jour")["_a"].sum()
_b = pd.cut(_npj, [-1, 0, 2, 5, 10, 15, 1e9], labels=["aucun", "1 à 2", "3 à 5", "6 à 10", "11 à 15", "plus de 15"]).value_counts().reindex(["aucun", "1 à 2", "3 à 5", "6 à 10", "11 à 15", "plus de 15"]).fillna(0)
tuiles([("Jours sans aucun GAB atypique", nb(int((_npj == 0).sum())), "sur {} jours : la liste du jour était vide".format(nb(len(_npj)))),
        ("GAB en alerte un jour typique", nb(_npj.median(), 0), "le plus haut : {} le {}".format(nb(_npj.max()), _npj.idxmax().strftime("%d/%m/%Y"))),
        ("Jours où la liste est pleine", nb(int((_npj >= TOP_LISTE).sum())), "plus de {} GAB en alerte : seuls les {} premiers sont listés".format(TOP_LISTE, TOP_LISTE))])
g_barres(list(_b.index), [("jours", _b.values, BLEU)], "Combien de GAB en alerte chaque jour ? (nombre de jours)", hauteur=260)

<a href="#sommaire" style="font-size:12px">↑ retour au sommaire</a>
<a id="s12"></a>
## 12. Le dernier jour : les GAB à regarder en priorité
**C'est la liste quotidienne transmise au métier** : les GAB les plus atypiques du dernier jour complet, les 10 premiers marqués
prioritaires, chacun avec son diagnostic en clair et ses voisins. Puis la carte du jour et les chiffres du parc ce jour-là.

In [ ]:
DERNIER = FIN
photo = jour[jour["date_jour"] == DERNIER].copy()
photo["score"] = photo["score"].fillna(0)
photo = photo.sort_values(["score_regles", "gravite_max", "score"], ascending=[False, False, False]).reset_index(drop=True)
photo["rang"] = np.arange(1, len(photo) + 1)
EN_ALERTE = photo["niveau"] != "Normal"
photo["prioritaire"] = np.where((photo["rang"] <= TOP_PRIORITAIRE) & EN_ALERTE, "oui", "")
photo["liste"] = ((photo["rang"] <= TOP_LISTE) & EN_ALERTE).astype(int)          # un GAB « Normal » n'entre jamais dans la liste
photo.loc[photo["diagnostic"] == "", "diagnostic"] = np.where(photo.loc[photo["diagnostic"] == "", "hab_retraits"].isnull(), "historique insuffisant pour juger", "rien d'anormal")
photo["voisins"] = photo["num_automate"].map(lambda x: ", ".join("{} ({} km)".format(v, d) for v, d in VOISINS.get(x, [])))
top = photo[photo["liste"] == 1].copy()
bandeau("{} {} — {} GAB à regarder".format(JOURS_FR[DERNIER.dayofweek].capitalize(), DERNIER.strftime("%d/%m/%Y"), len(top)) if len(top) else "{} {} — aucun GAB atypique".format(JOURS_FR[DERNIER.dayofweek].capitalize(), DERNIER.strftime("%d/%m/%Y")),
        "parc ce jour-là : " + pastilles(photo["niveau"]) + (" — " + photo["ferie"].iloc[0] if "ferie" in photo.columns and photo["ferie"].iloc[0] else ""))
_t = top.copy()
_t["diagnostic"] = _t["diagnostic"].map(court)
_t = _t.rename(columns={"hab_retraits": "habitude"})
if len(_t):
    display(tableau(_t[["rang", "prioritaire", "niveau"] + ident + ["nb_retraits", "habitude", "montant_total", "le chiffre qui le prouve", "diagnostic", "action conseillée", "voisins"]], n=TOP_LISTE,
                    libelles=dict(LIBELLES, montant_total="montant retiré (€)"), longues=("diagnostic", "le chiffre qui le prouve", "action conseillée"), dec=0))
else:
    note("<b>Aucun GAB atypique ce jour-là</b> : aucun GAB ne s'est écarté de ses habitudes au point de déclencher une alerte. "
         "C'est une bonne nouvelle, pas une panne de l'outil : la liste reste vide plutôt que d'afficher des GAB normaux.", "ok")
    _n = photo[photo["nb_causes"] > 0].head(5)
    if len(_n):
        display(tableau(_n[["num_automate", "anomalies", "diagnostic"]].rename(columns={"num_automate": "GAB"}), "Pour information seulement : les écarts expliqués du jour (aucun n'est une alerte)"))

### Carte du jour
Tous les GAB en gris ; les GAB de la liste du jour en couleur (rouge Critique, orange Élevé, jaune À surveiller), les prioritaires en plus
gros. **Survoler un point** affiche le GAB et son diagnostic.

In [ ]:
def carte_svg(photo_, top_, largeur=640):
    """Repli sans Plotly : carte SVG avec infobulles <title>."""
    p = photo_.copy()
    p["lat"], p["lon"] = pd.to_numeric(p["latitude"], errors="coerce"), pd.to_numeric(p["longitude"], errors="coerce")
    p = p[p["lat"].between(41, 51.5) & p["lon"].between(-5.5, 10)]
    lon0, lon1, lat0, lat1 = -5.5, 10.0, 41.0, 51.5
    hauteur = int(largeur * (lat1 - lat0) / ((lon1 - lon0) * np.cos(np.radians(46.5))))
    x = lambda lo: (lo - lon0) / (lon1 - lon0) * largeur
    y = lambda la: (lat1 - la) / (lat1 - lat0) * hauteur
    fond = "".join('<circle cx="{:.1f}" cy="{:.1f}" r="1.3" fill="#c8c8c8"/>'.format(x(lo), y(la)) for la, lo in zip(p["lat"], p["lon"]))
    pts = []
    for _, r in top_.merge(p[["num_automate", "lat", "lon"]], on="num_automate").sort_values("rang", ascending=False).iterrows():
        titre = "#{} {} — {}\n{}".format(int(r["rang"]), r["num_automate"], r["niveau"], r["resume"])
        pts.append('<circle cx="{:.1f}" cy="{:.1f}" r="{}" fill="{}" fill-opacity="0.85" stroke="#222" stroke-width="0.8"><title>{}</title></circle>'
                   .format(x(r["lon"]), y(r["lat"]), 9 if r["rang"] <= TOP_PRIORITAIRE else 6, COULEURS_NIVEAU.get(r["niveau"], "#9a9a9a"), _html(titre)))
    return '<svg xmlns="http://www.w3.org/2000/svg" width="{}" height="{}" style="background:#fafafa;border:1px solid #ddd">{}{}</svg>'.format(largeur, hauteur, fond, "".join(pts))


if "latitude" in photo.columns and "longitude" in photo.columns:
    _p = photo.copy()
    _p["lat"], _p["lon"] = pd.to_numeric(_p["latitude"], errors="coerce"), pd.to_numeric(_p["longitude"], errors="coerce")
    _p = _p[_p["lat"].between(41, 51.5) & _p["lon"].between(-5.5, 10)]
    if PLOTLY:
        _v = lambda r: "GAB {}{}".format(r["num_automate"], " — " + str(r["ville"]) if "ville" in r and pd.notnull(r["ville"]) else "")
        _groupes = []
        for niv in ["À surveiller", "Élevé", "Critique"]:
            _tp = top.merge(_p[["num_automate", "lat", "lon"]], on="num_automate")
            _tp = _tp[_tp["niveau"] == niv]
            if len(_tp):
                _groupes.append((niv, _tp["lon"], _tp["lat"], ["#{} {}<br>{}<br>{}".format(int(r["rang"]), _v(r), r["niveau"], court(r["diagnostic"], 1)) for _, r in _tp.iterrows()],
                                 COULEURS_NIVEAU[niv], [16 if r_ <= TOP_PRIORITAIRE else 11 for r_ in _tp["rang"]]))
        g_points(_p["lon"], _p["lat"], "Carte du {} : les {} GAB les plus atypiques (en couleur) sur le parc (en gris)".format(DERNIER.strftime("%d/%m/%Y"), len(top)),
                 textes=["{}<br>{}<br>{}".format(_v(r), r["niveau"], court(r["diagnostic"], 1)) for _, r in _p.iterrows()], couleurs=GRIS_CLAIR, tailles=4, hauteur=600, groupes=_groupes, aspect=1.4, axes=False)
    else:
        display(HTML(carte_svg(photo, top)))
else:
    note("Pas de latitude / longitude dans la fiche : carte impossible.", "warn")

### Le parc ce jour-là, comparé à un jour ordinaire
Chaque chiffre du jour est comparé à la moyenne des 4 mêmes jours de semaine précédents : un dimanche se compare aux dimanches.

In [ ]:
ref28 = jour[(jour["date_jour"] < DERNIER) & (jour["date_jour"] >= DERNIER - pd.Timedelta(days=28)) & (jour["jour_semaine"] == DERNIER.dayofweek + 1)]


def kpi_ligne(df):
    d = {"GAB dans la fiche": df["num_automate"].nunique(), "GAB actifs (au moins 1 retrait)": int((df["nb_retraits"] > 0).sum()),
         "GAB sans retrait": int((df["nb_retraits"] == 0).sum()), "retraits": int(df["nb_retraits"].sum()), "montant retiré (€)": int(df["montant_total"].sum()),
         "retraits par GAB actif": round(df.loc[df["nb_retraits"] > 0, "nb_retraits"].mean(), 1) if (df["nb_retraits"] > 0).any() else 0,
         "montant moyen par retrait (€)": round(df["montant_total"].sum() / max(1, df["nb_retraits"].sum()), 1)}
    for col, lib in [("nb_inaboutis", "retraits inaboutis"), ("nb_captures", "cartes capturées"), ("nb_ret_exotique", "retraits cartes étrangères"),
                     ("nb_retraits_repetes_10min", "retraits refaits < 10 min"), ("nb_0h_6h", "retraits de nuit (0 h – 6 h)")]:
        if col in df.columns:
            d[lib] = int(df[col].sum())
    return d


kpi_jour = kpi_ligne(photo)
kpi_ref = pd.DataFrame([kpi_ligne(g_) for _, g_ in ref28.groupby("date_jour")]).mean().round(1).to_dict() if len(ref28) else {}
kpi = pd.DataFrame({"jour analysé": pd.Series(kpi_jour), "moyenne des 4 mêmes jours de semaine précédents": pd.Series(kpi_ref)})
kpi["écart"] = ((kpi["jour analysé"] / kpi["moyenne des 4 mêmes jours de semaine précédents"].replace(0, np.nan) - 1) * 100).round(1)
_tk = []
for k_ in ["retraits", "montant retiré (€)", "GAB actifs (au moins 1 retrait)", "GAB sans retrait", "retraits inaboutis", "cartes capturées", "retraits cartes étrangères"]:
    if k_ in kpi.index:
        e_ = kpi.loc[k_, "écart"]
        _tk.append((k_, nb(kpi.loc[k_, "jour analysé"]), "{} % face aux 4 mêmes jours de semaine".format(("+" if e_ >= 0 else "") + nb(e_, 1)) if pd.notnull(e_) else "pas de référence",
                    "" if pd.isnull(e_) or abs(e_) < 10 or kpi.loc[k_, "moyenne des 4 mêmes jours de semaine précédents"] < 10 else ("up" if (e_ > 0) == (k_ in ("retraits", "montant retiré (€)", "GAB actifs (au moins 1 retrait)")) else "down")))
pod_dernier = chiffres_du_jour(DERNIER)
for niv in ORDRE_NIVEAUX:
    kpi.loc["GAB " + niv, "jour analysé"] = int((photo["niveau"] == niv).sum())
regles_jour = evenements[(evenements["date_jour"] == DERNIER) & (evenements["gravite"] >= 2)].groupby("anomalie").size().sort_values(ascending=False)
for r_, n_ in regles_jour.items():
    kpi.loc["alertes : " + r_, "jour analysé"] = int(n_)
kpi["écart"] = kpi["écart"].map(lambda v: "{:+.1f} %".format(v).replace(".", ",") if pd.notnull(v) else "")
kpi = kpi.fillna("")
display(tableau(kpi.reset_index().rename(columns={"index": "indicateur"}), "Tous les indicateurs du jour", n=60))

### Vérifier le diagnostic des premiers GAB de la liste
Pour les 3 premiers GAB en alerte du dernier jour, la **fiche de preuve** : chaque indicateur du jour à côté des 4 mêmes jours de
semaine précédents, de l'habitude et du plus bas / plus haut des 8 semaines d'avant. Les indicateurs marqués ● sont ceux de l'anomalie.
On voit ainsi, chiffres à l'appui, si le diagnostic est vrai. Pour n'importe quel GAB : `verifier_gab("numéro", "AAAA-MM-JJ")`.

In [ ]:
for _g in photo.loc[photo["niveau"] != "Normal", "num_automate"].head(3):
    verifier_gab(_g, DERNIER)

### Rejouer n'importe quel jour
**Pour une démonstration en direct** : `afficher_jour("AAAA-MM-JJ")` affiche, pour la date choisie, les chiffres du parc, les GAB les plus
atypiques avec leur diagnostic, et la carte. C'est exactement ce que contient l'onglet **Top du jour** de l'Excel pour cette date.

In [ ]:
def afficher_jour(date_, n=TOP_LISTE, preuves=3):
    """Le parc et les GAB les plus atypiques d'une date donnée (chiffres, liste, carte)."""
    d_ = pd.Timestamp(date_)
    ph = jour[jour["date_jour"] == d_].sort_values("rang_jour")
    if ph.empty:
        note("Le {} n'est pas dans la fiche (du {} au {}).".format(d_.strftime("%d/%m/%Y"), jour["date_jour"].min().strftime("%d/%m/%Y"), FIN.strftime("%d/%m/%Y")), "bad")
        return None
    fer = ph["ferie"].iloc[0] if "ferie" in ph.columns else ""
    n_alerte = int((ph["niveau"] != "Normal").sum())
    bandeau("{} — {}".format(jfr(d_).capitalize(), "{} GAB à regarder".format(min(n, n_alerte)) if n_alerte else "aucun GAB atypique"), "parc ce jour-là : " + pastilles(ph["niveau"]) + (" — " + fer if fer else ""))
    chiffres_du_jour(d_)
    ref = jour.iloc[0:0]
    nref = 1
    items = []
    for lib_, f_ in [("Retraits", lambda t: t["nb_retraits"].sum()), ("Montant retiré (€)", lambda t: t["montant_total"].sum()),
                     ("GAB actifs", lambda t: (t["nb_retraits"] > 0).sum()), ("GAB sans retrait", lambda t: (t["nb_retraits"] == 0).sum()),
                     ("Cartes capturées", lambda t: t["nb_captures"].sum()), ("Retraits inaboutis", lambda t: t["nb_inaboutis"].sum())] + \
                    ([("Cartes étrangères", lambda t: t["nb_ret_exotique"].sum())] if "nb_ret_exotique" in ph.columns else []):
        v_, r_ = f_(ph), f_(ref) / nref if len(ref) else np.nan
        e_ = (v_ / r_ - 1) * 100 if r_ and pd.notnull(r_) else np.nan
        pass
    t_ = ph[ph["niveau"] != "Normal"].head(n).copy()
    if t_.empty:
        note("<b>Aucun GAB atypique ce jour-là</b> : la liste est vide, le parc s'est comporté normalement.", "ok")
    t_["diagnostic"] = np.where(t_["diagnostic"] == "", np.where(t_["hab_retraits"].isnull(), "historique insuffisant pour juger", "rien d'anormal"), t_["diagnostic"])
    t_["diagnostic"] = t_["diagnostic"].map(court)
    t_["voisins"] = t_["num_automate"].map(lambda x: ", ".join("{} ({} km)".format(v, d) for v, d in VOISINS.get(x, [])))
    display(tableau(t_[["rang_jour", "niveau"] + ident + [c for c in ["ville"] if c in t_.columns] + ["nb_retraits", "hab_retraits", "montant_total", "anomalies", "le chiffre qui le prouve", "diagnostic", "action conseillée", "voisins"]].rename(
        columns={"hab_retraits": "habitude", "montant_total": "montant retiré (€)"}), libelles=LIBELLES, n=n, dec=0, longues=("diagnostic", "le chiffre qui le prouve", "action conseillée"))) if len(t_) else None
    if "latitude" in ph.columns:
        p_ = ph.copy()
        p_["lat"], p_["lon"] = pd.to_numeric(p_["latitude"], errors="coerce"), pd.to_numeric(p_["longitude"], errors="coerce")
        p_ = p_[p_["lat"].between(41, 51.5) & p_["lon"].between(-5.5, 10)]
        tp_ = p_[(p_["rang_jour"] <= n) & (p_["niveau"] != "Normal")]
        grp = []
        for niv in ["Normal", "À surveiller", "Élevé", "Critique"]:
            x_ = tp_[tp_["niveau"] == niv]
            if len(x_):
                grp.append((niv, x_["lon"], x_["lat"], ["#{} GAB {}<br>{}<br>{}".format(int(r["rang_jour"]), r["num_automate"], r["niveau"], court(r["diagnostic"], 1)) for _, r in x_.iterrows()],
                            COULEURS_NIVEAU[niv], [16 if r_ <= TOP_PRIORITAIRE else 11 for r_ in x_["rang_jour"]]))
        g_points(p_["lon"], p_["lat"], "Carte du {} : les {} GAB les plus atypiques (en couleur) sur le parc (en gris)".format(d_.strftime("%d/%m/%Y"), n),
                 textes=["GAB {}".format(x) for x in p_["num_automate"]], couleurs=GRIS_CLAIR, tailles=4, hauteur=560, groupes=grp, aspect=1.4, axes=False)
    for g_ in ph.loc[ph["niveau"] != "Normal", "num_automate"].head(preuves):
        verifier_gab(g_, d_, graphique=False)
    return ph.head(n)


_rejoue = afficher_jour(JOUR_A_REJOUER or CAS_INVERSION["date"])
note("Pour un autre jour : <code>afficher_jour(\"AAAA-MM-JJ\")</code> dans une nouvelle cellule. Pour le même jour dans l'Excel : onglet <b>Top du jour</b>, filtrer la colonne « jour ».")

### À quoi ressemble chaque anomalie ? Un exemple réel par type
Pour chaque anomalie déclenchée sur la période, le cas le plus net : le GAB, la date, la phrase de diagnostic, et la courbe de l'indicateur
concerné sur les 3 semaines autour. **Le point rouge marque le jour détecté.** Ces exemples sont aussi dans l'onglet « Exemples par
anomalie » de l'Excel.

In [ ]:
INDIC_EXEMPLE = {"R05": ("nb_inaboutis", None, "retraits inaboutis"), "R05s": ("nb_inaboutis", None, "retraits inaboutis"), "R06": ("nb_captures", None, "cartes capturées"),
                 "R06s": ("nb_captures", None, "cartes capturées"), "R21": ("nb_captures", None, "cartes capturées"), "R22": ("nb_inaboutis", None, "retraits inaboutis"),
                 "R28": ("nb_captures", None, "cartes capturées"), "R07": ("nb_ret_exotique", None, "retraits cartes étrangères"), "R07f": ("nb_ret_exotique", None, "retraits cartes étrangères"),
                 "R20": ("nb_ret_exotique", None, "retraits cartes étrangères"), "R08": ("nb_0h_6h", None, "retraits entre 0 h et 6 h"),
                 "R18": ("part_10", "hab_part_10", "part des billets de 10 €"), "R18b": ("part_10", "hab_part_10", "part des billets de 10 €"),
                 "R18c": ("nb_retraits_repetes_10min", "hab_repetes", "retraits refaits < 10 min"), "R18d": ("nb_plans_cassettes", None, "plans de cassettes vus dans la journée"),
                 "R30": ("nb_billets_inferieurs_montant", None, "retraits servis en moins"), "R31": ("nb_billets_superieurs_montant", None, "retraits servis en trop"),
                 "R32": ("nb_inaboutis_avec_billets", None, "billets sortis sur des inaboutis"), "R33": ("nb_retraits_100plus_sans_billet_50", None, "retraits ≥ 100 € sans billet de 50 €"),
                 "R34": ("nb_incoherences_encaisse", None, "incohérences d'encaisse"), "R35": ("heure_dernier_retrait", "hab_dernier_retrait", "heure du dernier retrait"),
                 "R35z": ("heure_dernier_retrait", "hab_dernier_retrait", "heure du dernier retrait"), "R35v": ("heure_dernier_retrait", "hab_dernier_retrait", "heure du dernier retrait"),
                 "R11": ("montant_moyen", "hab_montant_moyen", "montant moyen par retrait (€)"), "R10M3": ("montant_total", "hab_montant_total", "montant retiré (€)"),
                 "R10M2": ("montant_total", "hab_montant_total", "montant retiré (€)"), "R10Mj": ("montant_total", "hab_montant_total", "montant retiré (€)"),
                 "R01": ("plus_long_trou_min", None, "plus long trou sans opération (min)"), "R29": ("part_porteur", None, "part des porteurs de la banque")}
_evx = EV[EV["gravite"] >= 1].merge(jour[["num_automate", "date_jour", "rang_jour", "niveau"]], on=["num_automate", "date_jour"], how="left")
_ordre = {c_: i for i, c_ in enumerate(catalogue["code technique"])} if "code technique" in catalogue.columns else {}
EXEMPLES = _evx.sort_values(["rang_jour", "gravite"], ascending=[True, False]).groupby("code").head(1)
EXEMPLES = EXEMPLES.assign(_o=EXEMPLES["code"].map(_ordre)).sort_values("_o").drop(columns="_o")
tuiles([("Anomalies illustrées", nb(len(EXEMPLES)), "sur {} au catalogue (celles déclenchées au moins une fois)".format(len(catalogue))),
        ("Meilleur rang moyen", nb(EXEMPLES["rang_jour"].median(), 0), "rang du jour médian des exemples choisis")])
for _, e_ in EXEMPLES.iterrows():
    col_, hab_, lib_ = INDIC_EXEMPLE.get(e_["code"], ("nb_retraits", "hab_retraits", "retraits"))
    if col_ not in jour.columns:
        col_, hab_, lib_ = "nb_retraits", "hab_retraits", "retraits"
    s_ = jour[(jour["num_automate"] == e_["num_automate"]) & (jour["date_jour"] >= e_["date_jour"] - pd.Timedelta(days=21)) & (jour["date_jour"] <= min(FIN, e_["date_jour"] + pd.Timedelta(days=7)))]
    k_ = 100 if col_.startswith("part_") else 1
    display(HTML(CSS + '<div class="g-ret" style="border-left-color:{}"><b class="t" style="color:{}">{} — {}</b><span class="pill {}">{}</span> GAB <b>{}</b> le {} (rang {} du jour)<br>{}</div>'.format(
        COULEURS_NIVEAU.get(NIVEAU_GRAVITE.get(e_["gravite"], ""), GRIS), COULEURS_NIVEAU.get(NIVEAU_GRAVITE.get(e_["gravite"], ""), GRIS), _html(e_["famille_anomalie"]), _html(e_["anomalie"]),
        CLASSE_NIVEAU.get(NIVEAU_GRAVITE.get(e_["gravite"], ""), "i"), NIVEAU_GRAVITE.get(e_["gravite"], ""), e_["num_automate"], jfr(e_["date_jour"]), int(e_["rang_jour"]), _html(e_["texte"]))))
    ser_ = [(lib_, (s_[col_] * k_).round(1), BLEU)] + ([("habitude", (s_[hab_] * k_).round(1), GRIS, "dash")] if hab_ and hab_ in s_.columns else [])
    x0_ = s_[s_["date_jour"] == e_["date_jour"]]
    g_courbes(s_["date_jour"], ser_, "GAB {} : {}".format(e_["num_automate"], lib_), hauteur=220,
              marqueurs=[("jour détecté", x0_["date_jour"], (x0_[col_] * k_).round(1), COULEURS_NIVEAU["Critique"], [e_["anomalie"]])] if len(x0_) else None)

<a href="#sommaire" style="font-size:12px">↑ retour au sommaire</a>
<a id="s13"></a>
## 13. La méthode retrouve-t-elle les incidents déjà connus ?
**En bref** : on vérifie que les incidents confirmés par le métier ressortent bien, alors qu'ils n'ont jamais servi à régler la méthode.

**Ce que vous voyez** : pour chaque incident, les jours autour, le niveau et le **rang du jour** du GAB. Un incident est **retrouvé** quand le
GAB est Critique ou Élevé et dans la liste des GAB les plus atypiques du jour. Les graphiques montrent ses indicateurs : la rupture doit se
voir à l'œil nu là où la détection l'a signalée (points rouges).

In [ ]:
def masque_gab(code):
    m = jour["num_automate"] == str(code)
    if "code_site_implantation" in jour.columns:
        m = m | jour["code_site_implantation"].astype(str).str.startswith(str(code))
    return m


def tracer_gab(gc, d0=None, jours_avant=56, jours_apres=14, titre=""):
    """Les indicateurs d'un GAB sur une fenêtre, avec son habitude et ses jours en alerte."""
    fin_ = min(FIN, d0 + pd.Timedelta(days=jours_apres)) if d0 is not None else FIN
    deb_ = (d0 - pd.Timedelta(days=jours_avant)) if d0 is not None else FIN - pd.Timedelta(days=84)
    s_ = jour[(jour["num_automate"] == gc) & (jour["date_jour"] >= deb_) & (jour["date_jour"] <= fin_)]
    if s_.empty:
        note("GAB {} : pas de données sur la fenêtre.".format(gc), "warn")
        return
    al_ = s_[s_["niveau"].isin(["Critique", "Élevé"])]
    lv_ = [(d0, "incident")] if d0 is not None else None
    marq = lambda col: [("jour en alerte", al_["date_jour"], al_[col], COULEURS_NIVEAU["Critique"], [court(t_, 1) for t_ in al_["diagnostic"]])] if len(al_) else None
    g_courbes(s_["date_jour"], [("retraits", s_["nb_retraits"], BLEU), ("habitude", s_["hab_retraits"], GRIS, "dash")],
              "GAB {}{} : retraits par jour, habitude, et jours en alerte (rouge)".format(gc, " — " + titre if titre else ""), y_titre="retraits", hauteur=300, marqueurs=marq("nb_retraits"), lignes_v=lv_)
    series_ = [("part des billets de 10 €", (s_["part_10"] * 100).round(0), SERIE[0]), ("part des billets de 20 €", (s_["part_20"] * 100).round(0), SERIE[1]), ("part des billets de 50 €", (s_["part_50"] * 100).round(0), SERIE[2])]
    g_courbes(s_["date_jour"], series_, "Le même GAB : répartition des billets (%)", y_titre="%", hauteur=260, lignes_v=lv_, hover="%{x|%a %d/%m/%Y}<br>%{y:.0f} %<extra>%{fullData.name}</extra>")
    series_ = [("retraits refaits par la même carte < 10 min", s_["nb_retraits_repetes_10min"], SERIE[3])]
    for c, lib, k in [("nb_captures", "cartes capturées", 4), ("nb_inaboutis", "retraits inaboutis", 5), ("nb_ret_exotique", "cartes étrangères", 6), ("nb_0h_6h", "retraits de nuit", 7)]:
        if c in s_.columns:
            series_.append((lib, s_[c], SERIE[k % len(SERIE)]))
    g_courbes(s_["date_jour"], series_, "Le même GAB : incidents et événements rares (nombre par jour)", y_titre="nombre", hauteur=280, lignes_v=lv_)


bilan_cas = []
if not CAS_TESTS:
    note("Section non exécutée : CAS_TESTS vide.", "warn")
for cas in CAS_TESTS:
    d0 = pd.Timestamp(cas["date"])
    avant, apres = int(cas.get("avant", 3)), int(cas.get("apres", 2))
    m = masque_gab(cas["gab"])
    cas_df = jour[m & (jour["date_jour"] >= d0 - pd.Timedelta(days=avant)) & (jour["date_jour"] <= d0 + pd.Timedelta(days=apres))].copy()
    bandeau("Cas {} : {}".format(cas["gab"], cas.get("quoi", "")), "incident du {} ; fenêtre de {} jours avant à {} jours après".format(d0.strftime("%d/%m/%Y"), avant, apres))
    if cas_df.empty:
        note("Le GAB n'est pas dans les données sur cette fenêtre (hors période, ou numéro inconnu).", "bad")
        bilan_cas.append({"cas": cas["gab"], "quoi": cas.get("quoi", ""), "date": d0.strftime("%d/%m/%Y"), "résultat": "absent des données"})
        continue
    t = cas_df[["date_jour", "niveau", "rang_jour", "nb_retraits", "hab_retraits", "anomalies", "diagnostic"]].copy()
    t["hab_retraits"] = t["hab_retraits"].round(0)
    t["diagnostic"] = t["diagnostic"].where(t["diagnostic"] != "", np.where(t["hab_retraits"].isnull(), "historique insuffisant", "rien d'anormal"))
    display(tableau(t, libelles=LIBELLES))
    fen = cas_df[cas_df["date_jour"] >= d0 - pd.Timedelta(days=1)]
    ok_jours = fen[fen["niveau"].isin(["Critique", "Élevé"]) & (fen["rang_jour"] <= TOP_LISTE)]
    if len(ok_jours):
        r = ok_jours.sort_values(["score_regles", "gravite_max", "score"], ascending=False).iloc[0]
        p0 = ok_jours["date_jour"].min()
        premier = (p0 - d0).days
        note("<b>Retrouvé.</b> Meilleure journée le {} : niveau <b>{}</b>, rang <b>{}</b> sur {} GAB. Première alerte le {} ({} jour(s) {} l'incident), "
             "{} jour(s) en alerte sur la fenêtre.<br>Diagnostic : {}".format(r["date_jour"].strftime("%d/%m/%Y"), r["niveau"], int(r["rang_jour"]), nb((jour["date_jour"] == r["date_jour"]).sum()),
                                                                             p0.strftime("%d/%m/%Y"), abs(premier), "après" if premier >= 0 else "avant", len(ok_jours), r["diagnostic"]), "ok")
        bilan_cas.append({"cas": cas["gab"], "quoi": cas.get("quoi", ""), "date": d0.strftime("%d/%m/%Y"), "résultat": "retrouvé", "première alerte": p0.strftime("%d/%m/%Y"),
                          "niveau": r["niveau"], "rang du jour": int(r["rang_jour"]), "jours en alerte": len(ok_jours)})
    else:
        meilleur = fen.sort_values(["score_regles", "score"], ascending=[False, False]).iloc[0]
        note("<b>Non retrouvé.</b> Au mieux niveau {}, rang {} le {}. Diagnostic : {}. À faire : regarder les anomalies de volume et de silence pour ce GAB."
             .format(meilleur["niveau"], int(meilleur["rang_jour"]), meilleur["date_jour"].strftime("%d/%m/%Y"), meilleur["diagnostic"] or "aucune anomalie"), "bad")
        bilan_cas.append({"cas": cas["gab"], "quoi": cas.get("quoi", ""), "date": d0.strftime("%d/%m/%Y"), "résultat": "NON retrouvé", "niveau": meilleur["niveau"], "rang du jour": int(meilleur["rang_jour"])})
    tracer_gab(cas_df["num_automate"].iloc[0], d0, jours_avant=56 + avant, jours_apres=apres + 7, titre=cas.get("quoi", ""))
bilan_cas = pd.DataFrame(bilan_cas)
if len(bilan_cas):
    display(tableau(bilan_cas.fillna(""), "Bilan des cas connus", pills=("niveau",)))
    retenir(["<b>{}</b> cas retrouvé(s) sur {} : {}.".format(int((bilan_cas["résultat"] == "retrouvé").sum()), len(bilan_cas),
                                                            ", ".join("{} ({}, rang {})".format(r_["cas"], r_["niveau"], r_["rang du jour"]) for _, r_ in bilan_cas[bilan_cas["résultat"] == "retrouvé"].iterrows()) or "aucun")]
            + (["Non retrouvé(s) : {}.".format(", ".join(bilan_cas.loc[bilan_cas["résultat"] != "retrouvé", "cas"].astype(str)))] if (bilan_cas["résultat"] != "retrouvé").any() else []))
elif len(top):
    tracer_gab(top["num_automate"].iloc[0], titre="le GAB le plus atypique du dernier jour")

# Partie C — Vérification sur les cas connus (usage interne)
**Pour l'équipe data** : ces sections vérifient que la détection statistique retrouve des incidents réels connus (inversion de cassettes,
GAB dégradés, rafale de cartes étrangères), et comptent les cas du même type sur les 3 ans.

<a href="#sommaire" style="font-size:12px">↑ retour au sommaire</a>
<a id="s14"></a>
## 14. Cas 1 — L'inversion de cassettes
**Ce qui se passe** : lors d'un chargement, la cassette de billets de 10 € est mise à la place de celle de 20 €. Le GAB croit donner des
billets de 20 € et sort des billets de 10 €.

**Comment on le voit** : la part des billets de 10 € bondit, les clients mal servis refont aussitôt leur retrait, et les billets sortis ne
correspondent plus au montant demandé.

In [ ]:
def bandeau_cas(titre, sous_titre):
    bandeau(titre, sous_titre)


def fenetre_gab(gab, d0, avant, apres):
    m_ = masque_gab(gab)
    return jour[m_ & (jour["date_jour"] >= d0 - pd.Timedelta(days=avant)) & (jour["date_jour"] <= d0 + pd.Timedelta(days=apres))].copy()


d_inv = pd.Timestamp(CAS_INVERSION["date"])
bandeau_cas("Cas 1 — Inversion de cassettes : GAB {}".format(CAS_INVERSION["gab"]), "incident du {}".format(jfr(d_inv)))
s_inv = fenetre_gab(CAS_INVERSION["gab"], d_inv, 45, 10)
if s_inv.empty:
    note("Le GAB {} n'est pas dans la fiche autour du {}.".format(CAS_INVERSION["gab"], d_inv.strftime("%d/%m/%Y")), "bad")
else:
    gab_inv = s_inv["num_automate"].iloc[0]
    r0 = s_inv[s_inv["date_jour"] == d_inv]
    if len(r0):
        r0 = r0.iloc[0]
        _it = [("Niveau ce jour-là", r0["niveau"], "rang {} sur {} GAB".format(int(r0["rang_jour"]), nb((jour["date_jour"] == d_inv).sum()))),
               ("Billets de 10 €", pct(r0["part_10"], 0), "des billets sortis, au lieu de {} d'habitude".format(pct(r0["hab_part_10"], 0))),
               ("Retraits refaits < 10 min", nb(r0["nb_retraits_repetes_10min"]), "au lieu de {} d'habitude".format(nb(r0["hab_repetes"], 1)))]
        for c, l in [("nb_billets_inferieurs_montant", "Retraits servis en moins"), ("nb_billets_superieurs_montant", "Retraits servis en trop")]:
            if c in s_inv.columns:
                _it.append((l, nb(r0[c]), "billets ≠ montant demandé"))
        tuiles(_it)
        note("<b>Diagnostic de la détection</b> : " + (r0["diagnostic"] or "aucune anomalie ce jour-là"), "ok" if r0["niveau"] in ("Critique", "Élevé") else "warn")
    _s = s_inv.copy()
    g_courbes(_s["date_jour"], [("billets de 10 € (%)", (_s["part_10"] * 100).round(0), SERIE[0]), ("habitude 10 € (%)", (_s["hab_part_10"] * 100).round(0), SERIE[0], "dash"),
                                ("billets de 20 € (%)", (_s["part_20"] * 100).round(0), SERIE[1]), ("habitude 20 € (%)", (_s["hab_part_20"] * 100).round(0), SERIE[1], "dash")],
              "GAB {} : la répartition des billets bascule le jour de l'inversion".format(gab_inv), y_titre="% des billets", hauteur=300, lignes_v=[(d_inv, "inversion")],
              hover="%{x|%d/%m/%Y}<br>%{y:.0f} %<extra>%{fullData.name}</extra>")
    _ser = [("retraits refaits par la même carte < 10 min", _s["nb_retraits_repetes_10min"], SERIE[3])]
    for c, l, k in [("nb_billets_inferieurs_montant", "retraits servis en moins", 4), ("nb_billets_superieurs_montant", "retraits servis en trop", 5), ("nb_incoherences_encaisse", "incohérences d'encaisse", 6)]:
        if c in _s.columns:
            _ser.append((l, _s[c], SERIE[k]))
    g_courbes(_s["date_jour"], _ser, "GAB {} : les clients refont leur retrait, les billets ne font plus le montant".format(gab_inv), y_titre="nombre par jour", hauteur=300, lignes_v=[(d_inv, "inversion")])
    _j = _s[(_s["date_jour"] >= d_inv - pd.Timedelta(days=3)) & (_s["date_jour"] <= d_inv + pd.Timedelta(days=3))][["date_jour", "niveau", "rang_jour", "nb_retraits", "anomalies", "diagnostic"]]
    display(tableau(_j, "Les jours autour de l'incident", libelles=LIBELLES))
# les autres inversions possibles sur toute la période
inv_toutes = EV[EV["code"] == "R18"].merge(jour[["num_automate", "date_jour", "rang_jour"] + [c for c in ["ville", "code_postal"] if c in jour.columns]], on=["num_automate", "date_jour"], how="left")
tuiles([("Inversions possibles sur la période", nb(len(inv_toutes)), "GAB-jours ; {} GAB distincts".format(nb(inv_toutes["num_automate"].nunique()))),
        ("Dans la liste du jour", nb(int((inv_toutes["rang_jour"] <= TOP_LISTE).sum())), "rang ≤ {} le jour même".format(TOP_LISTE))])
if len(inv_toutes):
    _m = inv_toutes.assign(mois=inv_toutes["date_jour"].dt.to_period("M").astype(str)).groupby("mois").size()
    g_barres(_m.index, [("inversions possibles", _m.values, BLEU)], "Inversions de cassettes possibles, mois par mois (GAB-jours)", hauteur=260)
    display(tableau(inv_toutes.sort_values("date_jour", ascending=False)[["date_jour", "num_automate"] + [c for c in ["ville"] if c in inv_toutes.columns] + ["rang_jour", "texte"]].rename(columns={"texte": "diagnostic"}),
                    "Toutes les inversions possibles détectées (les plus récentes d'abord)", n=25, libelles=LIBELLES))
retenir(["Le jour de l'inversion, le GAB {} est <b>{}</b> et classé <b>{}</b> du parc.".format(CAS_INVERSION["gab"], r0["niveau"], "premier" if int(r0["rang_jour"]) == 1 else "{}e".format(int(r0["rang_jour"])))
         if not s_inv.empty and isinstance(r0, pd.Series) else "Le GAB du cas n'est pas dans la fiche : vérifier `CAS_INVERSION`.",
         "La signature est double : la part des billets de 10 € bondit <b>et</b>, le même jour, les clients refont leur retrait ou les billets ne font plus le montant. Une seule des deux ne suffit pas (une cassette vide fait bouger les parts sans retraits refaits).",
         "Sur la période, la méthode signale <b>{}</b> autres journées du même type sur <b>{}</b> GAB : autant de vérifications possibles pour le métier.".format(nb(max(0, len(inv_toutes) - 1)), nb(inv_toutes["num_automate"].nunique()))])

<a href="#sommaire" style="font-size:12px">↑ retour au sommaire</a>
<a id="s15"></a>
## 15. Cas 2 — Les GAB dégradés pendant les manifestations
**Ce qui se passe** : un GAB vandalisé cesse de servir d'un coup, souvent en pleine journée, puis reste à l'arrêt ou ne sert plus qu'en
partie. Ses voisins peuvent être touchés en même temps.

**Comment on le voit** : effondrement des retraits face à l'habitude, dernier retrait des heures plus tôt que d'ordinaire, voisins touchés.
L'argent restant dans le GAB permet d'écarter un GAB simplement vide.

**Ce que vous voyez** : la zone suivie jour par jour (combien de GAB à l'arrêt ou en forte baisse, face à une période normale), la liste des
GAB touchés avec leur premier jour, puis le détail des GAB connus du suivi vandalisme.

In [ ]:
d_man = pd.Timestamp(CAS_MANIFESTATION["date"])
av_, ap_ = int(CAS_MANIFESTATION.get("jours_avant", 7)), int(CAS_MANIFESTATION.get("jours_apres", 21))
zone = pd.Series(True, index=jour.index)
if CAS_MANIFESTATION.get("codes_postaux") and "code_postal" in jour.columns:
    zone = jour["code_postal"].astype(str).str.zfill(5).str.startswith(tuple(str(c) for c in CAS_MANIFESTATION["codes_postaux"]))
libzone = "codes postaux " + ", ".join(CAS_MANIFESTATION["codes_postaux"]) if CAS_MANIFESTATION.get("codes_postaux") else "toute la France"
bandeau_cas("Cas 2 — GAB dégradés à partir du {}".format(jfr(d_man)), "zone suivie : {} ; du {} au {}".format(libzone, (d_man - pd.Timedelta(days=av_)).strftime("%d/%m"), (d_man + pd.Timedelta(days=ap_)).strftime("%d/%m/%Y")))
# les anomalies d'un arrêt brutal (pas les dérives lentes, qui commencent avant l'événement)
CODES_DEGRADATION = ["R02a", "R02", "R02b", "R35", "R35z", "R19", "R10b", "R10bj", "R01"]
_ev_z = EV[EV["code"].isin(CODES_DEGRADATION)].merge(jour.loc[zone, ["num_automate", "date_jour"]], on=["num_automate", "date_jour"])
_par_jour = _ev_z.groupby("date_jour")["num_automate"].nunique()
_jours = pd.date_range(d_man - pd.Timedelta(days=av_ + 56), min(FIN, d_man + pd.Timedelta(days=ap_)), freq="D")
_par_jour = _par_jour.reindex(_jours).fillna(0)
_base = _par_jour[_par_jour.index < d_man - pd.Timedelta(days=av_)]
_ref = _base.groupby(_base.index.dayofweek).mean()
_ser_ref = pd.Series([_ref.get(d_.dayofweek, np.nan) for d_ in _par_jour.index], index=_par_jour.index)
_vis = _par_jour.index >= d_man - pd.Timedelta(days=av_)
g_barres(_par_jour.index[_vis].strftime("%d/%m"), [("GAB à l'arrêt ou en forte baisse", _par_jour[_vis].values, COULEURS_NIVEAU["Critique"]), ("habituel (même jour de semaine, 8 semaines avant)", _ser_ref[_vis].round(1).values, GRIS_CLAIR)],
         "Zone suivie : GAB à l'arrêt ou en forte baisse, jour par jour (rouge), face à une période normale (gris)", y_titre="GAB distincts", hauteur=320)
_win = (jour["date_jour"] >= d_man) & (jour["date_jour"] <= d_man + pd.Timedelta(days=ap_)) & zone
_tou = _ev_z[(_ev_z["date_jour"] >= d_man) & (_ev_z["date_jour"] <= d_man + pd.Timedelta(days=ap_))]
gab_touches = _tou.groupby("num_automate").agg(premier_jour=("date_jour", "min"), jours_en_alerte=("date_jour", "nunique"), anomalie=("anomalie", "first"), diagnostic=("texte", "first")).reset_index()
_perte = jour[_win].assign(_p=(jour.loc[_win, "hab_retraits"] - jour.loc[_win, "nb_retraits"]).clip(lower=0)).groupby("num_automate")["_p"].sum().round(0)
gab_touches["retraits perdus (estimation)"] = gab_touches["num_automate"].map(_perte)
if "ville" in jour.columns:
    gab_touches["ville"] = gab_touches["num_automate"].map(jour.drop_duplicates("num_automate").set_index("num_automate")["ville"])
gab_touches["connu du suivi"] = np.where(gab_touches["num_automate"].isin([str(x) for x in CAS_MANIFESTATION.get("gab", [])]), "oui", "")
gab_touches = gab_touches.sort_values(["premier_jour", "retraits perdus (estimation)"], ascending=[True, False])
_avant = _base.mean() if len(_base) else np.nan
tuiles([("GAB touchés dans la zone", nb(len(gab_touches)), "au moins un jour à l'arrêt ou en forte baisse sur la période suivie"),
        ("Pic", nb(_par_jour[_vis].max()), "GAB le {} ; habituellement {} par jour".format(_par_jour[_vis].idxmax().strftime("%d/%m") if _vis.any() else "-", nb(_avant, 1))),
        ("Arrêts en cours de journée", nb(int(_tou["code"].isin(["R35", "R35z"]).sum())), "GAB-jours où le dernier retrait arrive des heures trop tôt"),
        ("Retraits perdus (estimation)", nb(gab_touches["retraits perdus (estimation)"].sum()), "habitude moins retraits réels, sur les GAB touchés")])
display(tableau(gab_touches.rename(columns={"num_automate": "GAB", "premier_jour": "premier jour", "jours_en_alerte": "jours en alerte"}), "Les GAB touchés, dans l'ordre d'apparition", n=40))
if "latitude" in jour.columns and len(gab_touches):
    _p = jour.loc[zone & (jour["date_jour"] == d_man)].copy()
    _p["lat"], _p["lon"] = pd.to_numeric(_p["latitude"], errors="coerce"), pd.to_numeric(_p["longitude"], errors="coerce")
    _p = _p[_p["lat"].between(41, 51.5) & _p["lon"].between(-5.5, 10)]
    _t = _p[_p["num_automate"].isin(gab_touches["num_automate"])].merge(gab_touches[["num_automate", "premier_jour", "anomalie"]], on="num_automate")
    g_points(_p["lon"], _p["lat"], "Les GAB touchés (rouge) dans la zone suivie (gris)", textes=["GAB " + str(x) for x in _p["num_automate"]], couleurs=GRIS_CLAIR, tailles=4, hauteur=520, aspect=1.4, axes=False,
             groupes=[("GAB touchés", _t["lon"], _t["lat"], ["GAB {}<br>premier jour : {}<br>{}".format(a_, b_.strftime("%d/%m"), c_) for a_, b_, c_ in zip(_t["num_automate"], _t["premier_jour"], _t["anomalie"])], COULEURS_NIVEAU["Critique"], 11)])
for g_ in CAS_MANIFESTATION.get("gab", []):
    s_ = fenetre_gab(g_, d_man, av_, ap_)
    if s_.empty:
        note("GAB {} (suivi vandalisme) : absent de la fiche sur la période.".format(g_), "bad")
        continue
    al_ = s_[s_["niveau"].isin(["Critique", "Élevé"])]
    note("<b>GAB {}</b> (suivi vandalisme) : {}".format(g_, "première alerte le {} ({}), {} jour(s) en alerte".format(
        jfr(al_["date_jour"].min()), al_.sort_values("date_jour").iloc[0]["anomalies"], len(al_)) if len(al_) else "aucune alerte Critique ou Élevé sur la période"), "ok" if len(al_) else "warn")
    g_courbes(s_["date_jour"], [("retraits", s_["nb_retraits"], BLEU), ("habitude", s_["hab_retraits"], GRIS, "dash")], "GAB {} : retraits face à l'habitude".format(g_), y_titre="retraits", hauteur=260,
              lignes_v=[(d_man, "manifestations")], marqueurs=[("jour en alerte", al_["date_jour"], al_["nb_retraits"], COULEURS_NIVEAU["Critique"], [court(t_, 1) for t_ in al_["diagnostic"]])] if len(al_) else None)
retenir(["Dans la zone suivie, <b>{}</b> GAB ont été à l'arrêt ou en forte baisse entre le {} et le {}, avec un pic de <b>{}</b> GAB le même jour (habituellement {} par jour).".format(
             nb(len(gab_touches)), d_man.strftime("%d/%m"), (d_man + pd.Timedelta(days=ap_)).strftime("%d/%m"), nb(_par_jour[_vis].max()), nb(_avant, 1)),
         "La détection date l'arrêt (jour et heure du dernier retrait) et distingue un GAB dégradé d'un GAB simplement vide grâce à l'encaisse restante.",
         "Les GAB du suivi vandalisme sont à saisir dans `CAS_MANIFESTATION[\"gab\"]` : la section vérifie alors, GAB par GAB, le jour de la première alerte." if not CAS_MANIFESTATION.get("gab") else
         "<b>{}</b> GAB du suivi vandalisme sur {} sont dans la liste des GAB touchés.".format(int((gab_touches["connu du suivi"] == "oui").sum()), len(CAS_MANIFESTATION["gab"]))])

<a href="#sommaire" style="font-size:12px">↑ retour au sommaire</a>
<a id="s16"></a>
## 16. Cas 3 — La rafale de cartes étrangères
**Ce qui se passe** : en une journée, des GAB voient arriver des retraits d'un réseau étranger qu'ils ne voient presque jamais (UnionPay,
Diners ou JCB via Trionis ; Amex). Quand plusieurs GAB du même département sont touchés le même jour, c'est une attaque coordonnée
possible.

**Ce que vous voyez** : tous les GAB qui ont vu ce réseau ce jour-là, face à leur habitude, avec la probabilité qu'un tel nombre arrive par
hasard ; la courbe du parc autour de la date ; la carte des GAB touchés.

In [ ]:
LIB_EXO = {"cup": "UnionPay", "amex": "Amex", "diners": "Diners", "jcb": "JCB", "trionis": "Trionis (UnionPay + Diners)", "exotique": "tous réseaux étrangers"}
res_x = CAS_EXOTIQUE["reseau"]
col_x = "nb_ret_" + res_x
d_x = pd.Timestamp(CAS_EXOTIQUE["date"])
bandeau_cas("Cas 3 — Rafale de cartes {} le {}".format(LIB_EXO.get(res_x, res_x), jfr(d_x)), "tous les GAB qui ont vu ce réseau ce jour-là, face à leur habitude")
if col_x not in jour.columns:
    note("La fiche n'a pas de colonne {} : vérifier `CAS_EXOTIQUE[\"reseau\"]`.".format(col_x), "bad")
else:
    jour["hab_" + col_x] = glisse(jour[col_x], ["num_automate"], 28, 10, "mean")
    _jx = jour[(jour["date_jour"] == d_x) & (jour[col_x] > 0)].copy()
    if _jx.empty:
        note("Aucun retrait {} le {} dans la fiche.".format(LIB_EXO.get(res_x, res_x), d_x.strftime("%d/%m/%Y")), "warn")
    else:
        _jx["chances"] = chances(pd.Series(poisson.sf(_jx[col_x] - 1, _jx["hab_" + col_x].fillna(0).clip(lower=0.05)), index=_jx.index))
        _jx["département"] = _jx["code_postal"].astype(str).str.zfill(5).str[:2] if "code_postal" in _jx.columns else "?"
        _jx = _jx.sort_values(col_x, ascending=False)
        _parc_x = jour.groupby("date_jour")[col_x].sum()
        _rang_jour = int((_parc_x > _parc_x.get(d_x, 0)).sum()) + 1
        _dep = _jx["département"].value_counts()
        tuiles([("Retraits {} ce jour-là".format(LIB_EXO.get(res_x, res_x)), nb(_jx[col_x].sum()), "pour le parc ; {}e journée la plus forte de la fiche".format(_rang_jour) if _rang_jour > 1 else "pour le parc : record de la fiche"),
                ("GAB concernés", nb(len(_jx)), "dont {} au-dessus de 5 retraits".format(nb(int((_jx[col_x] >= 5).sum())))),
                ("Département le plus touché", _dep.index[0], "{} GAB".format(_dep.iloc[0])),
                ("Un jour ordinaire", nb(_parc_x[_parc_x.index < d_x].tail(56).median(), 1), "retraits {} pour le parc (médiane des 8 semaines d'avant)".format(LIB_EXO.get(res_x, res_x)))])
        _cols = ["num_automate"] + [c for c in ["ville", "département"] if c in _jx.columns] + [col_x] + (["montant_" + res_x] if "montant_" + res_x in _jx.columns else []) + ["hab_" + col_x, "chances", "niveau", "rang_jour", "resume"]
        display(tableau(_jx[_cols].rename(columns={"num_automate": "GAB", col_x: "retraits " + LIB_EXO.get(res_x, res_x), "montant_" + res_x: "montant (€)", "hab_" + col_x: "habitude par jour (28 j)",
                                                   "chances": "probabilité", "rang_jour": "rang du jour", "resume": "résumé"}), "Les GAB qui ont vu des cartes {} le {}".format(LIB_EXO.get(res_x, res_x), d_x.strftime("%d/%m/%Y")), n=30,
                        longues=("résumé",)))
        _fx = _parc_x[(_parc_x.index >= d_x - pd.Timedelta(days=60)) & (_parc_x.index <= d_x + pd.Timedelta(days=30))]
        g_courbes(_fx.index, [("retraits {} du parc".format(LIB_EXO.get(res_x, res_x)), _fx.values, "#e36209")], "Retraits {} du parc, jour par jour, autour du {}".format(LIB_EXO.get(res_x, res_x), d_x.strftime("%d/%m/%Y")),
                  y_titre="retraits par jour", hauteur=280, lignes_v=[(d_x, "rafale")])
        if "latitude" in _jx.columns:
            _jx["lat"], _jx["lon"] = pd.to_numeric(_jx["latitude"], errors="coerce"), pd.to_numeric(_jx["longitude"], errors="coerce")
            _fond = jour.loc[jour["date_jour"] == d_x, ["latitude", "longitude"]].apply(pd.to_numeric, errors="coerce")
            _fond = _fond[_fond["latitude"].between(41, 51.5) & _fond["longitude"].between(-5.5, 10)]
            g_points(_fond["longitude"], _fond["latitude"], "Où sont les GAB qui ont vu des cartes {} le {} (taille = nombre de retraits)".format(LIB_EXO.get(res_x, res_x), d_x.strftime("%d/%m/%Y")),
                     couleurs=GRIS_CLAIR, tailles=3, hauteur=520, aspect=1.4, axes=False,
                     groupes=[("GAB concernés", _jx["lon"], _jx["lat"], ["GAB {}<br>{} retraits {}<br>habitude {}".format(a_, int(b_), LIB_EXO.get(res_x, res_x), nb(c_, 1)) for a_, b_, c_ in zip(_jx["num_automate"], _jx[col_x], _jx["hab_" + col_x])],
                               "#e36209", list(np.clip(6 + _jx[col_x].values, 6, 26)))])
        retenir(["Le {}, <b>{}</b> retraits {} sur <b>{}</b> GAB, contre {} un jour ordinaire pour tout le parc.".format(
                     d_x.strftime("%d/%m/%Y"), nb(_jx[col_x].sum()), LIB_EXO.get(res_x, res_x), nb(len(_jx)), nb(_parc_x[_parc_x.index < d_x].tail(56).median(), 1)),
                 "Le département <b>{}</b> concentre {} des GAB touchés : {}.".format(_dep.index[0], _dep.iloc[0], "plusieurs GAB du même département le même jour, signature d'une attaque coordonnée" if _dep.iloc[0] >= 3 else "pas de concentration géographique nette"),
                 "Pour chaque GAB, la probabilité dit si ce nombre était plausible vu son habitude : un GAB de gare en voit tous les jours, un GAB de quartier jamais."])

<a href="#sommaire" style="font-size:12px">↑ retour au sommaire</a>
<a id="s17"></a>
## 17. Les records des réseaux étrangers
**Pour la présentation** : pour chaque réseau étranger, les journées où un GAB a vu le plus de retraits de ce réseau, et les journées les
plus fortes pour tout le parc. À côté de chaque record, l'habitude du GAB : on distingue ainsi un GAB qui en voit toujours beaucoup (gare,
aéroport) d'une vraie rafale.

La fiche donne, par réseau, le nombre et le montant des retraits de la journée. Le plus gros retrait individuel n'existe que tous réseaux
confondus.

In [ ]:
RESEAUX_RECORDS = [r for r in ["cup", "diners", "jcb", "amex", "trionis", "exotique"] if "nb_ret_" + r in jour.columns]
records, synthese_records = [], []
for r in RESEAUX_RECORDS:
    c_ = "nb_ret_" + r
    if "hab_" + c_ not in jour.columns:
        jour["hab_" + c_] = glisse(jour[c_], ["num_automate"], 28, 10, "mean")
    _top = jour[jour[c_] > 0].nlargest(TOP_RECORDS, c_).copy()
    _top["réseau"] = LIB_EXO.get(r, r)
    _top["montant (€)"] = _top["montant_" + r] if "montant_" + r in _top.columns else np.nan
    _top["fois l'habitude"] = np.where(_top["hab_" + c_].fillna(0) < 0.1, "n'en voit jamais", "× " + (_top[c_] / _top["hab_" + c_].clip(lower=0.1)).round(0).astype(int).astype(str))
    _top = _top.rename(columns={c_: "retraits", "hab_" + c_: "habitude par jour (28 j)"})
    records.append(_top[["réseau", "date_jour", "num_automate"] + [c for c in ["ville", "code_postal"] if c in _top.columns] + ["retraits", "montant (€)", "habitude par jour (28 j)", "fois l'habitude", "niveau", "rang_jour"]])
    _pj = jour.groupby("date_jour")[c_].sum()
    _g1 = _top.iloc[0] if len(_top) else None
    synthese_records.append({"réseau": LIB_EXO.get(r, r), "retraits sur la période": jour[c_].sum(), "GAB qui en ont vu": int((jour.groupby("num_automate")[c_].sum() > 0).sum()),
                             "record d'un GAB en un jour": "{} retraits — GAB {} le {}".format(int(_g1["retraits"]), _g1["num_automate"], _g1["date_jour"].strftime("%d/%m/%Y")) if _g1 is not None else "-",
                             "record du parc en un jour": "{} retraits le {}".format(int(_pj.max()), _pj.idxmax().strftime("%d/%m/%Y")) if _pj.max() > 0 else "-",
                             "un jour ordinaire (parc, médiane)": _pj.median()})
records = pd.concat(records, ignore_index=True) if records else pd.DataFrame()
synthese_records = pd.DataFrame(synthese_records)
bandeau("Les records des réseaux étrangers", "sur toute la fiche, du {} au {}".format(DEBUT_PERIODE.strftime("%d/%m/%Y"), FIN.strftime("%d/%m/%Y")))
display(tableau(synthese_records, "Réseau par réseau : volume, GAB concernés et records", dec=1))
for r in RESEAUX_RECORDS:
    _t = records[records["réseau"] == LIB_EXO.get(r, r)]
    if len(_t):
        g_barres(["{} — {}".format(a_.strftime("%d/%m/%Y"), b_) for a_, b_ in zip(_t["date_jour"], _t["num_automate"])][::-1],
                 [("retraits", _t["retraits"].values[::-1], "#e36209"), ("habitude du GAB", _t["habitude par jour (28 j)"].round(1).values[::-1], GRIS_CLAIR)],
                 "{} : les {} journées record d'un GAB (orange), face à son habitude (gris)".format(LIB_EXO.get(r, r), len(_t)), hauteur=max(280, 30 * len(_t)), horizontal=True)
display(tableau(records.rename(columns={"date_jour": "jour", "num_automate": "GAB", "rang_jour": "rang du jour", "code_postal": "code postal"}), "Tous les records (l'Excel les reprend dans l'onglet « Records réseaux étrangers »)", n=60))

# Partie D — Ce que l'analyse apprend au métier

<a href="#sommaire" style="font-size:12px">↑ retour au sommaire</a>
<a id="s18"></a>
## 18. Des réponses concrètes pour le quotidien
Au-delà des anomalies, la même fiche répond à des questions de tous les jours :

- combien de billets prévoir, et quel jour ;
- quels GAB se vident trop vite ;
- quand intervenir sans gêner les clients ;
- quels GAB tombent souvent en panne ;
- quels GAB sont sous-utilisés ou saturés par rapport à des GAB comparables.

Chaque réponse porte sur les **12 derniers mois** et a son onglet dans l'Excel.

In [ ]:
_12 = jour[(jour["date_jour"] > FIN - pd.Timedelta(days=365))]
_nj12 = _12["date_jour"].nunique()

### Combien de billets prévoir, et quel jour ?
Le nombre de billets distribués par jour pour tout le parc, selon le jour de semaine et selon le mois. L'onglet « Besoin en billets » donne
la même chose GAB par GAB : c'est la base d'un plan de chargement.

In [ ]:
COUPURES = [(v, "nb_billets_{}".format(v)) for v in (10, 20, 50, 100) if "nb_billets_{}".format(v) in jour.columns and jour["nb_billets_{}".format(v)].sum() > 0]
if COUPURES:
    _b = _12.groupby("jour_semaine")[[c for _, c in COUPURES]].sum().div(_12.groupby("jour_semaine")["date_jour"].nunique(), axis=0)
    g_barres([JOURS_FR[int(d) - 1] for d in _b.index], [("billets de {} €".format(v), _b[c].round(0).values, SERIE[i]) for i, (v, c) in enumerate(COUPURES)],
             "Billets distribués par jour pour tout le parc, selon le jour de semaine (12 derniers mois)", y_titre="billets par jour", hauteur=320, empile=True)
    _m = _12.assign(mois=_12["date_jour"].dt.to_period("M").dt.to_timestamp()).groupby("mois")[[c for _, c in COUPURES]].sum()
    g_courbes(_m.index, [("billets de {} €".format(v), _m[c].values, SERIE[i]) for i, (v, c) in enumerate(COUPURES)], "Billets distribués par mois, par coupure", y_titre="billets", hauteur=280,
              hover="%{x|%m/%Y}<br>%{y:,.0f} billets<extra>%{fullData.name}</extra>")
    BESOIN_BILLETS = _12.groupby(["num_automate", "jour_semaine"])[[c for _, c in COUPURES]].mean().round(0).reset_index()
    BESOIN_BILLETS["jour de semaine"] = BESOIN_BILLETS["jour_semaine"].map(lambda d: JOURS_FR[int(d) - 1])
    BESOIN_BILLETS = BESOIN_BILLETS.drop(columns="jour_semaine").rename(columns=dict({"num_automate": "GAB"}, **{c: "billets de {} € (moyenne)".format(v) for v, c in COUPURES}))
    _jmax = _b.sum(axis=1).idxmax()
    retenir(["Le jour qui demande le plus de billets est le <b>{}</b> ({} billets pour le parc), le plus calme le <b>{}</b> ({}).".format(
                 JOURS_FR[int(_jmax) - 1], nb(_b.sum(axis=1).max()), JOURS_FR[int(_b.sum(axis=1).idxmin()) - 1], nb(_b.sum(axis=1).min())),
             "Les billets de 20 € font <b>{}</b> des billets distribués.".format(pct(_b[[c for v, c in COUPURES if v == 20]].sum().sum() / max(1, _b.sum().sum()), 0)) if any(v == 20 for v, _ in COUPURES) else ""])

### Quels GAB se vident trop souvent ?
Un GAB qui finit souvent la journée presque vide (moins de 1 000 €) ou qui s'arrête faute d'argent perd des clients. Il faut le charger
davantage ou plus souvent.

In [ ]:
GAB_VIDES = pd.DataFrame()
if "encaisse_fin" in jour.columns and jour["encaisse_fin"].notnull().any():
    _a = _12[_12["nb_retraits"] > 0]
    GAB_VIDES = pd.DataFrame({"jours actifs": _a.groupby("num_automate").size(), "jours finis presque vides": _a.assign(_v=(_a["encaisse_fin"] < 1000).astype(int)).groupby("num_automate")["_v"].sum(),
                              "encaisse de fin médiane (€)": _a.groupby("num_automate")["encaisse_fin"].median().round(0),
                              "chargements par semaine": (_12.groupby("num_automate")["nb_recharges"].sum() / max(1, _nj12) * 7).round(1) if "nb_recharges" in _12.columns else np.nan,
                              "retraits par jour": _a.groupby("num_automate")["nb_retraits"].mean().round(0)})
    GAB_VIDES["arrêts pour encaisse vide"] = EV[(EV["code"] == "R35v") & (EV["date_jour"] > FIN - pd.Timedelta(days=365))].groupby("num_automate").size().reindex(GAB_VIDES.index).fillna(0).astype(int)
    GAB_VIDES["part des jours presque vides"] = (GAB_VIDES["jours finis presque vides"] / GAB_VIDES["jours actifs"]).round(3)
    GAB_VIDES = GAB_VIDES.sort_values(["part des jours presque vides", "arrêts pour encaisse vide"], ascending=False).reset_index().rename(columns={"num_automate": "GAB"})
    _nv = int((GAB_VIDES["part des jours presque vides"] >= 0.05).sum())
    tuiles([("GAB souvent vides", nb(_nv), "au moins 1 jour sur 20 fini sous 1 000 €"), ("Arrêts pour encaisse vide", nb(GAB_VIDES["arrêts pour encaisse vide"].sum()), "GAB-jours sur 12 mois")])
    _t = GAB_VIDES.head(15).copy()
    _t["part des jours presque vides"] = _t["part des jours presque vides"].map(lambda v: pct(v, 1))
    display(tableau(_t, "Les 15 GAB qui finissent le plus souvent presque vides (12 derniers mois)"))

### Quand intervenir sans gêner les clients ?
Pour chaque GAB, la tranche horaire de la journée (entre 6 h et 22 h) où il sert le moins de clients par heure : c'est le meilleur moment
pour une maintenance ou un chargement.

In [ ]:
CRENEAUX = [("nb_6h_9h", "6 h – 9 h", 3), ("nb_9h_12h", "9 h – 12 h", 3), ("nb_12h_14h", "12 h – 14 h", 2), ("nb_14h_18h", "14 h – 18 h", 4), ("nb_18h_22h", "18 h – 22 h", 4)]
CRENEAUX = [x for x in CRENEAUX if x[0] in jour.columns]
MAINTENANCE = pd.DataFrame()
if CRENEAUX:
    _c = _12.groupby("num_automate")[[c for c, _, _ in CRENEAUX]].mean()
    _ph = pd.DataFrame({l: _c[c] / h for c, l, h in CRENEAUX})
    MAINTENANCE = pd.DataFrame({"créneau le plus calme": _ph.idxmin(axis=1), "retraits par heure à ce créneau": _ph.min(axis=1).round(1),
                                "créneau le plus chargé": _ph.idxmax(axis=1), "retraits par heure au pic": _ph.max(axis=1).round(1)})
    _r = MAINTENANCE["créneau le plus calme"].value_counts().reindex([l for _, l, _ in CRENEAUX]).fillna(0)
    g_barres(_r.index, [("GAB", _r.values, BLEU)], "Le créneau de journée le plus calme de chaque GAB (nombre de GAB)", hauteur=280)
    MAINTENANCE = MAINTENANCE.reset_index().rename(columns={"num_automate": "GAB"})
    retenir(["Pour <b>{}</b> GAB sur {}, le créneau le plus calme est <b>{}</b>.".format(nb(_r.max()), nb(_r.sum()), _r.idxmax())])

### Quels GAB tombent souvent en panne ?
Chaque arrêt (journée sans retrait ou arrêt en cours de journée) compte comme un épisode. Le temps moyen entre deux arrêts dit si un GAB
est fragile.

In [ ]:
_arr = EV[EV["code"].isin(["R02a", "R02", "R02b", "R35", "R35z"]) & (EV["date_jour"] > FIN - pd.Timedelta(days=365))][["num_automate", "date_jour"]].drop_duplicates().sort_values(["num_automate", "date_jour"])
_arr["nouvel_episode"] = (_arr.groupby("num_automate")["date_jour"].diff().dt.days.fillna(99) > 1).astype(int)
FIABILITE = _arr.groupby("num_automate").agg(**{"épisodes d'arrêt": ("nouvel_episode", "sum"), "jours signalés à l'arrêt": ("date_jour", "size"), "dernier arrêt": ("date_jour", "max")})
FIABILITE["jours entre deux arrêts (moyenne)"] = (365 / FIABILITE["épisodes d'arrêt"].clip(lower=1)).round(0)
FIABILITE = FIABILITE.sort_values(["épisodes d'arrêt", "jours signalés à l'arrêt"], ascending=False).reset_index().rename(columns={"num_automate": "GAB"})
tuiles([("GAB tombés au moins une fois à l'arrêt", nb(len(FIABILITE)), "sur {} GAB, 12 derniers mois".format(nb(jour["num_automate"].nunique()))),
        ("Épisodes d'arrêt", nb(FIABILITE["épisodes d'arrêt"].sum()), "sur 12 mois"),
        ("GAB à 3 arrêts ou plus", nb(int((FIABILITE["épisodes d'arrêt"] >= 3).sum())), "à faire vérifier en priorité")])
if len(FIABILITE):
    display(tableau(FIABILITE.head(15), "Les 15 GAB les moins fiables (12 derniers mois)"))

### Quels GAB sont sous-utilisés ou saturés ?
Chaque GAB est comparé aux GAB de sa famille sur les 3 derniers mois. **Moins de la moitié** de ses pareils : sous-utilisé (emplacement,
visibilité, concurrence). **Plus du double** : saturé (à charger plus, ou à doubler).

In [ ]:
USAGE = pd.DataFrame()
if "famille" in jour.columns:
    _3 = jour[(jour["date_jour"] > FIN - pd.Timedelta(days=91))]
    _u = _3.groupby("num_automate").agg(famille=("famille", "first"), retraits_par_jour=("nb_retraits", "mean"))
    _u["médiane de la famille"] = _u.groupby("famille")["retraits_par_jour"].transform("median")
    _u["rapport à la famille"] = (_u["retraits_par_jour"] / _u["médiane de la famille"].replace(0, np.nan)).round(2)
    _u["usage"] = np.select([_u["rapport à la famille"] < 0.5, _u["rapport à la famille"] > 2], ["sous-utilisé", "saturé"], "dans la norme")
    USAGE = _u.round(1).reset_index().rename(columns={"num_automate": "GAB", "retraits_par_jour": "retraits par jour (3 mois)"}).sort_values("rapport à la famille")
    _uc = USAGE["usage"].value_counts()
    tuiles([("Sous-utilisés", nb(_uc.get("sous-utilisé", 0)), "moins de la moitié de leur famille"), ("Saturés", nb(_uc.get("saturé", 0)), "plus du double de leur famille"),
            ("Dans la norme", nb(_uc.get("dans la norme", 0)), "GAB")])
    display(tableau(pd.concat([USAGE.head(8), USAGE.tail(8)]), "Les GAB les plus sous-utilisés (haut) et les plus saturés (bas) face à leur famille"))

<a href="#sommaire" style="font-size:12px">↑ retour au sommaire</a>
<a id="s19"></a>
## 19. Le tableau de bord des 3 ans
**La page à montrer en premier** : le parc, les alertes, les trois cas de la présentation et les records des réseaux étrangers, en un coup
d'œil.

In [ ]:
bandeau("Tableau de bord — du {} au {}".format(DEBUT_PERIODE.strftime("%d/%m/%Y"), FIN.strftime("%d/%m/%Y")), "{} GAB ; {} jours ; toute la fiche".format(nb(P["num_automate"].nunique()), nb(P["date_jour"].nunique())))
_ce = P["niveau"].isin(["Critique", "Élevé"])
tuiles([("Retraits", nb(P["nb_retraits"].sum()), "{} par jour pour le parc".format(nb(P.groupby("date_jour")["nb_retraits"].sum().mean()))),
        ("Montant retiré", nb(P["montant_total"].sum() / 1e6, 1) + " M€", "sur la période"),
        ("Alertes Critique ou Élevé", nb(_ce.sum()), "GAB-jours, soit {} par jour".format(nb(_ce.sum() / max(1, P["date_jour"].nunique()), 1))),
        ("GAB distincts alertés", nb(P.loc[_ce, "num_automate"].nunique()), "{} des GAB au moins une fois".format(pct(P.loc[_ce, "num_automate"].nunique() / max(1, P["num_automate"].nunique()), 0))),
        ("GAB jamais en alerte", nb(P["num_automate"].nunique() - P.loc[_ce, "num_automate"].nunique()), "sur la période"),
        ("Épisodes d'arrêt (12 mois)", nb(FIABILITE["épisodes d'arrêt"].sum()) if len(FIABILITE) else "-", "{} GAB concernés".format(nb(len(FIABILITE))))])
_cas = []
if "s_inv" in globals() and len(s_inv) and isinstance(r0, pd.Series):
    _cas.append({"cas": "Inversion de cassettes", "GAB / date": "{} le {}".format(CAS_INVERSION["gab"], d_inv.strftime("%d/%m/%Y")), "détecté": "oui, {} au rang {}".format(r0["niveau"], int(r0["rang_jour"])),
                 "même type sur la période": "{} GAB-jours, {} GAB".format(len(inv_toutes), inv_toutes["num_automate"].nunique())})
if "gab_touches" in globals():
    _cas.append({"cas": "GAB dégradés (manifestations)", "GAB / date": "à partir du {} ({})".format(d_man.strftime("%d/%m/%Y"), libzone),
                 "détecté": "{} GAB touchés, pic de {} le même jour".format(len(gab_touches), nb(_par_jour[_vis].max())),
                 "même type sur la période": "{} arrêts en cours de journée".format(nb(int(EV["code"].isin(["R35", "R35z"]).sum())))})
if "_jx" in globals() and len(_jx):
    _cas.append({"cas": "Rafale {}".format(LIB_EXO.get(res_x, res_x)), "GAB / date": "le {}".format(d_x.strftime("%d/%m/%Y")),
                 "détecté": "{} retraits sur {} GAB, {} Critique".format(nb(_jx[col_x].sum()), len(_jx), int((_jx["niveau"] == "Critique").sum())),
                 "même type sur la période": "{} GAB-jours d'afflux de cartes étrangères".format(nb(int(EV["code"].isin(["R07", "R07f", "R20"]).sum())))})
if _cas:
    display(tableau(pd.DataFrame(_cas), "Les trois cas de la présentation"))
_fm = EV[EV["gravite"] >= 1].assign(mois=EV["date_jour"].dt.to_period("M").astype(str)).groupby(["mois", "famille_anomalie"])["num_automate"].nunique().unstack(fill_value=0)
_fm = _fm[[f for f in ORDRE_FAMILLES_ANOMALIES if f in _fm.columns]]
g_barres(_fm.index, [(f, _fm[f].values, SERIE[i % len(SERIE)]) for i, f in enumerate(_fm.columns)], "GAB distincts en alerte chaque mois, par famille d'anomalie (À surveiller ou plus)", y_titre="GAB distincts", hauteur=360, empile=True)
_top_an = catalogue[catalogue["gravite"] >= 1].sort_values("GAB distincts", ascending=False).head(10)[["famille", "anomalie", "niveau si seule", "GAB-jours", "GAB distincts"]]
display(tableau(_top_an, "Les 10 anomalies qui touchent le plus de GAB", pills=("niveau si seule",)))
if "synthese_records" in globals() and len(synthese_records):
    display(tableau(synthese_records[["réseau", "retraits sur la période", "record d'un GAB en un jour", "record du parc en un jour"]], "Les records des réseaux étrangers"))

<a href="#sommaire" style="font-size:12px">↑ retour au sommaire</a>
<a id="s19bis"></a>
## 19 bis. La vision annuelle
**En bref** : chaque année côte à côte. Le parc grandit-il ou baisse-t-il ? Les alertes sont-elles plus nombreuses ? Les mêmes
anomalies reviennent-elles ? Les mêmes GAB posent-ils problème d'une année sur l'autre ?

**Ce que vous voyez** : un tableau d'une colonne par année, les retraits et les GAB en alerte mois par mois avec une courbe par année
(pour comparer le même mois d'une année à l'autre), les familles d'anomalies par année, et les GAB les plus souvent en alerte chaque année.
Une année incomplète (la première ou l'année en cours) est signalée par son nombre de jours.

In [ ]:
_Y = P.assign(annee=P["date_jour"].dt.year, mois=P["date_jour"].dt.month)
_ce = _Y["niveau"].isin(["Critique", "Élevé"])
_al = _Y["niveau"] != "Normal"
_gy = _Y.groupby("annee")
VUE_ANNUELLE = pd.DataFrame({"jours analysés": _gy["date_jour"].nunique(), "GAB présents": _gy["num_automate"].nunique(),
                             "GAB qui ont servi": _Y[_Y["nb_retraits"] > 0].groupby("annee")["num_automate"].nunique(),
                             "retraits": _gy["nb_retraits"].sum(), "retraits par jour (parc)": (_gy["nb_retraits"].sum() / _gy["date_jour"].nunique()).round(0),
                             "montant retiré (M€)": (_gy["montant_total"].sum() / 1e6).round(1),
                             "GAB-jours en alerte (À surveiller ou plus)": _al.groupby(_Y["annee"]).sum(), "dont Critique ou Élevé": _ce.groupby(_Y["annee"]).sum(),
                             "GAB différents en alerte Critique ou Élevé": _Y[_ce].groupby("annee")["num_automate"].nunique(),
                             "jours sans aucun GAB atypique": _Y.assign(_a=_al.astype(int)).groupby(["annee", "date_jour"])["_a"].sum().eq(0).groupby("annee").sum()}).fillna(0)
VUE_ANNUELLE["alertes Critique ou Élevé par jour"] = (VUE_ANNUELLE["dont Critique ou Élevé"] / VUE_ANNUELLE["jours analysés"]).round(1)
_ev_y = EV[EV["gravite"] >= 1].assign(annee=EV["date_jour"].dt.year)
VUE_ANNUELLE["anomalie la plus fréquente"] = _ev_y.groupby("annee")["anomalie"].agg(lambda s: s.value_counts().index[0] if len(s) else "")
for c in VUE_ANNUELLE.columns:
    if c not in ("anomalie la plus fréquente", "alertes Critique ou Élevé par jour", "montant retiré (M€)"):
        VUE_ANNUELLE[c] = VUE_ANNUELLE[c].astype(int)
bandeau("Vision annuelle", "du {} au {}".format(DEBUT_PERIODE.strftime("%d/%m/%Y"), FIN.strftime("%d/%m/%Y")))
_va = VUE_ANNUELLE.T.reset_index().rename(columns={"index": "indicateur"})
_va.columns = [str(c) for c in _va.columns]
display(tableau(_va, "Une colonne par année", n=30))
_jm = _Y.groupby(["annee", "mois"])["date_jour"].nunique()
_mr = _Y.groupby(["annee", "mois"])["nb_retraits"].sum().div(_jm).where(_jm >= 20).unstack(0)          # un mois de moins de 20 jours n'est pas tracé (mois en cours)
g_courbes(list(MOIS_FR), [(str(a), _mr[a].reindex(range(1, 13)).round(0).values, SERIE[i % len(SERIE)]) for i, a in enumerate(_mr.columns)],
          "Retraits du parc par jour, mois par mois : une courbe par année", y_titre="retraits par jour", hauteur=320, hover="%{x}<br>%{y:,.0f} retraits par jour<extra>%{fullData.name}</extra>")
_ma = _Y[_ce].groupby(["annee", "mois"])["num_automate"].nunique().reindex(_jm.index).fillna(0).where(_jm >= 20).unstack(0)
if len(_ma):
    g_courbes(list(MOIS_FR), [(str(a), _ma[a].reindex(range(1, 13)).values, SERIE[i % len(SERIE)]) for i, a in enumerate(_ma.columns)],
              "GAB différents en alerte Critique ou Élevé, mois par mois : une courbe par année", y_titre="GAB", hauteur=300,
              hover="%{x}<br>%{y:,.0f} GAB<extra>%{fullData.name}</extra>")
_fy = _ev_y.groupby(["famille_anomalie", "annee"])["num_automate"].nunique().unstack(1).reindex([f for f in ORDRE_FAMILLES_ANOMALIES if f in _ev_y["famille_anomalie"].unique()]).fillna(0)
g_barres(list(_fy.index), [(str(a), _fy[a].values, SERIE[i % len(SERIE)]) for i, a in enumerate(_fy.columns)], "GAB différents touchés par chaque famille d'anomalie, année par année", hauteur=340)
_tg = _Y[_ce].groupby(["annee", "num_automate"]).size().reset_index(name="jours en alerte").sort_values(["annee", "jours en alerte"], ascending=[False, False]).groupby("annee").head(5)
_tg["anomalies les plus fréquentes"] = [", ".join(_ev_y[(_ev_y["annee"] == a_) & (_ev_y["num_automate"] == g_)]["anomalie"].value_counts().head(2).index) for a_, g_ in zip(_tg["annee"], _tg["num_automate"])]
display(tableau(_tg.rename(columns={"annee": "année", "num_automate": "GAB"}), "Les 5 GAB les plus souvent en alerte (Critique ou Élevé), chaque année"))
_a1, _a0 = (VUE_ANNUELLE.index[-1], VUE_ANNUELLE.index[-2]) if len(VUE_ANNUELLE) >= 2 else (VUE_ANNUELLE.index[-1], None)
retenir(["<b>{}</b> : {} retraits par jour pour le parc{}.".format(_a1, nb(VUE_ANNUELLE.loc[_a1, "retraits par jour (parc)"]),
             " contre {} en {} ({}{} %)".format(nb(VUE_ANNUELLE.loc[_a0, "retraits par jour (parc)"]), _a0, "+" if VUE_ANNUELLE.loc[_a1, "retraits par jour (parc)"] >= VUE_ANNUELLE.loc[_a0, "retraits par jour (parc)"] else "",
                                             nb((VUE_ANNUELLE.loc[_a1, "retraits par jour (parc)"] / max(1, VUE_ANNUELLE.loc[_a0, "retraits par jour (parc)"]) - 1) * 100, 1)) if _a0 is not None else ""),
         "<b>{}</b> alertes Critique ou Élevé par jour en {}{}.".format(nb(VUE_ANNUELLE.loc[_a1, "alertes Critique ou Élevé par jour"], 1), _a1,
             " contre {} en {}".format(nb(VUE_ANNUELLE.loc[_a0, "alertes Critique ou Élevé par jour"], 1), _a0) if _a0 is not None else ""),
         "L'anomalie la plus fréquente en {} : <b>{}</b>.".format(_a1, VUE_ANNUELLE.loc[_a1, "anomalie la plus fréquente"])])

# Partie E — Les livrables

<a href="#sommaire" style="font-size:12px">↑ retour au sommaire</a>
<a id="s20"></a>
## 20. Le fichier Excel et les exports
**Le classeur de synthèse** (lien de téléchargement dans la cellule) donne au métier toute la période, un onglet par usage :

| Onglet | Contenu | Une ligne par |
|---|---|---|
| Lisez-moi | période, réglages, lecture des niveaux et des comptes | — |
| Par année, Par mois | activité, incidents, GAB-jours **et GAB distincts** par niveau, part du parc en alerte, anomalies les plus fréquentes | année / mois |
| Parc par jour | activité, incidents, cartes étrangères, GAB par niveau, GAB le plus atypique du jour, anomalies du jour | jour |
| Top du jour | les `TOP_LISTE` GAB les plus atypiques de chaque jour, avec diagnostic, voisins et chiffres ; colonne « avis métier » à remplir | jour × GAB listé |
| Alertes | tous les événements Critique ou Élevé (une anomalie sur un GAB un jour) | événement |
| GAB | le profil de chaque GAB sur la période : famille, habitude, jours à 0, jours en alerte, anomalies les plus fréquentes, voisins | GAB |
| Familles | le parc et les alertes par famille de GAB | famille |
| Anomalies | le catalogue : famille, comment on décide, niveau, déclenchements (GAB-jours et GAB distincts) | anomalie |
| Cas inversion | toutes les inversions de cassettes possibles détectées | GAB-jour |
| Cas manifestations | les GAB de la zone suivie à l'arrêt ou en forte baisse après la date des manifestations | GAB |
| Cas réseau étranger | les GAB qui ont vu le réseau étranger le jour de la rafale | GAB |
| Records réseaux étrangers | pour chaque réseau étranger, les journées record d'un GAB et la synthèse | GAB-jour |
| Exemples par anomalie | pour chaque anomalie, le cas réel le mieux classé | anomalie |
| Besoin en billets, GAB souvent vides, Créneau de maintenance, Fiabilité, Usage comparé | les réponses de la section 18 | GAB |
| Podiums par jour | chaque jour, les 3 GAB en tête pour chaque indicateur (retraits, montant, cartes avalées, cartes étrangères…) | jour × indicateur |

Dans les onglets Top du jour et Alertes, la colonne **le chiffre qui le prouve** (ou « indicateur clé / valeur du jour / valeur habituelle »)
donne, à côté de chaque diagnostic, le chiffre du jour et l'habitude du GAB : de quoi vérifier le diagnostic sans refaire le calcul.

**Rejouer un cas** : onglet **Top du jour**, filtrer la colonne « jour » sur la date voulue : les GAB les plus atypiques de ce jour
apparaissent dans l'ordre, avec leur diagnostic. Pour un GAB précis à une date précise, onglet **Alertes** (filtrer GAB et jour).

**La photo complète GAB × jour** (tous les GAB, atypiques et normaux, chaque jour) va dans des fichiers séparés : un onglet
Excel est limité à 1 048 576 lignes, donc chaque fichier prend le maximum de jours possible (1 048 575 ÷ nombre de GAB, soit
quelques mois pour un grand parc) et `NB_FICHIERS_PHOTO` fichiers remontent dans le temps. Un fichier plein pèse ~120 Mo : il est
déposé dans le dossier Dataiku `NOM_DOSSIER_DATAIKU` ; le lien direct n'est proposé que sous 50 Mo.

In [ ]:
def create_download_link_excel(df, filename="data.xlsx"):
    """Lien de téléchargement Excel encodé en base64 (fonction des notebooks précédents, inchangée)."""
    towrite = BytesIO()
    df.to_excel(towrite, index=False, engine="openpyxl")
    towrite.seek(0)
    b64 = base64.b64encode(towrite.read()).decode()
    href = ('<a href="data:application/vnd.openxmlformats-officedocument.spreadsheetml.sheet;base64,{}" download="{}">'
            "Télécharger le fichier Excel ({})</a>".format(b64, filename, filename))
    return HTML(href)


def deposer_dans_dossier_dataiku(df, filename):
    """Écrit un DataFrame en Excel dans un dossier géré Dataiku (fonction des notebooks précédents, inchangée)."""
    flux = BytesIO()
    df.to_excel(flux, index=False, engine="openpyxl")
    flux.seek(0)
    dataiku.Folder(NOM_DOSSIER_DATAIKU).upload_stream(filename, flux)
    print("Déposé dans le dossier Dataiku '{}' : {}".format(NOM_DOSSIER_DATAIKU, filename))


def lien_telechargement(octets, filename):
    mime = "application/zip" if filename.endswith(".zip") else "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet"
    b64 = base64.b64encode(octets).decode()
    return HTML(CSS + '<div class="g-note ok"><a href="data:{};base64,{}" download="{}" style="font-size:1.1em"><b>Télécharger {}</b></a> ({} Mo)</div>'
                .format(mime, b64, filename, filename, nb(len(octets) / 1e6, 1)))


LIMITE_EXCEL = 1048575


def classeur(feuilles, nom):
    """feuilles = [(onglet, DataFrame)]. Excel avec openpyxl, sinon xlsxwriter ; sinon un zip de CSV. Retourne (octets, nom, moteur)."""
    for moteur in ("openpyxl", "xlsxwriter"):
        flux = BytesIO()
        try:
            with pd.ExcelWriter(flux, engine=moteur) as w:
                for onglet, df in feuilles:
                    df.head(LIMITE_EXCEL).to_excel(w, sheet_name=onglet[:31], index=False)
            return flux.getvalue(), nom, moteur
        except ImportError:
            continue
    flux = BytesIO()
    with zipfile.ZipFile(flux, "w", zipfile.ZIP_DEFLATED) as z:
        for onglet, df in feuilles:
            nom_csv_ = unicodedata.normalize("NFKD", onglet).encode("ascii", "ignore").decode() + ".csv"
            z.writestr(nom_csv_, "﻿" + df.to_csv(index=False, sep=";"))
    return flux.getvalue(), nom.replace(".xlsx", "") + "_csv.zip", "zip de CSV (ni openpyxl ni xlsxwriter installés)"


def jj(d):
    return d.dt.strftime("%Y-%m-%d")


def photo_de(df):
    """La photo lisible d'un ensemble de GAB-jours : identité, niveau, rang du jour, diagnostic, chiffres clés, voisins."""
    p = df.copy()
    p["diagnostic"] = p["diagnostic"].where(p["diagnostic"] != "", np.where(p["hab_retraits"].isnull(), "historique insuffisant pour juger", "rien d'anormal"))
    p["resume"] = p["diagnostic"].str.split(" + ").str[0]
    p["voisins"] = p["num_automate"].map(lambda x: ", ".join("{} ({} km)".format(v, d) for v, d in VOISINS.get(x, [])))
    p["prioritaire"] = ((p["rang_jour"] <= TOP_PRIORITAIRE) & (p["niveau"] != "Normal")).astype(int)
    p["liste_du_jour"] = ((p["rang_jour"] <= TOP_LISTE) & (p["niveau"] != "Normal")).astype(int)
    p["avis_metier"] = ""
    cols = ["date_jour", "rang_jour", "prioritaire", "liste_du_jour", "niveau", "score_regles"] + ident \
        + [c for c in ["ville", "code_postal", "latitude", "longitude", "statut_gab", "type_gab_e_i"] if c in p.columns] \
        + ["resume", "nb_causes", "anomalies", "diagnostic", "indicateur clé", "valeur du jour", "valeur habituelle", "action conseillée", "nb_retraits", "hab_retraits"] + (["attendu_retraits"] if "attendu_retraits" in p.columns else []) \
        + ["montant_total"] + [c for c in ["nb_inaboutis", "nb_captures", "nb_retraits_repetes_10min", "nb_ret_exotique", "nb_0h_6h", "plan", "ferie"] if c in p.columns] \
        + ["voisins", "avis_metier"]
    p = p[cols].sort_values(["date_jour", "rang_jour"], ascending=[False, True])
    p["date_jour"] = jj(p["date_jour"])
    for c in ("hab_retraits", "attendu_retraits"):
        if c in p.columns:
            p[c] = p[c].round(1)
    return p.rename(columns=COLONNES_EXCEL).reset_index(drop=True)


COLONNES_EXCEL = {"date_jour": "jour", "rang_jour": "rang du jour", "prioritaire": "prioritaire", "liste_du_jour": "dans la liste du jour", "niveau": "niveau", "score_regles": "score",
                  "num_automate": "GAB", "code_site_implantation": "code site", "famille": "famille de GAB", "ville": "ville", "code_postal": "code postal", "latitude": "latitude",
                  "longitude": "longitude", "statut_gab": "statut", "type_gab_e_i": "type", "resume": "résumé", "nb_causes": "nombre de causes", "anomalies": "anomalies",
                  "diagnostic": "diagnostic complet", "nb_retraits": "retraits", "hab_retraits": "habitude (retraits)", "attendu_retraits": "attendu avec la saison", "montant_total": "montant retiré (€)",
                  "nb_inaboutis": "inaboutis", "nb_captures": "captures", "nb_retraits_repetes_10min": "retraits refaits < 10 min", "nb_ret_exotique": "cartes étrangères",
                  "nb_0h_6h": "retraits 0 h – 6 h", "plan": "plan de cassettes", "ferie": "jour férié", "voisins": "voisins", "avis_metier": "avis métier", "codes": "codes techniques"}


def libelles_frequents(ev_, cle, n):
    if not len(ev_):
        return pd.Series(dtype=object)
    t_ = ev_.groupby([cle, "anomalie"]).size().reset_index(name="n").sort_values([cle, "n"], ascending=[True, False])
    return t_.groupby(cle).apply(lambda x_: ", ".join("{} ({})".format(r_, n_) for r_, n_ in zip(x_["anomalie"].head(n), x_["n"].head(n))))


EV2 = EV[EV["gravite"] >= 2]

# --- Parc par jour
g_ = P.groupby("date_jour")
parc_jour = pd.DataFrame({"jour de semaine": g_["jour_semaine"].first().map(lambda d: JOURS_FR[int(d) - 1]), "GAB dans la fiche": g_["num_automate"].nunique(),
                          "GAB actifs": P.assign(_a=(P["nb_retraits"] > 0).astype(int)).groupby("date_jour")["_a"].sum(),
                          "retraits": g_["nb_retraits"].sum().astype(int), "montant retiré (€)": g_["montant_total"].sum().round(0)})
if "ferie" in P.columns:
    parc_jour.insert(1, "jour férié", g_["ferie"].first())
parc_jour["GAB sans retrait"] = parc_jour["GAB dans la fiche"] - parc_jour["GAB actifs"]
parc_jour["dont à l'arrêt (servent d'habitude ce jour-là)"] = P.assign(_a=((P["nb_retraits"] == 0) & (P["part_zero"].fillna(0) <= 0.15) & (P["hab_retraits"].fillna(0) >= 10)).astype(int)).groupby("date_jour")["_a"].sum()
if "statut_gab" in P.columns:
    parc_jour["GAB actifs au référentiel"] = P.assign(_s=(P["statut_gab"].astype(str).str.upper() == "ACTIF").astype(int)).groupby("date_jour")["_s"].sum()
    parc_jour["GAB non actifs au référentiel"] = parc_jour["GAB dans la fiche"] - parc_jour["GAB actifs au référentiel"]
parc_jour["données incomplètes"] = np.where(parc_jour.index.isin(jours_incomplets), "oui", "")
parc_jour["retraits par GAB actif"] = (parc_jour["retraits"] / parc_jour["GAB actifs"].clip(lower=1)).round(1)
parc_jour["montant moyen d'un retrait (€)"] = (parc_jour["montant retiré (€)"] / parc_jour["retraits"].clip(lower=1)).round(1)
for col, lib in [("nb_inaboutis", "inaboutis"), ("nb_captures", "captures"), ("nb_ret_cos", "retraits COS"), ("nb_ret_exotique", "retraits cartes étrangères"),
                 ("nb_ret_cup", "retraits UnionPay"), ("nb_ret_jcb", "retraits JCB"), ("nb_ret_diners", "retraits Diners"), ("nb_ret_amex", "retraits Amex"),
                 ("nb_retraits_repetes_10min", "retraits refaits < 10 min"), ("nb_0h_6h", "retraits 0 h – 6 h")]:
    if col in P.columns:
        parc_jour[lib] = g_[col].sum().astype(int)
_js = P.groupby("date_jour")["jour_semaine"].first()
parc_jour["retraits des 4 mêmes jours de semaine avant"] = parc_jour["retraits"].groupby(_js).transform(lambda s_: s_.shift(1).rolling(4, min_periods=2).mean()).round(0)
parc_jour["écart retraits (%)"] = ((parc_jour["retraits"] / parc_jour["retraits des 4 mêmes jours de semaine avant"] - 1) * 100).round(1)
for niv in ORDRE_NIVEAUX:
    parc_jour["GAB " + niv] = P.assign(_n=(P["niveau"] == niv).astype(int)).groupby("date_jour")["_n"].sum()
parc_jour["GAB en alerte (liste du jour)"] = parc_jour["GAB Critique"] + parc_jour["GAB Élevé"] + parc_jour["GAB À surveiller"]
parc_jour["jour sans GAB atypique"] = np.where(parc_jour["GAB en alerte (liste du jour)"] == 0, "oui", "")
parc_jour["GAB sans historique"] = P.assign(_h=(P["hab_retraits"].isnull() & (P["nb_retraits"] > 0)).astype(int)).groupby("date_jour")["_h"].sum()
_top1 = P[P["rang_jour"] == 1].set_index("date_jour")
parc_jour["GAB le plus atypique"] = _top1["num_automate"]
parc_jour["son niveau"] = _top1["niveau"]
parc_jour["son résumé"] = _top1["resume"]
parc_jour["anomalies du jour (Critique ou Élevé)"] = libelles_frequents(EV2, "date_jour", 4).reindex(parc_jour.index).fillna("")
parc_jour = parc_jour.sort_index(ascending=False).reset_index().rename(columns={"date_jour": "jour"})
parc_jour["jour"] = jj(parc_jour["jour"])

# --- Top du jour et photo complète
top_jours = photo_de(P[(P["rang_jour"] <= TOP_LISTE) & (P["niveau"] != "Normal")])          # seuls les GAB en alerte
NB_GAB_PERIODE = P["num_automate"].nunique()
JOURS_MAX_ONGLET = max(1, LIMITE_EXCEL // max(1, NB_GAB_PERIODE))
N_JOURS_PHOTO = JOURS_MAX_ONGLET if str(JOURS_PHOTO_COMPLETE_EXCEL).lower() == "max" else min(int(JOURS_PHOTO_COMPLETE_EXCEL), JOURS_MAX_ONGLET)

# --- Alertes (événements Critique ou Élevé)
alertes_excel = EV2.merge(jour[["num_automate", "date_jour", "niveau", "rang_jour", "le chiffre qui le prouve", "action conseillée"] + [c for c in ident + ["ville", "code_postal"] if c != "num_automate" and c in jour.columns]],
                          on=["num_automate", "date_jour"], how="left").sort_values(["date_jour", "gravite", "num_automate"], ascending=[False, False, True])
alertes_excel = alertes_excel[["date_jour"] + ident + [c for c in ["ville", "code_postal"] if c in alertes_excel.columns] + ["niveau", "rang_jour", "famille_anomalie", "anomalie", "gravite", "texte", "le chiffre qui le prouve", "action conseillée"]]
alertes_excel["date_jour"] = jj(alertes_excel["date_jour"])
alertes_excel = alertes_excel.rename(columns=dict(COLONNES_EXCEL, **{"famille_anomalie": "famille d'anomalie", "anomalie": "anomalie", "gravite": "gravité", "texte": "diagnostic", "code": "code technique"}))

# --- GAB : le profil de chaque GAB sur la période
gg = P.groupby("num_automate")
fiche_gab = pd.DataFrame({"jours observés": gg.size(), "retraits par jour": gg["nb_retraits"].mean().round(1), "montant par jour (€)": gg["montant_total"].mean().round(0),
                          "part de jours sans retrait (%)": (P.assign(_z=(P["nb_retraits"] == 0).astype(float)).groupby("num_automate")["_z"].mean() * 100).round(1),
                          "retraits max un jour": gg["nb_retraits"].max()})
for col, lib in [("nb_inaboutis", "inaboutis par jour"), ("nb_captures", "captures par jour"), ("nb_ret_exotique", "cartes étrangères par jour"), ("nb_0h_6h", "retraits de nuit par jour")]:
    if col in P.columns:
        fiche_gab[lib] = gg[col].mean().round(2)
for niv in ["Critique", "Élevé", "À surveiller"]:
    fiche_gab["jours " + niv] = P.assign(_n=(P["niveau"] == niv).astype(int)).groupby("num_automate")["_n"].sum()
fiche_gab["jours dans le top {}".format(TOP_PRIORITAIRE)] = P.assign(_t=(P["rang_jour"] <= TOP_PRIORITAIRE).astype(int)).groupby("num_automate")["_t"].sum()
_da = P[P["niveau"].isin(["Critique", "Élevé"])].groupby("num_automate")["date_jour"].max()
fiche_gab["dernière alerte"] = jj(_da.reindex(fiche_gab.index)).fillna("")
fiche_gab["anomalies les plus fréquentes"] = libelles_frequents(EV2, "num_automate", 3).reindex(fiche_gab.index).fillna("")
fiche_gab["voisins"] = fiche_gab.index.map(lambda x: ", ".join("{} ({} km)".format(v, d) for v, d in VOISINS.get(x, [])))
_cols_id = [c for c in ident + ["ville", "code_postal", "latitude", "longitude", "statut_gab", "type_gab_e_i"] if c != "num_automate" and c in P.columns]
_id = P.sort_values("date_jour").groupby("num_automate")[_cols_id].last() if _cols_id else pd.DataFrame(index=fiche_gab.index)
if "PROFILS_GAB" in globals():
    fiche_gab = fiche_gab.join(PROFILS_GAB.drop(columns=["retraits par jour (3 ans)"]), how="left")
fiche_gab = _id.join(fiche_gab).reset_index().sort_values(["jours Critique", "jours Élevé", "retraits par jour"], ascending=[False, False, False]).rename(columns=COLONNES_EXCEL)


# --- Par année et par mois
def synthese_par(cle_fmt, nom_cle):
    k = P["date_jour"].dt.strftime(cle_fmt).rename(nom_cle)
    gk = P.groupby(k)
    t = pd.DataFrame({"jours": gk["date_jour"].nunique(), "GAB": gk["num_automate"].nunique(), "GAB-jours": gk.size(),
                      "retraits": gk["nb_retraits"].sum().astype(int), "montant retiré (€)": gk["montant_total"].sum().round(0),
                      "retraits par GAB et par jour": (gk["nb_retraits"].sum() / gk.size()).round(1),
                      "GAB-jours sans retrait (%)": (P.assign(_z=(P["nb_retraits"] == 0).astype(float)).groupby(k)["_z"].mean() * 100).round(1)})
    for col, lib in [("nb_inaboutis", "inaboutis"), ("nb_captures", "captures"), ("nb_ret_exotique", "retraits cartes étrangères"), ("nb_0h_6h", "retraits 0 h – 6 h")]:
        if col in P.columns:
            t[lib] = gk[col].sum().astype(int)
    for niv in ["Critique", "Élevé", "À surveiller"]:
        t["GAB-jours " + niv] = P.assign(_n=(P["niveau"] == niv).astype(int)).groupby(k)["_n"].sum()
        t["GAB distincts " + niv] = P[P["niveau"] == niv].groupby(k)["num_automate"].nunique().reindex(t.index).fillna(0).astype(int)
    t["alertes par jour (Critique ou Élevé)"] = ((t["GAB-jours Critique"] + t["GAB-jours Élevé"]) / t["jours"]).round(1)
    t["part du parc en alerte (%)"] = ((t["GAB-jours Critique"] + t["GAB-jours Élevé"]) / t["GAB-jours"] * 100).round(2)
    t["GAB distincts en alerte (Critique ou Élevé)"] = P[P["niveau"].isin(["Critique", "Élevé"])].groupby(k)["num_automate"].nunique().reindex(t.index).fillna(0).astype(int)
    t["GAB jamais en alerte"] = t["GAB"] - t["GAB distincts en alerte (Critique ou Élevé)"]
    ke = EV["date_jour"].dt.strftime(cle_fmt).rename(nom_cle)
    t["anomalies les plus fréquentes"] = libelles_frequents(EV2.assign(**{nom_cle: ke}), nom_cle, 5).reindex(t.index).fillna("")
    _tg = P[P["niveau"].isin(["Critique", "Élevé"])].groupby([k, "num_automate"]).size().reset_index(name="n").sort_values([nom_cle, "n"], ascending=[True, False])
    t["GAB le plus souvent en alerte"] = _tg.groupby(nom_cle).apply(lambda x_: "{} ({} jours)".format(x_["num_automate"].iloc[0], int(x_["n"].iloc[0]))).reindex(t.index).fillna("") if len(_tg) else ""
    return t.reset_index()


par_annee = synthese_par("%Y", "année")
par_mois = synthese_par("%Y-%m", "mois")
bandeau("Synthèse par année", "ce que le métier retrouve dans l'onglet « Par année »")
_pa = par_annee.set_index("année").drop(columns=["anomalies les plus fréquentes", "GAB le plus souvent en alerte"]).T.reset_index().rename(columns={"index": "indicateur"})
_pa.columns = [str(c) for c in _pa.columns]
display(tableau(_pa, "Une colonne par année", n=40))
display(tableau(par_annee[["année", "anomalies les plus fréquentes", "GAB le plus souvent en alerte"]], "Les anomalies les plus fréquentes de chaque année", longues=("anomalies les plus fréquentes",)))

# --- Familles
par_famille = None
if "famille" in P.columns:
    gf = P.groupby("famille")
    par_famille = pd.DataFrame({"GAB": gf["num_automate"].nunique(), "GAB-jours": gf.size(), "retraits par GAB et par jour": (gf["nb_retraits"].sum() / gf.size()).round(1)})
    for niv in ["Critique", "Élevé", "À surveiller"]:
        par_famille["GAB-jours " + niv] = P.assign(_n=(P["niveau"] == niv).astype(int)).groupby("famille")["_n"].sum()
    par_famille["GAB distincts en alerte (Critique ou Élevé)"] = P[P["niveau"].isin(["Critique", "Élevé"])].groupby("famille")["num_automate"].nunique().reindex(par_famille.index).fillna(0).astype(int)
    par_famille["part des GAB-jours en alerte (%)"] = ((par_famille["GAB-jours Critique"] + par_famille["GAB-jours Élevé"]) / par_famille["GAB-jours"] * 100).round(2)
    par_famille["anomalies les plus fréquentes"] = libelles_frequents(EV2.merge(P[["num_automate", "date_jour", "famille"]], on=["num_automate", "date_jour"], how="left"), "famille", 4).reindex(par_famille.index).fillna("")
    par_famille = par_famille.reindex([f for f in ORDRE_FAMILLES if f in par_famille.index]).reset_index().rename(columns={"famille": "famille de GAB"})

# --- Lisez-moi
lisez_moi = pd.DataFrame({"rubrique": ["Période", "Dernier jour complet", "Jours écartés en fin de fiche", "GAB", "Entrée", "Voisins", "GAB-jour", "Niveau", "Score", "Rang du jour", "Prioritaire",
                                       "Habitude", "Attendu saisonnier", "Onglets Par année / Par mois", "Onglet Parc par jour", "Onglet Top du jour", "Onglet Alertes", "Onglet GAB", "Onglet Anomalies", "Photo complète", "Généré le"],
                          "contenu": ["du {} au {} ({} jours)".format(DEBUT_PERIODE.strftime("%d/%m/%Y"), FIN.strftime("%d/%m/%Y"), P["date_jour"].nunique()), FIN.strftime("%d/%m/%Y"),
                                      ", ".join(d_.strftime("%d/%m/%Y") for d_ in sorted(ecartes)) or "aucun", "{} GAB dans la fiche".format(P["num_automate"].nunique()), DATASET_ENTREE, source_voisins or "aucun",
                                      "un GAB un jour donné ; un GAB en alerte 10 jours = 10 GAB-jours et 1 GAB distinct",
                                      "niveau de l'anomalie la plus grave du jour : Critique (gravité 3), Élevé (2), À surveiller (1), Normal (aucune)",
                                      "somme des gravités des anomalies du jour ; ne sert qu'à classer", "1 = le GAB le plus atypique du jour (score, puis gravité, puis écart total)",
                                      "les {} premiers rangs de chaque jour (la liste en compte {})".format(TOP_PRIORITAIRE, TOP_LISTE),
                                      "médiane du même jour de semaine sur les 4 semaines précédentes (sinon 28 jours) ; événements rares : moyenne 28 jours",
                                      "habitude x niveau de l'an dernier à la même date (±2 semaines) / 4 semaines avant",
                                      "la synthèse : activité, incidents, GAB-jours et GAB distincts par niveau, part du parc en alerte, anomalies les plus fréquentes",
                                      "une ligne par jour : activité, incidents, niveaux, GAB le plus atypique, anomalies du jour",
                                      "les {} GAB les plus atypiques de chaque jour, avec diagnostic et voisins ; colonne « avis métier » à remplir".format(TOP_LISTE),
                                      "un événement = une anomalie déclenchée sur un GAB un jour (Critique ou Élevé seulement)", "le profil de chaque GAB sur la période",
                                      "le catalogue : famille, comment on décide, niveau, déclenchements",
                                      "tous les GAB x tous les jours : fichiers {}_photo_gab_jour_*.xlsx ({} jours par fichier, {} fichier(s)) dans le dossier {} ; dataset {}".format(
                                          NOM_EXCEL_SORTIE.replace(".xlsx", ""), N_JOURS_PHOTO, NB_FICHIERS_PHOTO, NOM_DOSSIER_DATAIKU or "non demandé", DATASET_SORTIE_HISTO or "non demandé"),
                                      pd.Timestamp.now().strftime("%d/%m/%Y %H:%M")]})
FEUILLES = [("Lisez-moi", lisez_moi), ("Par année", par_annee), ("Par mois", par_mois), ("Parc par jour", parc_jour), ("Top du jour", top_jours), ("Alertes", alertes_excel), ("GAB", fiche_gab)] \
    + ([("Familles", par_famille)] if par_famille is not None else []) + [("Anomalies", catalogue.drop(columns=["code technique"], errors="ignore"))]


def _x(df_):
    d_ = df_.copy()
    for c in d_.columns:
        if hasattr(d_[c], "dt"):
            d_[c] = d_[c].dt.strftime("%Y-%m-%d")
    return d_


if "inv_toutes" in globals() and len(inv_toutes):
    FEUILLES.append(("Cas inversion", _x(inv_toutes.drop(columns=["gravite", "code"], errors="ignore").rename(columns=dict(COLONNES_EXCEL, texte="diagnostic", code="code technique", famille_anomalie="famille d'anomalie")))))
if "gab_touches" in globals() and len(gab_touches):
    FEUILLES.append(("Cas manifestations", _x(gab_touches.rename(columns={"num_automate": "GAB", "premier_jour": "premier jour", "jours_en_alerte": "jours en alerte"}))))
if "_jx" in globals() and len(_jx):
    FEUILLES.append(("Cas réseau étranger", _x(_jx[[c for c in _cols if c in _jx.columns]].rename(columns={"num_automate": "GAB", "rang_jour": "rang du jour", "resume": "résumé"}))))
if "records" in globals() and len(records):
    FEUILLES.append(("Records réseaux étrangers", _x(records.rename(columns={"date_jour": "jour", "num_automate": "GAB", "rang_jour": "rang du jour"}))))
    FEUILLES.append(("Records synthèse", synthese_records))
if "EXEMPLES" in globals() and len(EXEMPLES):
    FEUILLES.append(("Exemples par anomalie", _x(EXEMPLES[["famille_anomalie", "anomalie", "num_automate", "date_jour", "rang_jour", "niveau", "texte"]].rename(
        columns={"famille_anomalie": "famille d'anomalie", "num_automate": "GAB", "date_jour": "jour", "rang_jour": "rang du jour", "texte": "diagnostic"}))))
# podiums de chaque jour : les 3 GAB en tête par indicateur (tri du parc entier, une fois par indicateur)
_pods = []
for col_, titre_, unite_, hab_ in PODIUM:
    t_ = P[P[col_] > 0].sort_values(["date_jour", col_], ascending=[True, False]).groupby("date_jour").head(3)
    t_ = t_.assign(indicateur=titre_, rang=t_.groupby("date_jour").cumcount() + 1, valeur=t_[col_], unité=unite_,
                   habitude=t_[hab_] if hab_ and hab_ in t_.columns else t_.get("ref_" + col_, np.nan))
    _pods.append(t_[["date_jour", "indicateur", "rang", "num_automate"] + [c for c in ["ville"] if c in t_.columns] + ["valeur", "unité", "habitude", "nb_retraits", "montant_total", "niveau"]])
if _pods:
    PODIUMS = pd.concat(_pods, ignore_index=True).sort_values(["date_jour", "indicateur", "rang"], ascending=[False, True, True]).rename(
        columns={"date_jour": "jour", "num_automate": "GAB", "habitude": "habitude du GAB", "nb_retraits": "retraits ce jour-là", "montant_total": "montant retiré (€)"})
    FEUILLES.append(("Podiums par jour", _x(PODIUMS.round(1))))
if "VUE_ANNUELLE" in globals():
    FEUILLES.insert(1, ("Vision annuelle", VUE_ANNUELLE.reset_index().rename(columns={"annee": "année"})))
for _nom, _var in [("Besoin en billets", "BESOIN_BILLETS"), ("GAB souvent vides", "GAB_VIDES"), ("Créneau de maintenance", "MAINTENANCE"), ("Fiabilité", "FIABILITE"), ("Usage comparé", "USAGE")]:
    if _var in globals() and len(globals()[_var]):
        FEUILLES.append((_nom, _x(globals()[_var])))
display(tableau(pd.DataFrame([(o_, len(d_), d_.shape[1]) for o_, d_ in FEUILLES], columns=["onglet", "lignes", "colonnes"]), "Les onglets du classeur"))

### Télécharger le fichier Excel
Cliquer sur le lien vert ci-dessous. Le fichier est aussi déposé dans le dossier Dataiku indiqué en section 1.

In [ ]:
LIMITE_LIEN_MO = 50


def livrer(feuilles, nom):
    debut_ = pd.Timestamp.now()
    octets_, nom_, moteur_ = classeur(feuilles, nom)
    print("{} : {} Mo, construit avec {} en {:.0f} s.".format(nom_, nb(len(octets_) / 1e6, 1), moteur_, (pd.Timestamp.now() - debut_).total_seconds()))
    if NOM_DOSSIER_DATAIKU:
        dataiku.Folder(NOM_DOSSIER_DATAIKU).upload_stream(nom_, BytesIO(octets_))
        note("Déposé dans le dossier Dataiku <b>{}</b> : {}".format(NOM_DOSSIER_DATAIKU, nom_), "ok")
    if len(octets_) <= LIMITE_LIEN_MO * 1e6:
        display(lien_telechargement(octets_, nom_))
    elif not NOM_DOSSIER_DATAIKU:
        note("{} fait {} Mo : trop lourd pour un lien dans le notebook, et aucun dossier Dataiku n'est renseigné (NOM_DOSSIER_DATAIKU).".format(nom_, nb(len(octets_) / 1e6)), "bad")
    return nom_


livrer(FEUILLES, NOM_EXCEL_SORTIE)

### Le détail de chaque GAB, chaque jour
Le fichier complet (tous les GAB, tous les jours) est trop gros pour un seul onglet Excel : il est découpé en plusieurs fichiers, du plus
récent au plus ancien.

In [ ]:
print("Photo complète : {} GAB -> au plus {} jours par fichier Excel ; {} jour(s) par fichier, {} fichier(s) demandé(s) (période : {} jours).".format(
    nb(NB_GAB_PERIODE), JOURS_MAX_ONGLET, N_JOURS_PHOTO, NB_FICHIERS_PHOTO, P["date_jour"].nunique()))
fichiers_photo = []
for k_ in range(int(NB_FICHIERS_PHOTO)):
    fin_k = DERNIER - pd.Timedelta(days=k_ * N_JOURS_PHOTO)
    debut_k = fin_k - pd.Timedelta(days=N_JOURS_PHOTO - 1)
    tranche = P[(P["date_jour"] >= debut_k) & (P["date_jour"] <= fin_k)]
    if tranche.empty:
        print("Fichier {} : plus de données avant le {} (période épuisée).".format(k_ + 1, fin_k.strftime("%d/%m/%Y")))
        break
    debut_k = tranche["date_jour"].min()
    ph = photo_de(tranche)
    entete = pd.DataFrame({"rubrique": ["Période", "GAB", "Lignes", "Lecture"],
                           "contenu": ["du {} au {}".format(debut_k.strftime("%d/%m/%Y"), fin_k.strftime("%d/%m/%Y")), tranche["num_automate"].nunique(), len(ph),
                                       "une ligne par GAB et par jour ; rang du jour 1 = le plus atypique du jour ; niveau Critique / Élevé / À surveiller / Normal"]})
    nom_k = "{}_photo_gab_jour_{}_{}.xlsx".format(NOM_EXCEL_SORTIE.replace(".xlsx", ""), debut_k.strftime("%Y%m%d"), fin_k.strftime("%Y%m%d"))
    print("Fichier {} / {} : {} GAB-jours du {} au {}".format(k_ + 1, NB_FICHIERS_PHOTO, nb(len(ph)), debut_k.strftime("%d/%m/%Y"), fin_k.strftime("%d/%m/%Y")))
    fichiers_photo.append(livrer([("Lisez-moi", entete), ("Photo GAB x jour", ph)], nom_k))

### Les mêmes résultats dans Dataiku
Pour les équipes qui préfèrent travailler dans Dataiku : la photo du dernier jour, les chiffres du jour et la photo complète sont aussi
écrits dans des tables Dataiku.

In [ ]:
if DATASET_SORTIE:
    sortie = photo_de(photo)
    dataiku.Dataset(DATASET_SORTIE).write_with_schema(sortie)
    note("Photo du parc du {} écrite dans {} ({} lignes).".format(DERNIER.strftime("%d/%m/%Y"), DATASET_SORTIE, nb(len(sortie))), "ok")
if DATASET_SORTIE_KPI:
    kpi_sortie = kpi.reset_index().rename(columns={"index": "indicateur"})
    kpi_sortie.insert(0, "date_jour", DERNIER.strftime("%Y-%m-%d"))
    dataiku.Dataset(DATASET_SORTIE_KPI).write_with_schema(kpi_sortie)
    note("KPI du jour écrits dans {}.".format(DATASET_SORTIE_KPI), "ok")
if DATASET_SORTIE_HISTO or NOM_DOSSIER_DATAIKU:
    photo_complete = photo_de(P)
    if DATASET_SORTIE_HISTO:
        dataiku.Dataset(DATASET_SORTIE_HISTO).write_with_schema(photo_complete)
        note("Photo GAB x jour de toute la période écrite dans {} ({} lignes).".format(DATASET_SORTIE_HISTO, nb(len(photo_complete))), "ok")
    if NOM_DOSSIER_DATAIKU:
        flux_csv = BytesIO()
        photo_complete.to_csv(flux_csv, index=False, sep=";", encoding="utf-8-sig", compression="gzip")
        flux_csv.seek(0)
        nom_csv = NOM_EXCEL_SORTIE.replace(".xlsx", "") + "_complet_gab_jour.csv.gz"
        dataiku.Folder(NOM_DOSSIER_DATAIKU).upload_stream(nom_csv, flux_csv)
        note("Photo complète ({} lignes) déposée dans le dossier {} : {}.".format(nb(len(photo_complete)), NOM_DOSSIER_DATAIKU, nom_csv), "ok")
retenir(["Le classeur <b>{}</b> contient la synthèse de la période ({} onglets) ; la photo complète GAB × jour est dans {} fichier(s) séparé(s).".format(NOM_EXCEL_SORTIE, len(FEUILLES), len(fichiers_photo)),
         "Pour le métier : commencer par l'onglet <b>Par année</b> (la vue d'ensemble), puis <b>Parc par jour</b> (quel jour regarder), puis <b>Top du jour</b> (quels GAB, et pourquoi).",
         "La colonne « avis métier » de l'onglet Top du jour est faite pour recueillir le retour terrain : c'est ce retour qui permettra de régler les seuils et, plus tard, d'entraîner un modèle."], "Pour finir")

### Remplissage de « L'essentiel pour le métier » (en haut du notebook)
Cette cellule recopie en tête du notebook ce dont le métier a besoin en premier.

In [ ]:
def _carte_france_svg(ph, top_, largeur=640):
    """Carte de France en SVG (fonctionne partout, sans bibliothèque) : GAB normaux en gris, à surveiller en jaune, atypiques en rouge,
    les GAB de la liste numérotés. Survoler un point affiche le GAB et son diagnostic."""
    lon0, lon1, lat0, lat1 = -5.3, 9.8, 41.2, 51.3
    hauteur = int(largeur * (lat1 - lat0) / ((lon1 - lon0) * np.cos(np.radians(46.5))))
    X = lambda lo: (lo - lon0) / (lon1 - lon0) * largeur
    Y = lambda la: (lat1 - la) / (lat1 - lat0) * hauteur
    fond = "".join('<path d="M {} Z" fill="#f3f5f8" stroke="#b9c2cc" stroke-width="1"/>'.format(" L ".join("{:.1f} {:.1f}".format(X(a), Y(b)) for a, b in poly)) for poly in (FRANCE, CORSE))
    p_ = ph.copy()
    p_["lat"], p_["lon"] = pd.to_numeric(p_["latitude"], errors="coerce"), pd.to_numeric(p_["longitude"], errors="coerce")
    p_ = p_[p_["lat"].between(lat0, lat1) & p_["lon"].between(lon0, lon1)]
    rangs = dict(zip(top_["num_automate"], range(1, len(top_) + 1)))
    pts = []
    ordre = {"Normal": 0, "À surveiller": 1, "Élevé": 2, "Critique": 3}
    for _, r in p_.assign(_o=p_["niveau"].map(ordre)).sort_values("_o").iterrows():
        atyp = r["niveau"] in ("Critique", "Élevé")
        coul = COULEURS_NIVEAU["Critique"] if atyp else (COULEURS_NIVEAU["À surveiller"] if r["niveau"] == "À surveiller" else "#c5cbd3")
        k_ = rangs.get(r["num_automate"])
        rayon = 9 if k_ else (4 if r["niveau"] != "Normal" else 2)
        titre = "GAB {}{} — {}{}".format(r["num_automate"], " (" + str(r["ville"]) + ")" if "ville" in r.index and pd.notnull(r["ville"]) else "", r["niveau"],
                                          "\n" + str(r["resume"]) if r["niveau"] != "Normal" and r["resume"] else "")
        pts.append('<circle cx="{:.1f}" cy="{:.1f}" r="{}" fill="{}" stroke="#fff" stroke-width="{}"><title>{}</title></circle>'.format(
            X(r["lon"]), Y(r["lat"]), rayon, coul, 1 if r["niveau"] != "Normal" else 0, _html(titre)))
        if k_:
            pts.append('<text x="{:.1f}" y="{:.1f}" font-size="10" font-weight="700" text-anchor="middle" fill="#fff" pointer-events="none">{}</text>'.format(X(r["lon"]), Y(r["lat"]) + 3.5, k_))
    leg = ""
    for i, (lib_, coul_) in enumerate([("atypique (Critique ou Élevé)", COULEURS_NIVEAU["Critique"]), ("à surveiller : s'écarte de son habitude", COULEURS_NIVEAU["À surveiller"]), ("normal", "#c5cbd3")]):
        leg += '<circle cx="14" cy="{}" r="6" fill="{}"/><text x="26" y="{}" font-size="12" fill="#1f2328">{}</text>'.format(hauteur - 66 + 20 * i, coul_, hauteur - 62 + 20 * i, lib_)
    return ('<svg xmlns="http://www.w3.org/2000/svg" width="{w}" height="{h}" viewBox="0 0 {w} {h}" style="font-family:Segoe UI,Arial,sans-serif;background:#fff">'
            '{f}{p}{l}</svg>').format(w=largeur, h=hauteur, f=fond, p="".join(pts), l=leg)


# les podiums de la première page : ce que le métier regarde en premier
PODIUM_ESSENTIEL = [("nb_retraits", "Le plus de retraits", "retraits", "hab_retraits", False), ("montant_total", "Le plus d'argent retiré", "€", "hab_montant_total", False),
                    ("nb_retraits", "Le moins de retraits (GAB qui ont servi)", "retraits", "hab_retraits", True), ("nb_inaboutis", "Le plus de retraits échoués", "retraits échoués", "hab_inaboutis", False),
                    ("nb_captures", "Le plus de cartes avalées", "cartes", "hab_captures", False), ("nb_ret_cup", "Le plus de cartes UnionPay (UPI)", "cartes", None, False),
                    ("nb_ret_jcb", "Le plus de cartes JCB", "cartes", None, False), ("nb_ret_diners", "Le plus de cartes Diners", "cartes", None, False)]
PODIUM_ESSENTIEL = [p_ for p_ in PODIUM_ESSENTIEL if p_[0] in jour.columns]


def _podiums_html(ph, n=3):
    cartes = []
    for col_, titre_, unite_, hab_, croissant in PODIUM_ESSENTIEL:
        base = ph[ph["nb_retraits"] > 0] if croissant else ph[ph[col_] > 0]
        t_ = base.nsmallest(n, col_) if croissant else base.nlargest(n, col_)
        lignes = ""
        for k_, (_, x) in enumerate(t_.iterrows()):
            ref_ = x[hab_] if hab_ and hab_ in x.index else x.get("ref_" + col_, np.nan)
            v_ = x[col_]
            u_ = unite_ if v_ > 1 else unite_.replace("retraits", "retrait").replace("cartes", "carte").replace("échoués", "échoué")
            val = (nb(v_) + " €") if unite_ == "€" else "{} {}".format(nb(v_), u_)
            habtxt = "d'habitude : " + ((nb(ref_) + " €") if unite_ == "€" else nb(ref_, 1 if ref_ < 10 else 0)) if pd.notnull(ref_) else ""
            lignes += ('<tr><td style="padding:3px 6px;border:0;width:22px"><span style="display:inline-block;width:18px;height:18px;border-radius:9px;background:{};color:#fff;font-size:11px;text-align:center;line-height:18px">{}</span></td>'
                       '<td style="padding:3px 6px;border:0;text-align:left"><b>{}</b><div style="font-size:11px;color:#57606a;white-space:nowrap">{} · {} retraits · {}</div></td>'
                       '<td style="padding:3px 6px;border:0;text-align:right;white-space:nowrap"><b>{}</b><div style="font-size:11px;color:#57606a">{}</div></td></tr>').format(
                ["#d4a017", "#8a8f98", "#b0703c"][min(k_, 2)], k_ + 1, x["num_automate"], _html(x.get("ville", "")) if pd.notnull(x.get("ville", "")) else "", nb(x["nb_retraits"]).replace(" ", "&nbsp;"),
                (nb(x["montant_total"]) + " €").replace(" ", "&nbsp;"), val.replace(" ", "&nbsp;"), habtxt)
        if not lignes:
            lignes = '<tr><td style="padding:6px;border:0;color:#57606a;font-size:12px">Aucun GAB concerné ce jour-là.</td></tr>'
        cartes.append('<div class="g-kpi" style="flex:1 1 300px;max-width:400px"><div class="l" style="text-transform:none;font-size:12.5px;font-weight:600;color:#1f2328">{}</div>'
                      '<table style="border-collapse:collapse;width:100%;font-size:12.5px;border:0">{}</table></div>'.format(titre_, lignes))
    return '<div class="g-kpis">{}</div>'.format("".join(cartes))


def _bloc_essentiel():
    h = []
    d_ = FIN
    ph = jour[jour["date_jour"] == d_].sort_values("rang_jour")
    ref_ = jour[(jour["date_jour"] < d_) & (jour["date_jour"] >= d_ - pd.Timedelta(days=28)) & (jour["jour_semaine"] == d_.dayofweek + 1)]
    nref = max(1, ref_["date_jour"].nunique())
    h.append('<div class="g-band"><b>L\'essentiel — {}</b><span class="sub">dernier jour complet de la fiche ; {} jour(s) de retard sur aujourd\'hui ; '
             'chaque chiffre est comparé aux 4 {}s précédents</span></div>'.format(jfr(d_).capitalize(), (pd.Timestamp.today().normalize() - FIN).days, JOURS_FR[d_.dayofweek]))

    # 1. le tableau de bord du jour
    def kpi(lib_, col_, euros=False, inverse=False):
        if col_ not in ph.columns:
            return None
        v_, r_ = ph[col_].sum(), ref_[col_].sum() / nref if len(ref_) else np.nan
        e_ = (v_ / r_ - 1) * 100 if r_ and pd.notnull(r_) and r_ > 0 else np.nan
        sens = "" if pd.isnull(e_) or abs(e_) < 15 or r_ < 10 else ("down" if (e_ > 0) == inverse else "up")
        return (lib_, nb(v_) + (" €" if euros else ""), "{}{} % (d'habitude {})".format("+" if e_ >= 0 else "−", nb(abs(e_), 0), nb(r_)) if pd.notnull(e_) else "pas de référence", sens)
    sans = ph["nb_retraits"] == 0
    arret = sans & (ph["part_zero"].fillna(0) <= 0.15) & (ph["hab_retraits"].fillna(0) >= 10)
    ligne1 = []
    if "statut_gab" in ph.columns:
        st_ = ph["statut_gab"].astype(str).str.upper()
        ligne1.append(("GAB actifs au référentiel", nb((st_ == "ACTIF").sum()), "{} non actifs".format(nb((st_ != "ACTIF").sum()))))
    ligne1 += [("GAB qui ont servi", nb((~sans).sum()), "{} sans aucun retrait, dont {} à l'arrêt".format(nb(sans.sum()), nb(arret.sum())), "down" if arret.sum() else ""),
               kpi("Retraits", "nb_retraits"), kpi("Montant retiré", "montant_total", euros=True),
               kpi("Retraits COS", "nb_ret_cos" if "nb_ret_cos" in ph.columns else "nb_retraits_cos")]
    ligne2 = [kpi("Retraits échoués (inaboutis)", "nb_inaboutis", inverse=True), kpi("Cartes avalées (captures)", "nb_captures", inverse=True),
              kpi("Retraits refaits < 10 min", "nb_retraits_repetes_10min", inverse=True), kpi("Retraits de nuit (0 h – 6 h)", "nb_0h_6h")]
    ligne3 = [kpi("Cartes UnionPay (UPI)", "nb_ret_cup"), kpi("Cartes JCB", "nb_ret_jcb"), kpi("Cartes Diners", "nb_ret_diners"), kpi("Cartes Amex", "nb_ret_amex")]
    for ligne in (ligne1, ligne2, ligne3):
        h.append(tuiles([x for x in ligne if x], retour=True))

    # 2. les podiums
    h.append('<div style="font-weight:600;font-size:15px;margin:14px 0 4px 0">Les 3 GAB en tête, indicateur par indicateur</div>'
             '<div class="g-leg">Avec, pour chaque GAB, ses retraits et son montant du jour, et ce qu\'il fait d\'habitude.</div>')
    h.append(_podiums_html(ph))

    # 3. les GAB atypiques, avec le diagnostic et les métriques qui le justifient
    al = ph[ph["niveau"] != "Normal"]
    n_aty = int(ph["niveau"].isin(["Critique", "Élevé"]).sum())
    n_surv = int((ph["niveau"] == "À surveiller").sum())
    h.append('<div class="g-band" style="margin-top:22px"><b>Détection : les {} GAB les plus atypiques du jour</b><span class="sub">{} atypiques (Critique ou Élevé) et {} à surveiller (comportement qui '
             's\'écarte de leur habitude) ; chaque diagnostic est accompagné des chiffres qui le justifient</span></div>'.format(min(TOP_LISTE, len(al)), n_aty, n_surv))
    if len(al):
        t_ = al.head(TOP_LISTE).copy()
        t_["diagnostic"] = t_["diagnostic"].map(court)
        t_["retraits jour / habitude"] = ["{} / {}".format(nb(a_), nb(b_)) for a_, b_ in zip(t_["nb_retraits"], t_["hab_retraits"])]
        t_["montant jour / habitude"] = ["{} € / {} €".format(nb(a_), nb(b_)) for a_, b_ in zip(t_["montant_total"], t_["hab_montant_total"])] if "hab_montant_total" in t_.columns else ""
        t_["statut"] = np.where(t_["niveau"].isin(["Critique", "Élevé"]), "atypique", "à surveiller")
        cols = ["rang_jour", "niveau", "num_automate"] + [c for c in ["ville"] if c in t_.columns] + ["diagnostic", "indicateur clé", "valeur du jour", "valeur habituelle",
                                                                                                     "retraits jour / habitude", "montant jour / habitude", "action conseillée"]
        h.append(tableau(t_[cols].rename(columns={"rang_jour": "rang", "num_automate": "GAB"}), longues=("diagnostic", "action conseillée"), dec=0)._repr_html_().replace(CSS, ""))
        if len(al) < TOP_LISTE:
            h.append('<div class="g-leg">Seuls {} GAB sont en alerte ce jour-là : la liste n\'est pas complétée avec des GAB normaux.</div>'.format(len(al)))
    else:
        h.append('<div class="g-note ok"><b>Aucun GAB atypique ce jour-là</b> : aucun GAB ne s\'est écarté de ses habitudes au point de déclencher une alerte.</div>')
    h.append('<div class="g-leg">« Indicateur clé » : l\'indicateur de l\'anomalie la plus grave ; « valeur habituelle » : ce que fait d\'ordinaire ce GAB ce jour de semaine. '
             'Pour vérifier un diagnostic chiffre par chiffre : section <a href="#s12">12</a>, fiches de preuve.</div>')

    # 4. la carte
    if "latitude" in ph.columns:
        h.append('<div style="font-weight:600;font-size:15px;margin:16px 0 4px 0">Où sont-ils ? (survoler un point pour voir le GAB et son diagnostic)</div>')
        h.append('<div style="display:flex;justify-content:center">{}</div>'.format(_carte_france_svg(ph, al.head(TOP_LISTE))))

    # 5. l'Excel
    h.append('<div class="g-note ok" style="font-size:14px"><b>Le fichier Excel</b> reprend tout, pour chaque jour des 3 ans : la liste du jour avec diagnostics et chiffres, '
             'les alertes, les podiums, le profil de chaque GAB. <a href="#s20"><b>Aller au lien de téléchargement</b></a>.</div>')
    h.append('<div class="g-leg">Aller plus loin : <a href="#s19">tableau de bord des 3 ans</a> · <a href="#s19bis">vision annuelle</a> · <a href="#s18">billets, pannes, GAB vides</a> · '
             '<a href="#sommaire">sommaire</a></div>')
    return CSS + "".join(h)


ESSENTIEL.update(HTML(_bloc_essentiel()))
note("La page « L'essentiel pour le métier », en haut du notebook, est remplie.", "ok")